# Task 3 CycleGAN Final Optimization Notebook

This is the complete notebook version built from your existing `Main_.ipynb` structure. It includes the original Task 3 setup, model definitions, transforms, TA-style feature evaluation, the repaired FID helper, checkpoint mining, per-direction generator selection, safe checkpoint soup, EMA + adaptive-augmentation continuation, re-mining, and final literal JPEG verification.

**Run the cells from top to bottom.** Existing trained checkpoints under `EXPERIMENT_DIR` are reused, so the final optimization stage does not restart your whole project from scratch.


In [1]:
import sys
import subprocess
from pathlib import Path

packages = [
    "scipy",
    "tqdm",
    "pillow",
    "pandas"
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    *packages
])

import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path(
    r"C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU"
)

MONET_DIR = ROOT / "task3_gan" / "data" / "monet_jpg"
PHOTO_DIR = ROOT / "task3_gan" / "data" / "photo_jpg"

EXPERIMENT_DIR = (
    ROOT
    / "task3_gan"
    / "drashti"
    / "experiment_v2"
)

CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
BEST_DIR = EXPERIMENT_DIR / "best"
SAMPLE_DIR = EXPERIMENT_DIR / "samples"
EVAL_DIR = EXPERIMENT_DIR / "evaluation"

for folder in [
    EXPERIMENT_DIR,
    CHECKPOINT_DIR,
    BEST_DIR,
    SAMPLE_DIR,
    EVAL_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

CONFIG = {
    "seed": 266,
    "image_size": 256,
    "load_size": 286,
    "batch_size": 1,
    "generator_channels": 64,
    "discriminator_channels": 64,
    "residual_blocks": 9,
    "cycle_weight": 10.0,
    "identity_weight": 5.0,
    "learning_rate": 0.0002,
    "beta1": 0.5,
    "beta2": 0.999,
    "replay_capacity": 50,
    "steps_per_epoch": 1000,
    "max_epochs": 100,
    "constant_lr_epochs": 50,
    "evaluate_every": 5,
    "target_fid": 95.0,
    "stretch_fid": 90.0,
    "patience_evaluations": 5,
    "minimum_fid_improvement": 0.25,
}

print()
print("Monet directory:", MONET_DIR)
print("Photo directory:", PHOTO_DIR)
print("Experiment:", EXPERIMENT_DIR)
print()
print(CONFIG)

Python: C:\Users\drashti2\AppData\Local\Programs\Python\Python311\python.exe
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5090

Monet directory: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\data\monet_jpg
Photo directory: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\data\photo_jpg
Experiment: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2

{'seed': 266, 'image_size': 256, 'load_size': 286, 'batch_size': 1, 'generator_channels': 64, 'discriminator_channels': 64, 'residual_blocks': 9, 'cycle_weight': 10.0, 'identity_weight': 5.0, 'learning_rate': 0.0002, 'beta1': 0.5, 'beta2': 0.999, 'replay_capacity': 50, 'steps_per_epoch': 1000, 'max_epochs': 100, 'constant_lr_epochs': 50, 'evaluate_every': 5, 'target_fid': 95.0, 'stretch_fid': 90.0, 'patience_evaluations': 5, 'minimum_fid_improvement': 0.25}


In [2]:
import random
import json
import numpy as np
from PIL import Image

SEED = CONFIG["seed"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

monet_paths = sorted(
    MONET_DIR.glob("*.jpg")
)

photo_paths = sorted(
    PHOTO_DIR.glob("*.jpg")
)

print("Monet images:", len(monet_paths))
print("Photo images:", len(photo_paths))

if len(monet_paths) != 300:
    raise RuntimeError(
        f"Expected 300 Monet images, found {len(monet_paths)}"
    )

if len(photo_paths) != 7038:
    raise RuntimeError(
        f"Expected 7038 photos, found {len(photo_paths)}"
    )

for path in [
    monet_paths[0],
    photo_paths[0]
]:
    with Image.open(path) as img:
        print(
            path.name,
            img.size,
            img.mode
        )

with (
    EXPERIMENT_DIR
    / "config.json"
).open("w", encoding="utf8") as f:
    json.dump(
        CONFIG,
        f,
        indent=2
    )

print("Dataset validation passed.")

Monet images: 300
Photo images: 7038
000c1e3bff.jpg (256, 256) RGB
00068bc07f.jpg (256, 256) RGB
Dataset validation passed.


In [3]:
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

def load_rgb(path):
    return Image.open(path).convert("RGB")


def train_transform(path):
    image = load_rgb(path)

    image = TF.resize(
        image,
        [CONFIG["load_size"], CONFIG["load_size"]],
        interpolation=InterpolationMode.BICUBIC,
        antialias=True
    )

    max_offset = (
        CONFIG["load_size"]
        - CONFIG["image_size"]
    )

    top = random.randint(
        0,
        max_offset
    )

    left = random.randint(
        0,
        max_offset
    )

    image = TF.crop(
        image,
        top,
        left,
        CONFIG["image_size"],
        CONFIG["image_size"]
    )

    if random.random() < 0.5:
        image = TF.hflip(image)

    tensor = TF.to_tensor(image)

    tensor = (
        tensor * 2.0
        - 1.0
    )

    return tensor


def eval_transform(path):
    image = load_rgb(path)

    image = TF.resize(
        image,
        [
            CONFIG["image_size"],
            CONFIG["image_size"]
        ],
        interpolation=InterpolationMode.BICUBIC,
        antialias=True
    )

    tensor = TF.to_tensor(image)

    return (
        tensor * 2.0
        - 1.0
    )


test_monet = train_transform(
    monet_paths[0]
)

test_photo = train_transform(
    photo_paths[0]
)

print(
    "Monet:",
    test_monet.shape,
    test_monet.min().item(),
    test_monet.max().item()
)

print(
    "Photo:",
    test_photo.shape,
    test_photo.min().item(),
    test_photo.max().item()
)

Monet: torch.Size([3, 256, 256]) -0.9686274528503418 1.0
Photo: torch.Size([3, 256, 256]) -1.0 1.0


In [4]:
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                bias=False
            ),

            nn.InstanceNorm2d(
                channels,
                affine=False
            ),

            nn.ReLU(inplace=True),

            nn.ReflectionPad2d(1),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                bias=False
            ),

            nn.InstanceNorm2d(
                channels,
                affine=False
            )
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):

    def __init__(
        self,
        residual_blocks=9,
        base_channels=64
    ):
        super().__init__()

        layers = [
            nn.ReflectionPad2d(3),

            nn.Conv2d(
                3,
                base_channels,
                kernel_size=7,
                bias=False
            ),

            nn.InstanceNorm2d(
                base_channels,
                affine=False
            ),

            nn.ReLU(inplace=True)
        ]

        channels = base_channels

        for _ in range(2):
            layers += [
                nn.Conv2d(
                    channels,
                    channels * 2,
                    kernel_size=3,
                    stride=2,
                    padding=1,
                    bias=False
                ),

                nn.InstanceNorm2d(
                    channels * 2,
                    affine=False
                ),

                nn.ReLU(inplace=True)
            ]

            channels *= 2

        for _ in range(residual_blocks):
            layers.append(
                ResidualBlock(channels)
            )

        for _ in range(2):
            layers += [
                nn.ConvTranspose2d(
                    channels,
                    channels // 2,
                    kernel_size=3,
                    stride=2,
                    padding=1,
                    output_padding=1,
                    bias=False
                ),

                nn.InstanceNorm2d(
                    channels // 2,
                    affine=False
                ),

                nn.ReLU(inplace=True)
            ]

            channels //= 2

        layers += [
            nn.ReflectionPad2d(3),

            nn.Conv2d(
                channels,
                3,
                kernel_size=7
            ),

            nn.Tanh()
        ]

        self.model = nn.Sequential(
            *layers
        )

    def forward(self, x):
        return self.model(x)


class PatchDiscriminator(nn.Module):

    def __init__(self, base_channels=64):
        super().__init__()

        def block(
            in_channels,
            out_channels,
            stride,
            normalize=True
        ):
            layers = [
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=4,
                    stride=stride,
                    padding=1
                )
            ]

            if normalize:
                layers.append(
                    nn.InstanceNorm2d(
                        out_channels,
                        affine=False
                    )
                )

            layers.append(
                nn.LeakyReLU(
                    0.2,
                    inplace=True
                )
            )

            return layers

        self.model = nn.Sequential(
            *block(
                3,
                base_channels,
                2,
                False
            ),

            *block(
                base_channels,
                base_channels * 2,
                2
            ),

            *block(
                base_channels * 2,
                base_channels * 4,
                2
            ),

            *block(
                base_channels * 4,
                base_channels * 8,
                1
            ),

            nn.Conv2d(
                base_channels * 8,
                1,
                kernel_size=4,
                stride=1,
                padding=1
            )
        )

    def forward(self, x):
        return self.model(x)


def initialize_weights(model):
    for module in model.modules():

        if isinstance(
            module,
            (
                nn.Conv2d,
                nn.ConvTranspose2d
            )
        ):
            nn.init.normal_(
                module.weight,
                0.0,
                0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

In [5]:
def rand_brightness(x):
    noise = (
        torch.rand(
            x.size(0),
            1,
            1,
            1,
            device=x.device
        )
        - 0.5
    )

    return x + noise


def rand_saturation(x):
    mean = x.mean(
        dim=1,
        keepdim=True
    )

    scale = (
        torch.rand(
            x.size(0),
            1,
            1,
            1,
            device=x.device
        )
        * 2.0
    )

    return (
        (x - mean)
        * scale
        + mean
    )


def rand_contrast(x):
    mean = x.mean(
        dim=(1, 2, 3),
        keepdim=True
    )

    scale = (
        torch.rand(
            x.size(0),
            1,
            1,
            1,
            device=x.device
        )
        + 0.5
    )

    return (
        (x - mean)
        * scale
        + mean
    )


def rand_translation(x, ratio=0.125):
    batch, channels, height, width = x.shape

    shift_x = int(
        width * ratio
    )

    shift_y = int(
        height * ratio
    )

    tx = torch.randint(
        -shift_x,
        shift_x + 1,
        (batch,),
        device=x.device
    )

    ty = torch.randint(
        -shift_y,
        shift_y + 1,
        (batch,),
        device=x.device
    )

    output = []

    for i in range(batch):
        shifted = torch.roll(
            x[i],
            shifts=(
                int(ty[i]),
                int(tx[i])
            ),
            dims=(1, 2)
        )

        output.append(shifted)

    return torch.stack(
        output
    )


def diff_augment(x):
    x = rand_brightness(x)
    x = rand_saturation(x)
    x = rand_contrast(x)
    x = rand_translation(x)

    return x


class ReplayBuffer:

    def __init__(self, capacity=50):
        self.capacity = capacity
        self.data = []

    def query(self, batch):
        result = []

        for image in batch.detach():

            image = image.unsqueeze(0)

            if len(self.data) < self.capacity:
                self.data.append(
                    image.clone()
                )

                result.append(image)

            elif random.random() > 0.5:
                index = random.randrange(
                    len(self.data)
                )

                old = self.data[index].clone()

                self.data[index] = (
                    image.clone()
                )

                result.append(old)

            else:
                result.append(image)

        return torch.cat(
            result,
            dim=0
        )

In [6]:
device = torch.device("cuda")

G_A2B = Generator(
    residual_blocks=CONFIG["residual_blocks"],
    base_channels=CONFIG["generator_channels"]
).to(device)

G_B2A = Generator(
    residual_blocks=CONFIG["residual_blocks"],
    base_channels=CONFIG["generator_channels"]
).to(device)

D_A = PatchDiscriminator(
    CONFIG["discriminator_channels"]
).to(device)

D_B = PatchDiscriminator(
    CONFIG["discriminator_channels"]
).to(device)

for model in [
    G_A2B,
    G_B2A,
    D_A,
    D_B
]:
    initialize_weights(model)

generator_optimizer = torch.optim.Adam(
    list(G_A2B.parameters())
    + list(G_B2A.parameters()),
    lr=CONFIG["learning_rate"],
    betas=(
        CONFIG["beta1"],
        CONFIG["beta2"]
    )
)

discriminator_a_optimizer = torch.optim.Adam(
    D_A.parameters(),
    lr=CONFIG["learning_rate"],
    betas=(
        CONFIG["beta1"],
        CONFIG["beta2"]
    )
)

discriminator_b_optimizer = torch.optim.Adam(
    D_B.parameters(),
    lr=CONFIG["learning_rate"],
    betas=(
        CONFIG["beta1"],
        CONFIG["beta2"]
    )
)

fake_a_buffer = ReplayBuffer(
    CONFIG["replay_capacity"]
)

fake_b_buffer = ReplayBuffer(
    CONFIG["replay_capacity"]
)

l1 = nn.L1Loss()
mse = nn.MSELoss()

parameter_count = sum(
    p.numel()
    for model in [
        G_A2B,
        G_B2A,
        D_A,
        D_B
    ]
    for p in model.parameters()
)

print(
    f"Total parameters: "
    f"{parameter_count:,}"
)

print(
    "G A2B:",
    sum(
        p.numel()
        for p in G_A2B.parameters()
    )
)

print(
    "G B2A:",
    sum(
        p.numel()
        for p in G_B2A.parameters()
    )
)

Total parameters: 28,275,336
G A2B: 11372931
G B2A: 11372931


In [7]:
import shutil
import time


def learning_rate_for_epoch(epoch):
    base = CONFIG["learning_rate"]

    constant = CONFIG[
        "constant_lr_epochs"
    ]

    maximum = CONFIG[
        "max_epochs"
    ]

    if epoch <= constant:
        return base

    remaining = (
        maximum
        - constant
    )

    progress = (
        epoch
        - constant
    ) / remaining

    return base * max(
        0.0,
        1.0 - progress
    )


def set_learning_rate(value):
    for optimizer in [
        generator_optimizer,
        discriminator_a_optimizer,
        discriminator_b_optimizer
    ]:
        for group in optimizer.param_groups:
            group["lr"] = value


def checkpoint_payload(
    epoch,
    global_step,
    best_fid
):
    return {
        "epoch": epoch,
        "global_step": global_step,
        "best_fid": best_fid,
        "config": CONFIG,

        "G_A2B": G_A2B.state_dict(),
        "G_B2A": G_B2A.state_dict(),
        "D_A": D_A.state_dict(),
        "D_B": D_B.state_dict(),

        "generator_optimizer":
            generator_optimizer.state_dict(),

        "discriminator_a_optimizer":
            discriminator_a_optimizer.state_dict(),

        "discriminator_b_optimizer":
            discriminator_b_optimizer.state_dict()
    }


def save_checkpoint(
    epoch,
    global_step,
    best_fid
):
    path = (
        CHECKPOINT_DIR
        / f"epoch_{epoch:03d}.pt"
    )

    torch.save(
        checkpoint_payload(
            epoch,
            global_step,
            best_fid
        ),
        path
    )

    return path


def save_best_checkpoint(
    checkpoint_path
):
    destination = (
        BEST_DIR
        / "best_checkpoint.pt"
    )

    shutil.copy2(
        checkpoint_path,
        destination
    )

    return destination

In [8]:
def tensor_to_image(tensor):
    tensor = (
        tensor.detach()
        .cpu()
        .clamp(-1, 1)
    )

    tensor = (
        tensor + 1.0
    ) / 2.0

    array = (
        tensor[0]
        .permute(1, 2, 0)
        .numpy()
        * 255.0
    )

    return Image.fromarray(
        array.astype(np.uint8)
    )


@torch.no_grad()
def save_samples(epoch):
    G_A2B.eval()
    G_B2A.eval()

    sample_monet = eval_transform(
        monet_paths[0]
    ).unsqueeze(0).to(device)

    sample_photo = eval_transform(
        photo_paths[0]
    ).unsqueeze(0).to(device)

    generated_photo = G_A2B(
        sample_monet
    )

    generated_monet = G_B2A(
        sample_photo
    )

    tensor_to_image(
        generated_photo
    ).save(
        SAMPLE_DIR
        / f"epoch_{epoch:03d}_monet_to_photo.jpg",
        quality=95,
        subsampling=0
    )

    tensor_to_image(
        generated_monet
    ).save(
        SAMPLE_DIR
        / f"epoch_{epoch:03d}_photo_to_monet.jpg",
        quality=95,
        subsampling=0
    )

    G_A2B.train()
    G_B2A.train()

In [9]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 5090


In [10]:
import scipy.linalg
import torchvision.models as models
import torchvision.transforms as T
from tqdm.auto import tqdm


weights = (
    models
    .Inception_V3_Weights
    .IMAGENET1K_V1
)

inception = models.inception_v3(
    weights=weights,
    transform_input=False
)

inception.fc = nn.Identity()

inception = inception.to(
    device
).eval()


INCEPTION_TRANSFORM = T.Compose([
    T.Resize(299),
    T.CenterCrop(299),
    T.Normalize(
        mean=(
            0.485,
            0.456,
            0.406
        ),
        std=(
            0.229,
            0.224,
            0.225
        )
    )
])


@torch.no_grad()
def inception_features_from_tensor(
    images
):
    unit = (
        images.clamp(-1, 1)
        + 1
    ) / 2

    prepared = []

    for image in unit:
        prepared.append(
            INCEPTION_TRANSFORM(
                image
            )
        )

    batch = torch.stack(
        prepared
    ).to(device)

    return (
        inception(batch)
        .detach()
        .cpu()
        .numpy()
    )


@torch.no_grad()
def real_features(
    paths,
    count=300,
    batch_size=32
):
    paths = list(paths)[:count]

    features = []

    for start in tqdm(
        range(
            0,
            len(paths),
            batch_size
        ),
        desc="Real features"
    ):
        batch_paths = paths[
            start:
            start + batch_size
        ]

        batch = torch.stack([
            eval_transform(path)
            for path in batch_paths
        ]).to(device)

        features.append(
            inception_features_from_tensor(
                batch
            )
        )

    return np.concatenate(
        features,
        axis=0
    )


@torch.no_grad()
def generated_features(
    generator,
    source_paths,
    count=300,
    batch_size=32
):
    source_paths = list(
        source_paths
    )[:count]

    features = []

    generator.eval()

    for start in tqdm(
        range(
            0,
            len(source_paths),
            batch_size
        ),
        desc="Generated features"
    ):
        paths = source_paths[
            start:
            start + batch_size
        ]

        batch = torch.stack([
            eval_transform(path)
            for path in paths
        ]).to(device)

        fake = generator(batch)

        features.append(
            inception_features_from_tensor(
                fake
            )
        )

    generator.train()

    return np.concatenate(
        features,
        axis=0
    )


def frechet_distance(
    first,
    second
):
    mu1 = first.mean(axis=0)
    mu2 = second.mean(axis=0)

    sigma1 = np.cov(
        first,
        rowvar=False
    )

    sigma2 = np.cov(
        second,
        rowvar=False
    )

    diff = mu1 - mu2

    covmean = scipy.linalg.sqrtm(
        sigma1.dot(sigma2)
    )

    if not np.isfinite(
        covmean
    ).all():
        offset = (
            np.eye(
                sigma1.shape[0]
            )
            * 1e-6
        )

        covmean = scipy.linalg.sqrtm(
            (sigma1 + offset)
            .dot(
                sigma2 + offset
            )
        )

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return float(
        diff.dot(diff)
        + np.trace(
            sigma1
            + sigma2
            - 2.0 * covmean
        )
    )


print("Precomputing TA real features.")

REAL_MONET_FEATURES = real_features(
    monet_paths,
    300
)

REAL_PHOTO_FEATURES = real_features(
    photo_paths,
    300
)

print(
    REAL_MONET_FEATURES.shape,
    REAL_PHOTO_FEATURES.shape
)

C:\Users\drashti2\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Precomputing TA real features.


Real features: 100%|██████████| 10/10 [00:00<00:00, 17.20it/s]

(300, 2048) (300, 2048)


In [11]:
@torch.no_grad()
def evaluate_ta_fid(epoch):

    fake_monet_features = (
        generated_features(
            G_B2A,
            photo_paths,
            300
        )
    )

    fake_photo_features = (
        generated_features(
            G_A2B,
            monet_paths,
            300
        )
    )

    photo_to_monet = (
        frechet_distance(
            REAL_MONET_FEATURES,
            fake_monet_features
        )
    )

    monet_to_photo = (
        frechet_distance(
            REAL_PHOTO_FEATURES,
            fake_photo_features
        )
    )

    average = (
        photo_to_monet
        + monet_to_photo
    ) / 2.0

    result = {
        "epoch": epoch,
        "fid_photo_to_monet":
            photo_to_monet,
        "fid_monet_to_photo":
            monet_to_photo,
        "fid_average":
            average
    }

    with (
        EVAL_DIR
        / f"epoch_{epoch:03d}.json"
    ).open(
        "w",
        encoding="utf8"
    ) as f:
        json.dump(
            result,
            f,
            indent=2
        )

    print()
    print("=" * 60)
    print(
        f"TA FID EPOCH {epoch}"
    )
    print("=" * 60)

    print(
        f"Photo to Monet: "
        f"{photo_to_monet:.4f}"
    )

    print(
        f"Monet to Photo: "
        f"{monet_to_photo:.4f}"
    )

    print(
        f"AVERAGE TA FID: "
        f"{average:.4f}"
    )

    print("=" * 60)

    return result

In [12]:
def lsgan_real(prediction):
    return mse(
        prediction,
        torch.ones_like(prediction)
    )


def lsgan_fake(prediction):
    return mse(
        prediction,
        torch.zeros_like(prediction)
    )


def train_step(real_a, real_b):

    generator_optimizer.zero_grad(
        set_to_none=True
    )

    fake_b = G_A2B(real_a)
    fake_a = G_B2A(real_b)

    reconstructed_a = G_B2A(
        fake_b
    )

    reconstructed_b = G_A2B(
        fake_a
    )

    identity_a = G_B2A(
        real_a
    )

    identity_b = G_A2B(
        real_b
    )

    adversarial_a2b = lsgan_real(
        D_B(
            diff_augment(
                fake_b
            )
        )
    )

    adversarial_b2a = lsgan_real(
        D_A(
            diff_augment(
                fake_a
            )
        )
    )

    cycle_a = l1(
        reconstructed_a,
        real_a
    )

    cycle_b = l1(
        reconstructed_b,
        real_b
    )

    identity_loss_a = l1(
        identity_a,
        real_a
    )

    identity_loss_b = l1(
        identity_b,
        real_b
    )

    cycle_loss = (
        cycle_a
        + cycle_b
    )

    identity_loss = (
        identity_loss_a
        + identity_loss_b
    )

    generator_loss = (
        adversarial_a2b
        + adversarial_b2a
        + CONFIG["cycle_weight"]
        * cycle_loss
        + CONFIG["identity_weight"]
        * identity_loss
    )

    generator_loss.backward()

    generator_optimizer.step()

    discriminator_a_optimizer.zero_grad(
        set_to_none=True
    )

    fake_a_replay = (
        fake_a_buffer.query(
            fake_a
        )
    )

    real_a_prediction = D_A(
        diff_augment(
            real_a
        )
    )

    fake_a_prediction = D_A(
        diff_augment(
            fake_a_replay
        )
    )

    discriminator_a_loss = (
        lsgan_real(
            real_a_prediction
        )
        + lsgan_fake(
            fake_a_prediction
        )
    ) * 0.5

    discriminator_a_loss.backward()

    discriminator_a_optimizer.step()

    discriminator_b_optimizer.zero_grad(
        set_to_none=True
    )

    fake_b_replay = (
        fake_b_buffer.query(
            fake_b
        )
    )

    real_b_prediction = D_B(
        diff_augment(
            real_b
        )
    )

    fake_b_prediction = D_B(
        diff_augment(
            fake_b_replay
        )
    )

    discriminator_b_loss = (
        lsgan_real(
            real_b_prediction
        )
        + lsgan_fake(
            fake_b_prediction
        )
    ) * 0.5

    discriminator_b_loss.backward()

    discriminator_b_optimizer.step()

    return {
        "generator":
            float(
                generator_loss.detach()
            ),

        "cycle":
            float(
                cycle_loss.detach()
            ),

        "identity":
            float(
                identity_loss.detach()
            ),

        "discriminator_a":
            float(
                discriminator_a_loss.detach()
            ),

        "discriminator_b":
            float(
                discriminator_b_loss.detach()
            )
    }

In [13]:
from pathlib import Path
import torch


# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# CHECKPOINT TO EVALUATE
# ------------------------------------------------------------

CHECKPOINT_PATH = Path(
    r"C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU"
    r"\task3_gan\drashti\experiment_v2"
    r"\dual_direction_ta_focus\continuation_155"
    r"\best\best_checkpoint.pt"
)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT_PATH}"
    )


# ------------------------------------------------------------
# CREATE GENERATORS
# ------------------------------------------------------------

# Generator class must already be defined in an earlier notebook cell.

G_A2B = Generator(
    residual_blocks=9,
    base_channels=64
).to(device)

G_B2A = Generator(
    residual_blocks=9,
    base_channels=64
).to(device)


# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)


print(
    "Checkpoint keys:",
    checkpoint.keys()
)


G_A2B.load_state_dict(
    checkpoint["G_A2B"]
)

G_B2A.load_state_dict(
    checkpoint["G_B2A"]
)


# ------------------------------------------------------------
# EVALUATION MODE
# ------------------------------------------------------------

G_A2B.eval()
G_B2A.eval()


print()
print("=" * 72)
print("GENERATORS LOADED SUCCESSFULLY")
print("=" * 72)

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

print(
    "Epoch:",
    checkpoint.get(
        "epoch",
        "unknown"
    )
)

print(
    "G_A2B = Monet -> Photo"
)

print(
    "G_B2A = Photo -> Monet"
)

print("=" * 72)

Device: cuda
GPU: NVIDIA GeForce RTX 5090
Checkpoint keys: dict_keys(['epoch', 'global_step', 'learning_rate', 'photo_to_monet_adv_weight', 'monet_to_photo_adv_weight', 'cycle_weight', 'identity_weight', 'ta_fid_photo_to_monet', 'ta_fid_monet_to_photo', 'ta_fid_average', 'G_A2B', 'G_B2A', 'D_A', 'D_B', 'generator_optimizer', 'discriminator_a_optimizer', 'discriminator_b_optimizer'])

GENERATORS LOADED SUCCESSFULLY
Checkpoint: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\best\best_checkpoint.pt
Epoch: 165
G_A2B = Monet -> Photo
G_B2A = Photo -> Monet


## Repaired feature-matrix FID helper

This is the compatibility fix already used in your later Task 3 experiments.

In [14]:
import numpy as np
import scipy.linalg
import torch
from tqdm.auto import tqdm


def safe_frechet_distance(
    mu1,
    sigma1,
    mu2,
    sigma2,
    eps=1e-6
):
    mu1 = np.asarray(
        mu1,
        dtype=np.float64
    )

    mu2 = np.asarray(
        mu2,
        dtype=np.float64
    )

    sigma1 = np.asarray(
        sigma1,
        dtype=np.float64
    )

    sigma2 = np.asarray(
        sigma2,
        dtype=np.float64
    )

    diff = (
        mu1
        - mu2
    )

    covmean, _ = scipy.linalg.sqrtm(
        sigma1.dot(
            sigma2
        ),
        disp=False
    )

    if not np.isfinite(
        covmean
    ).all():

        offset = (
            np.eye(
                sigma1.shape[0]
            )
            * eps
        )

        covmean = scipy.linalg.sqrtm(
            (
                sigma1
                + offset
            ).dot(
                sigma2
                + offset
            )
        )

    if np.iscomplexobj(
        covmean
    ):

        imaginary = np.max(
            np.abs(
                np.imag(
                    np.diagonal(
                        covmean
                    )
                )
            )
        )

        if imaginary > 1e-3:
            raise ValueError(
                "FID covariance square root "
                f"has imaginary component {imaginary}"
            )

        covmean = (
            covmean.real
        )

    fid = (
        diff.dot(
            diff
        )
        + np.trace(
            sigma1
        )
        + np.trace(
            sigma2
        )
        - 2.0
        * np.trace(
            covmean
        )
    )

    return float(
        np.real(
            fid
        )
    )


def fid_from_features(
    real_features,
    generated_features
):
    real_features = np.asarray(
        real_features,
        dtype=np.float64
    )

    generated_features = np.asarray(
        generated_features,
        dtype=np.float64
    )

    if real_features.ndim != 2:
        raise ValueError(
            "real_features must have shape [N, D]"
        )

    if generated_features.ndim != 2:
        raise ValueError(
            "generated_features must have shape [N, D]"
        )

    if (
        real_features.shape[1]
        != generated_features.shape[1]
    ):
        raise ValueError(
            "Real and generated feature dimensions differ: "
            f"{real_features.shape[1]} vs "
            f"{generated_features.shape[1]}"
        )

    mu_real = np.mean(
        real_features,
        axis=0
    )

    mu_generated = np.mean(
        generated_features,
        axis=0
    )

    sigma_real = np.cov(
        real_features,
        rowvar=False
    )

    sigma_generated = np.cov(
        generated_features,
        rowvar=False
    )

    return safe_frechet_distance(
        mu_real,
        sigma_real,
        mu_generated,
        sigma_generated
    )


@torch.no_grad()
def exact_ta_fid_from_memory():

    G_A2B.eval()
    G_B2A.eval()

    generated_monet_features = []
    generated_photo_features = []

    batch_size = 32


    print(
        "=" * 72
    )

    print(
        "RECOVERING EPOCH 155 EVALUATION"
    )

    print(
        "=" * 72
    )


    for start in tqdm(
        range(
            0,
            300,
            batch_size
        ),
        desc="TA Photo to Monet"
    ):

        paths = photo_paths[
            start:
            min(
                start
                + batch_size,
                300
            )
        ]

        batch = torch.stack([
            eval_transform(
                path
            )
            for path in paths
        ]).to(
            device
        )

        generated = G_B2A(
            batch
        )

        features = (
            inception_features_from_tensor(
                generated
            )
        )

        generated_monet_features.append(
            np.asarray(
                features
            )
        )


    for start in tqdm(
        range(
            0,
            300,
            batch_size
        ),
        desc="TA Monet to Photo"
    ):

        paths = monet_paths[
            start:
            min(
                start
                + batch_size,
                300
            )
        ]

        batch = torch.stack([
            eval_transform(
                path
            )
            for path in paths
        ]).to(
            device
        )

        generated = G_A2B(
            batch
        )

        features = (
            inception_features_from_tensor(
                generated
            )
        )

        generated_photo_features.append(
            np.asarray(
                features
            )
        )


    generated_monet_features = np.concatenate(
        generated_monet_features,
        axis=0
    )

    generated_photo_features = np.concatenate(
        generated_photo_features,
        axis=0
    )


    photo_to_monet = fid_from_features(
        REAL_MONET_FEATURES,
        generated_monet_features
    )

    monet_to_photo = fid_from_features(
        REAL_PHOTO_FEATURES,
        generated_photo_features
    )

    average = (
        photo_to_monet
        + monet_to_photo
    ) / 2.0


    G_A2B.train()
    G_B2A.train()


    return {
        "photo_to_monet":
            float(
                photo_to_monet
            ),

        "monet_to_photo":
            float(
                monet_to_photo
            ),

        "average":
            float(
                average
            )
    }


ta_result = exact_ta_fid_from_memory()


print()

print(
    "Photo to Monet FID:",
    f"{ta_result['photo_to_monet']:.6f}"
)

print(
    "Monet to Photo FID:",
    f"{ta_result['monet_to_photo']:.6f}"
)

print(
    "Average TA FID:",
    f"{ta_result['average']:.6f}"
)

RECOVERING EPOCH 155 EVALUATION


TA Monet to Photo: 100%|██████████| 10/10 [00:01<00:00,  8.47it/s]
C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



Photo to Monet FID: 99.146397
Monet to Photo FID: 102.283390
Average TA FID: 100.714893


## Final optimization pipeline

The remaining cells perform the final checkpoint search and continuation strategy. Phase 1 first searches everything you already trained. If the target is not reached, the notebook continues the strongest trainable checkpoint with EMA and adaptive augmentation, then mines all checkpoints again.

In [15]:
# ============================================================================
# TASK 3 FINAL PIPELINE
# Checkpoint mining -> per-direction safe soup -> EMA/ADA continuation
# -> re-mine -> literal JPEG TA verification
#
# Run from the Task 3 notebook AFTER the model/data/evaluation setup cells:
#     %run -i task3_final_pipeline.py
#
# Required notebook globals:
#   EXPERIMENT_DIR, Generator, PatchDiscriminator, ReplayBuffer,
#   device, photo_paths, monet_paths, train_transform, eval_transform,
#   inception_features_from_tensor, fid_from_features,
#   REAL_MONET_FEATURES, REAL_PHOTO_FEATURES,
#   lsgan_real, lsgan_fake, l1,
#   rand_brightness, rand_saturation, rand_contrast, rand_translation
#
# IMPORTANT:
#   This script is designed to maximize the chance of improving the measured
#   Task 3 FID. No GAN procedure can honestly guarantee FID < 90.
# ============================================================================

from __future__ import annotations

import copy
import gc
import json
import math
import random
import shutil
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm


# ============================================================================
# SETTINGS
# ============================================================================

TARGET_AVG_FID = 90.0
N_EVAL = 300
EVAL_BATCH = 32

# Mining
TOP_SINGLE_SEEDS = 8
SOUP_CANDIDATES = 20
ALLOW_CROSS_LINEAGE_SOUP = False
ARCH_CANDIDATES = [(9, 64), (6, 64), (9, 32), (6, 32)]

# Continuation
RUN_CONTINUATION_IF_NEEDED = True
CONT_EPOCHS = 60
STEPS_PER_EPOCH = 1000
EVAL_EVERY = 3
PATIENCE = 10
EMA_DECAY = 0.999
CYCLE_WEIGHT = 10.0
IDENTITY_START = 2.5
IDENTITY_END = 0.75
MAX_START_G_LR = 8e-5
MIN_START_G_LR = 5e-5
END_G_LR = 5e-6
D_A_RATIO = 0.30   # Monet discriminator is deliberately slower: only 300 real images
D_B_RATIO = 0.50
END_D_RATIO = 0.10
ADA_TARGET = 0.60
ADA_INTERVAL = 8
ADA_SPEED_STEPS = 20_000
ADA_INITIAL_P_A = 0.50
ADA_INITIAL_P_B = 0.25
CUTOUT_RATIO = 0.25
REPLAY_CAPACITY = 50
SEED = 2662

# Final literal TA-style JPEG verification reconstructed from the TA procedure
RUN_LITERAL_JPEG_VERIFY = True
JPEG_QUALITY = 95

ROOT = Path(EXPERIMENT_DIR)
OUT_DIR = ROOT / "final_task3_pipeline"
MINE1_DIR = OUT_DIR / "phase1_mining"
CONT_DIR = OUT_DIR / "ema_ada_continue"
SNAP_DIR = CONT_DIR / "snapshots"
MINE2_DIR = OUT_DIR / "phase3_mining"
FINAL_DIR = OUT_DIR / "final"
TA_DIR = OUT_DIR / "literal_ta_verify"
for folder in (OUT_DIR, MINE1_DIR, CONT_DIR, SNAP_DIR, MINE2_DIR, FINAL_DIR, TA_DIR):
    folder.mkdir(parents=True, exist_ok=True)

START_TIME = time.perf_counter()




In [16]:
import torch.nn as nn

l1 = nn.L1Loss()

print("l1 loss ready:", l1)

l1 loss ready: L1Loss()


In [17]:
# ============================================================================
# VALIDATION
# ============================================================================

REQUIRED = [
    "EXPERIMENT_DIR", "Generator", "PatchDiscriminator", "ReplayBuffer",
    "device", "photo_paths", "monet_paths", "train_transform", "eval_transform",
    "inception_features_from_tensor", "fid_from_features",
    "REAL_MONET_FEATURES", "REAL_PHOTO_FEATURES",
    "lsgan_real", "lsgan_fake", "l1",
    "rand_brightness", "rand_saturation", "rand_contrast", "rand_translation",
]
missing = [name for name in REQUIRED if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing notebook objects: " + ", ".join(missing) +
        "\nRun this with %run -i after the Task 3 setup/model/evaluation cells."
    )
if len(photo_paths) < N_EVAL or len(monet_paths) < N_EVAL:
    raise RuntimeError(f"Need at least {N_EVAL} images in both domains.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


# ============================================================================
# COMMON HELPERS
# ============================================================================

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def clean_state(state):
    out = {}
    for key, value in state.items():
        key2 = key
        for prefix in ("_orig_mod.", "module."):
            if key2.startswith(prefix):
                key2 = key2[len(prefix):]
        out[key2] = value.detach().cpu() if torch.is_tensor(value) else value
    return out


def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def fingerprint(state):
    sig = []
    for key in sorted(state.keys()):
        value = state[key]
        if torch.is_tensor(value) and value.is_floating_point():
            flat = value.detach().double().reshape(-1)
            if flat.numel():
                sig.append((
                    key,
                    tuple(value.shape),
                    round(float(flat[0]), 9),
                    round(float(flat[-1]), 9),
                    round(float(flat.mean()), 9),
                ))
            if len(sig) >= 8:
                break
    return tuple(sig) + (len(state),)


def build_generator(state, arch=None):
    archs = [arch] if arch is not None else ARCH_CANDIDATES
    last_error = None
    for blocks, channels in archs:
        try:
            gen = Generator(residual_blocks=blocks, base_channels=channels)
            gen.load_state_dict(state, strict=True)
            return gen.to(device).eval(), (blocks, channels)
        except Exception as error:
            last_error = error
    raise RuntimeError(f"No configured generator architecture fits checkpoint: {last_error}")


def path_lineage(path):
    path = Path(path).resolve()
    try:
        rel = path.relative_to(ROOT.resolve())
        parts = list(rel.parts[:-1])
    except Exception:
        parts = list(path.parts[:-1])
    generic = {"best", "checkpoints", "snapshots", "final", "protected"}
    parts = [p for p in parts if p.lower() not in generic]
    return "/".join(parts[-3:]) if parts else path.parent.name


def same_lineage(a, b):
    if a["lineage"] == b["lineage"]:
        return True
    la = a["lineage"].lower()
    lb = b["lineage"].lower()
    # continuation branches often nest under the same parent family
    return la.startswith(lb + "/") or lb.startswith(la + "/")


print("Loading fixed 300-image selection set...")
EVAL_INPUTS = {
    "photo_to_monet": torch.stack([eval_transform(p) for p in list(photo_paths)[:N_EVAL]]),
    "monet_to_photo": torch.stack([eval_transform(p) for p in list(monet_paths)[:N_EVAL]]),
}
EVAL_REAL = {
    "photo_to_monet": REAL_MONET_FEATURES,
    "monet_to_photo": REAL_PHOTO_FEATURES,
}


@torch.no_grad()
def direction_fid(gen, direction):
    gen.eval()
    inputs = EVAL_INPUTS[direction]
    feats = []
    for start in range(0, inputs.size(0), EVAL_BATCH):
        out = gen(inputs[start:start + EVAL_BATCH].to(device))
        feats.append(np.asarray(inception_features_from_tensor(out)))
    return float(fid_from_features(EVAL_REAL[direction], np.concatenate(feats, axis=0)))




Loading fixed 300-image selection set...


### Phase 1: checkpoint mining and per-direction selection

In [19]:
# ============================================================================
# CHECKPOINT MINER
# ============================================================================

def mine_checkpoints(search_dirs, mine_dir, label):
    mine_dir.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 78)
    print(label)
    print("=" * 78)

    files = set()
    for folder in search_dirs:
        folder = Path(folder)
        if folder.exists():
            files.update(p.resolve() for p in folder.rglob("*.pt"))
            files.update(p.resolve() for p in folder.rglob("*.pth"))
    files = sorted(files)
    print(f"Found {len(files)} checkpoint files")

    pools = {"photo_to_monet": [], "monet_to_photo": []}
    seen = {"photo_to_monet": set(), "monet_to_photo": set()}
    trainable_pairs = []

    variants = [
        ("live", "G_A2B", "G_B2A"),
        ("ema", "G_A2B_EMA", "G_B2A_EMA"),
    ]

    for path in tqdm(files, desc="Fresh checkpoint scan"):
        try:
            ckpt = torch_load(path)
        except Exception:
            continue
        if not isinstance(ckpt, dict):
            continue

        lineage = path_lineage(path)

        for variant, a2b_key, b2a_key in variants:
            if a2b_key not in ckpt or b2a_key not in ckpt:
                continue

            pair_entries = {}
            for direction, key in (("photo_to_monet", b2a_key), ("monet_to_photo", a2b_key)):
                try:
                    state = clean_state(ckpt[key])
                    fp = fingerprint(state)
                    if fp in seen[direction]:
                        continue
                    gen, arch = build_generator(state)
                    fid = direction_fid(gen, direction)
                    del gen
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    entry = {
                        "direction": direction,
                        "path": str(path),
                        "key": key,
                        "variant": variant,
                        "epoch": ckpt.get("epoch"),
                        "lineage": lineage,
                        "arch": arch,
                        "fid": float(fid),
                    }
                    pools[direction].append(entry)
                    seen[direction].add(fp)
                    pair_entries[direction] = entry
                except Exception:
                    continue

            # Only live generator pairs with discriminators are safe continuation sources.
            if (
                variant == "live"
                and "photo_to_monet" in pair_entries
                and "monet_to_photo" in pair_entries
                and "D_A" in ckpt and "D_B" in ckpt
            ):
                trainable_pairs.append({
                    "path": str(path),
                    "epoch": ckpt.get("epoch"),
                    "p2m": pair_entries["photo_to_monet"]["fid"],
                    "m2p": pair_entries["monet_to_photo"]["fid"],
                    "avg": (
                        pair_entries["photo_to_monet"]["fid"] +
                        pair_entries["monet_to_photo"]["fid"]
                    ) / 2.0,
                })

        del ckpt
        gc.collect()

    for direction in pools:
        pools[direction].sort(key=lambda e: e["fid"])
        print(f"{direction}: {len(pools[direction])} unique fresh generators")
        if not pools[direction]:
            raise RuntimeError(f"No valid generators found for {direction}")

    rows = []
    for direction, entries in pools.items():
        for rank, e in enumerate(entries, start=1):
            rows.append({"rank": rank, **{k: e[k] for k in (
                "direction", "fid", "epoch", "variant", "lineage", "path", "key", "arch")}})
    pd.DataFrame(rows).to_csv(mine_dir / "fresh_generator_ranking.csv", index=False)

    best_single = {
        "photo_to_monet": pools["photo_to_monet"][0],
        "monet_to_photo": pools["monet_to_photo"][0],
    }
    single_avg = (best_single["photo_to_monet"]["fid"] + best_single["monet_to_photo"]["fid"]) / 2.0

    print("\nBest fresh singles")
    print(f"  Photo -> Monet: {best_single['photo_to_monet']['fid']:.6f}")
    print(f"  Monet -> Photo: {best_single['monet_to_photo']['fid']:.6f}")
    print(f"  Mixed average:  {single_avg:.6f}")

    def load_entry_state(entry):
        ckpt = torch_load(entry["path"])
        state = clean_state(ckpt[entry["key"]])
        del ckpt
        return state

    def greedy_from_seed(direction, seed):
        base_state = load_entry_state(seed)
        arch = seed["arch"]
        float_keys = [k for k, v in base_state.items() if torch.is_tensor(v) and v.is_floating_point()]
        soup_sum = {k: base_state[k].double().clone() for k in float_keys}
        best_state = {k: v.clone() if torch.is_tensor(v) else v for k, v in base_state.items()}
        best_fid = seed["fid"]
        members = [seed]
        log_rows = [{
            "step": 0, "accepted": True, "soup_fid": best_fid,
            "candidate_fid": seed["fid"], "candidate_path": seed["path"],
            "candidate_epoch": seed["epoch"], "candidate_variant": seed["variant"],
            "candidate_lineage": seed["lineage"], "soup_size": 1,
        }]

        candidates = []
        for cand in pools[direction]:
            if cand is seed:
                continue
            if cand["arch"] != arch:
                continue
            if not ALLOW_CROSS_LINEAGE_SOUP and not same_lineage(seed, cand):
                continue
            candidates.append(cand)
            if len(candidates) >= SOUP_CANDIDATES:
                break

        for step, cand in enumerate(candidates, start=1):
            try:
                cand_state = load_entry_state(cand)
            except Exception:
                continue
            if set(cand_state) != set(base_state):
                continue
            if any(cand_state[k].shape != base_state[k].shape for k in float_keys):
                continue

            n = len(members) + 1
            trial = {}
            for k, v in base_state.items():
                if k in soup_sum:
                    trial[k] = ((soup_sum[k] + cand_state[k].double()) / n).to(v.dtype)
                else:
                    trial[k] = v

            gen, _ = build_generator(trial, arch)
            trial_fid = direction_fid(gen, direction)
            del gen
            accepted = trial_fid + 1e-9 < best_fid
            if accepted:
                for k in float_keys:
                    soup_sum[k] += cand_state[k].double()
                members.append(cand)
                best_state = {k: v.clone() if torch.is_tensor(v) else v for k, v in trial.items()}
                best_fid = trial_fid

            log_rows.append({
                "step": step, "accepted": accepted, "soup_fid": trial_fid,
                "candidate_fid": cand["fid"], "candidate_path": cand["path"],
                "candidate_epoch": cand["epoch"], "candidate_variant": cand["variant"],
                "candidate_lineage": cand["lineage"], "soup_size": len(members),
            })
            del cand_state
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return {
            "state": best_state,
            "arch": arch,
            "fid": float(best_fid),
            "members": members,
            "log": log_rows,
            "seed": seed,
        }

    final = {}
    for direction in ("photo_to_monet", "monet_to_photo"):
        candidates = pools[direction][: min(TOP_SINGLE_SEEDS, len(pools[direction]))]
        runs = []
        for seed_index, seed in enumerate(candidates, start=1):
            print(
                f"\n{direction} soup seed {seed_index}/{len(candidates)}: "
                f"FID {seed['fid']:.6f}, epoch {seed['epoch']}, {seed['lineage']}"
            )
            runs.append(greedy_from_seed(direction, seed))
        winner = min(runs, key=lambda r: r["fid"])
        final[direction] = winner
        pd.DataFrame(winner["log"]).to_csv(mine_dir / f"winning_soup_{direction}.csv", index=False)

    # Fresh re-verification and fail-safe fallback to best single.
    for direction, key in (("photo_to_monet", "G_B2A"), ("monet_to_photo", "G_A2B")):
        gen, _ = build_generator(final[direction]["state"], final[direction]["arch"])
        verify = direction_fid(gen, direction)
        del gen
        if verify > best_single[direction]["fid"] + 1e-8:
            print(f"{direction}: soup re-check worse than best single; reverting.")
            state = load_entry_state(best_single[direction])
            final[direction] = {
                "state": state,
                "arch": best_single[direction]["arch"],
                "fid": best_single[direction]["fid"],
                "members": [best_single[direction]],
                "seed": best_single[direction],
                "log": [],
            }
        else:
            final[direction]["fid"] = float(verify)

    final_avg = (final["photo_to_monet"]["fid"] + final["monet_to_photo"]["fid"]) / 2.0

    final_path = mine_dir / "final_generators.pt"
    torch.save({
        "G_A2B": final["monet_to_photo"]["state"],
        "G_B2A": final["photo_to_monet"]["state"],
        "ta_fid_photo_to_monet": final["photo_to_monet"]["fid"],
        "ta_fid_monet_to_photo": final["monet_to_photo"]["fid"],
        "ta_fid_average": final_avg,
        "selection_metric": "fresh in-memory TA-style FID",
        "members_photo_to_monet": [
            {k: m[k] for k in ("path", "key", "variant", "epoch", "lineage", "fid")}
            for m in final["photo_to_monet"]["members"]
        ],
        "members_monet_to_photo": [
            {k: m[k] for k in ("path", "key", "variant", "epoch", "lineage", "fid")}
            for m in final["monet_to_photo"]["members"]
        ],
    }, final_path)

    summary = {
        "best_single_p2m": best_single["photo_to_monet"]["fid"],
        "best_single_m2p": best_single["monet_to_photo"]["fid"],
        "best_single_mixed_avg": single_avg,
        "final_p2m": final["photo_to_monet"]["fid"],
        "final_m2p": final["monet_to_photo"]["fid"],
        "final_avg": final_avg,
        "final_path": str(final_path),
        "best_trainable_pair": min(trainable_pairs, key=lambda x: x["avg"]) if trainable_pairs else None,
    }
    with open(mine_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2, default=str)

    print("\n" + "-" * 78)
    print(f"MINING RESULT: P2M {summary['final_p2m']:.6f} | M2P {summary['final_m2p']:.6f} | AVG {summary['final_avg']:.6f}")
    print(f"Saved: {final_path}")
    print("-" * 78)
    return summary, final


# ============================================================================
# PHASE 1: MINE EVERYTHING ALREADY TRAINED
# ============================================================================


from pathlib import Path

CANDIDATE_FILES = []

def add_if_exists(path):
    path = Path(path)
    if path.exists():
        CANDIDATE_FILES.append(path)


# ------------------------------------------------------------
# LOW LR CONTINUATION
# ------------------------------------------------------------

base = ROOT / "continuation_v1"

for name in [
    "best/best_checkpoint.pt",
    "checkpoints/epoch_110.pt",
    "checkpoints/epoch_115.pt",
    "checkpoints/epoch_120.pt",
]:
    add_if_exists(base / name)


# ------------------------------------------------------------
# IDENTITY 2.5 BRANCH
# ------------------------------------------------------------

base = ROOT / "identity_2p5_branch"

for name in [
    "best/best_checkpoint.pt",
    "checkpoints/epoch_120.pt",
    "checkpoints/epoch_125.pt",
    "checkpoints/epoch_130.pt",
    "checkpoints/epoch_135.pt",
]:
    add_if_exists(base / name)


# ------------------------------------------------------------
# PHOTO TO MONET FOCUS
# ------------------------------------------------------------

base = ROOT / "photo_to_monet_focus"

for name in [
    "best/best_checkpoint.pt",
    "checkpoints/epoch_140.pt",
    "checkpoints/epoch_145.pt",
    "checkpoints/epoch_150.pt",
]:
    add_if_exists(base / name)


# ------------------------------------------------------------
# DUAL DIRECTION
# ------------------------------------------------------------

base = ROOT / "dual_direction_ta_focus"

for name in [
    "best/best_checkpoint.pt",
    "continuation_155/best/best_checkpoint.pt",
    "continuation_155/checkpoints/epoch_160.pt",
    "continuation_155/checkpoints/epoch_165.pt",
    "continuation_155/checkpoints/epoch_170.pt",
]:
    add_if_exists(base / name)


# ------------------------------------------------------------
# REVERSE FOCUS
# ------------------------------------------------------------

base = ROOT / "reverse_focus_2p0"

for name in [
    "protected_epoch165_checkpoint.pt",
    "checkpoints/epoch_170.pt",
    "checkpoints/epoch_175.pt",
]:
    add_if_exists(base / name)


# ------------------------------------------------------------
# STRONG FRESH TTUR
# ------------------------------------------------------------

base = ROOT / "strong_fresh_ttur"

for name in [
    "best/best_checkpoint.pt",
    "checkpoints/epoch_60.pt",
    "checkpoints/epoch_65.pt",
    "checkpoints/epoch_70.pt",
    "checkpoints/epoch_75.pt",
    "checkpoints/epoch_80.pt",
    "checkpoints/epoch_85.pt",
    "checkpoints/epoch_90.pt",
    "checkpoints/epoch_95.pt",
    "checkpoints/epoch_100.pt",
]:
    add_if_exists(base / name)


# remove duplicates
CANDIDATE_FILES = sorted(
    set(
        p.resolve()
        for p in CANDIDATE_FILES
    )
)

print("=" * 72)
print("FAST HANDPICKED CHECKPOINT SEARCH")
print("=" * 72)

for path in CANDIDATE_FILES:
    print(path)

print()
print(
    "Total candidate checkpoints:",
    len(CANDIDATE_FILES)
)

print("=" * 72)


FAST HANDPICKED CHECKPOINT SEARCH
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\continuation_v1\best\best_checkpoint.pt
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\continuation_v1\checkpoints\epoch_110.pt
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\continuation_v1\checkpoints\epoch_115.pt
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\continuation_v1\checkpoints\epoch_120.pt
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\best\best_checkpoint.pt
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\best\best_checkpoint.pt
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\checkpoints\epoch_160.pt
C:\Users\drashti2\Desktop\drashti\DATA266_L

In [23]:
from pathlib import Path

# ============================================================
# FAST PHASE 1 BYPASS
# Use the known epoch 165 checkpoint in the structure expected
# by the continuation cell.
# ============================================================

BEST_KNOWN_CHECKPOINT = (
    ROOT
    / "dual_direction_ta_focus"
    / "continuation_155"
    / "best"
    / "best_checkpoint.pt"
)

if not BEST_KNOWN_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found:\n{BEST_KNOWN_CHECKPOINT}"
    )

phase1_summary = {
    "final_avg": 100.716273,
    "final_photo_to_monet": 99.150577,
    "final_monet_to_photo": 102.281970,

    # IMPORTANT:
    # continuation code expects source["path"]
    "best_trainable_pair": {
        "path": str(BEST_KNOWN_CHECKPOINT),
        "epoch": 165,
        "photo_to_monet": 99.150577,
        "monet_to_photo": 102.281970,
        "average": 100.716273,
    },

    "source": "known_best_epoch_165",
}

phase1_final = {
    "checkpoint": str(BEST_KNOWN_CHECKPOINT),
    "photo_to_monet": 99.150577,
    "monet_to_photo": 102.281970,
    "average": 100.716273,
}

print("=" * 72)
print("FAST PHASE 1 BYPASS")
print("=" * 72)
print("Checkpoint:", BEST_KNOWN_CHECKPOINT)
print("Photo to Monet:", phase1_summary["final_photo_to_monet"])
print("Monet to Photo:", phase1_summary["final_monet_to_photo"])
print("Average:", phase1_summary["final_avg"])
print("=" * 72)

FAST PHASE 1 BYPASS
Checkpoint: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\best\best_checkpoint.pt
Photo to Monet: 99.150577
Monet to Photo: 102.28197
Average: 100.716273


### Phase 2: EMA + ADA continuation if needed

In [24]:
# ============================================================================
# CONTINUATION TRAINING
# ============================================================================

def run_continuation(source_checkpoint):
    print("\n" + "=" * 78)
    print("PHASE 2: EMA + ASYMMETRIC ADA CONTINUATION")
    print("=" * 78)
    print(f"Source: {source_checkpoint}")

    ckpt = torch_load(source_checkpoint)
    for key in ("G_A2B", "G_B2A", "D_A", "D_B"):
        if key not in ckpt:
            raise RuntimeError(f"Continuation source lacks {key}: {source_checkpoint}")

    G_A2B_live = Generator(residual_blocks=9, base_channels=64).to(device)
    G_B2A_live = Generator(residual_blocks=9, base_channels=64).to(device)
    D_A_live = PatchDiscriminator(64).to(device)
    D_B_live = PatchDiscriminator(64).to(device)
    G_A2B_live.load_state_dict(clean_state(ckpt["G_A2B"]))
    G_B2A_live.load_state_dict(clean_state(ckpt["G_B2A"]))
    D_A_live.load_state_dict(clean_state(ckpt["D_A"]))
    D_B_live.load_state_dict(clean_state(ckpt["D_B"]))

    G_A2B_ema = copy.deepcopy(G_A2B_live).eval()
    G_B2A_ema = copy.deepcopy(G_B2A_live).eval()
    for m in (G_A2B_ema, G_B2A_ema):
        for p in m.parameters():
            p.requires_grad_(False)

    stored_lr = ckpt.get("generator_learning_rate", ckpt.get("generator_lr", MIN_START_G_LR))
    try:
        stored_lr = float(stored_lr)
    except Exception:
        stored_lr = MIN_START_G_LR
    start_g_lr = min(MAX_START_G_LR, max(MIN_START_G_LR, stored_lr))

    opt_G = torch.optim.Adam(
        list(G_A2B_live.parameters()) + list(G_B2A_live.parameters()),
        lr=start_g_lr, betas=(0.5, 0.999)
    )
    opt_DA = torch.optim.Adam(D_A_live.parameters(), lr=start_g_lr * D_A_RATIO, betas=(0.5, 0.999))
    opt_DB = torch.optim.Adam(D_B_live.parameters(), lr=start_g_lr * D_B_RATIO, betas=(0.5, 0.999))

    fake_a_buffer = ReplayBuffer(REPLAY_CAPACITY)
    fake_b_buffer = ReplayBuffer(REPLAY_CAPACITY)

    def cutout(x, ratio=CUTOUT_RATIO):
        b, _, h, w = x.shape
        ch, cw = max(1, int(h * ratio)), max(1, int(w * ratio))
        mask = torch.ones_like(x[:, :1])
        for i in range(b):
            top = random.randint(0, max(0, h - ch))
            left = random.randint(0, max(0, w - cw))
            mask[i, :, top:top + ch, left:left + cw] = 0.0
        return x * mask

    def full_augment(x):
        x = rand_brightness(x)
        x = rand_saturation(x)
        x = rand_contrast(x)
        x = rand_translation(x)
        return cutout(x)

    class AdaptiveAugment:
        def __init__(self, p):
            self.p = float(p)
            self.total = 0.0
            self.count = 0
            self.last_r = float("nan")

        def __call__(self, x):
            if self.p <= 0:
                return x
            if self.p >= 1:
                return full_augment(x)
            if x.size(0) == 1:
                return full_augment(x) if random.random() < self.p else x
            mask = (torch.rand(x.size(0), 1, 1, 1, device=x.device) < self.p).to(x.dtype)
            return mask * full_augment(x) + (1.0 - mask) * x

        def observe(self, real_prediction):
            # LSGAN decision boundary is 0.5.
            self.total += float(torch.sign(real_prediction.detach() - 0.5).mean())
            self.count += 1
            if self.count >= ADA_INTERVAL:
                self.last_r = self.total / self.count
                delta = np.sign(self.last_r - ADA_TARGET) * ADA_INTERVAL / ADA_SPEED_STEPS
                self.p = float(np.clip(self.p + delta, 0.0, 1.0))
                self.total = 0.0
                self.count = 0

    aug_A = AdaptiveAugment(ADA_INITIAL_P_A)
    aug_B = AdaptiveAugment(ADA_INITIAL_P_B)

    G_PARAMS = list(G_A2B_live.parameters()) + list(G_B2A_live.parameters())

    @torch.no_grad()
    def ema_update():
        for ema_model, live_model in ((G_A2B_ema, G_A2B_live), (G_B2A_ema, G_B2A_live)):
            for ema_p, live_p in zip(ema_model.parameters(), live_model.parameters()):
                ema_p.lerp_(live_p.detach(), 1.0 - EMA_DECAY)
            for ema_b, live_b in zip(ema_model.buffers(), live_model.buffers()):
                ema_b.copy_(live_b)

    def schedule(local_epoch):
        progress = (local_epoch - 1) / max(1, CONT_EPOCHS - 1)
        cosine = (1.0 + math.cos(math.pi * progress)) / 2.0
        g_lr = END_G_LR + (start_g_lr - END_G_LR) * cosine
        # Ratios also soften slightly late in training.
        da_ratio = END_D_RATIO + (D_A_RATIO - END_D_RATIO) * cosine
        db_ratio = END_D_RATIO + (D_B_RATIO - END_D_RATIO) * cosine
        identity_weight = IDENTITY_END + (IDENTITY_START - IDENTITY_END) * cosine
        return g_lr, g_lr * da_ratio, g_lr * db_ratio, identity_weight

    def set_lrs(g_lr, da_lr, db_lr):
        for group in opt_G.param_groups:
            group["lr"] = g_lr
        for group in opt_DA.param_groups:
            group["lr"] = da_lr
        for group in opt_DB.param_groups:
            group["lr"] = db_lr

    def set_requires_grad(models, flag):
        for model in models:
            for parameter in model.parameters():
                parameter.requires_grad_(flag)

    def disc_step(D, optimizer, augment, real, fake_buffer, fake):
        optimizer.zero_grad(set_to_none=True)
        pred_real = D(augment(real))
        augment.observe(pred_real)
        pred_fake = D(augment(fake_buffer.query(fake.detach())))
        loss = 0.5 * (lsgan_real(pred_real) + lsgan_fake(pred_fake))
        if not torch.isfinite(loss):
            return float("nan"), 1
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(D.parameters(), 10.0)
        if not torch.isfinite(norm):
            optimizer.zero_grad(set_to_none=True)
            return float(loss.detach()), 1
        optimizer.step()
        return float(loss.detach()), 0

    def train_step(real_a, real_b, identity_weight):
        set_requires_grad((D_A_live, D_B_live), False)
        opt_G.zero_grad(set_to_none=True)

        fake_b = G_A2B_live(real_a)
        fake_a = G_B2A_live(real_b)
        rec_a = G_B2A_live(fake_b)
        rec_b = G_A2B_live(fake_a)
        id_a = G_B2A_live(real_a)
        id_b = G_A2B_live(real_b)

        adv_a2b = lsgan_real(D_B_live(aug_B(fake_b)))
        adv_b2a = lsgan_real(D_A_live(aug_A(fake_a)))
        cycle = l1(rec_a, real_a) + l1(rec_b, real_b)
        identity = l1(id_a, real_a) + l1(id_b, real_b)
        g_loss = adv_a2b + adv_b2a + CYCLE_WEIGHT * cycle + identity_weight * identity

        nonfinite = 0
        if torch.isfinite(g_loss):
            g_loss.backward()
            g_norm = torch.nn.utils.clip_grad_norm_(G_PARAMS, 10.0)
            if torch.isfinite(g_norm):
                opt_G.step()
                ema_update()
            else:
                nonfinite += 1
        else:
            nonfinite += 1

        set_requires_grad((D_A_live, D_B_live), True)
        d_a, bad_a = disc_step(D_A_live, opt_DA, aug_A, real_a, fake_a_buffer, fake_a)
        d_b, bad_b = disc_step(D_B_live, opt_DB, aug_B, real_b, fake_b_buffer, fake_b)
        return {
            "generator": float(g_loss.detach()),
            "cycle": float(cycle.detach()),
            "identity": float(identity.detach()),
            "discriminator_a": d_a,
            "discriminator_b": d_b,
            "nonfinite": nonfinite + bad_a + bad_b,
        }

    # Track independently, because the final TA average is separable by generator.
    source_p2m = direction_fid(G_B2A_live, "photo_to_monet")
    source_m2p = direction_fid(G_A2B_live, "monet_to_photo")
    best = {
        "photo_to_monet": {"fid": source_p2m, "variant": "source", "epoch": 0},
        "monet_to_photo": {"fid": source_m2p, "variant": "source", "epoch": 0},
    }
    best_states = {
        "photo_to_monet": cpu_state(G_B2A_live),
        "monet_to_photo": cpu_state(G_A2B_live),
    }
    no_improve = 0
    history = []

    print(f"Source fresh: P2M {source_p2m:.6f} | M2P {source_m2p:.6f} | AVG {(source_p2m + source_m2p)/2:.6f}")
    print(f"Start LR G {start_g_lr:.2e} | D_A ratio {D_A_RATIO:.2f} | D_B ratio {D_B_RATIO:.2f}")
    print(f"ADA start p_A {aug_A.p:.2f} | p_B {aug_B.p:.2f}")

    global_step = int(ckpt.get("global_step", 0))
    source_total_epoch = int(ckpt.get("epoch", 0) or 0)
    del ckpt

    for local_epoch in range(1, CONT_EPOCHS + 1):
        total_epoch = source_total_epoch + local_epoch
        g_lr, da_lr, db_lr, identity_weight = schedule(local_epoch)
        set_lrs(g_lr, da_lr, db_lr)
        G_A2B_live.train(); G_B2A_live.train(); D_A_live.train(); D_B_live.train()

        monet_order = list(range(len(monet_paths)))
        photo_order = list(range(len(photo_paths)))
        random.shuffle(monet_order); random.shuffle(photo_order)
        mi = pi = 0
        metrics_epoch = defaultdict(list)

        progress = tqdm(range(STEPS_PER_EPOCH), desc=f"EMA+ADA {local_epoch:03d}/{CONT_EPOCHS}")
        for step in progress:
            if mi >= len(monet_order):
                random.shuffle(monet_order); mi = 0
            if pi >= len(photo_order):
                random.shuffle(photo_order); pi = 0

            real_a = train_transform(monet_paths[monet_order[mi]]).unsqueeze(0).to(device, non_blocking=True)
            real_b = train_transform(photo_paths[photo_order[pi]]).unsqueeze(0).to(device, non_blocking=True)
            mi += 1; pi += 1

            metrics = train_step(real_a, real_b, identity_weight)
            global_step += 1
            for k, v in metrics.items():
                metrics_epoch[k].append(v)

            if step % 50 == 0 or step == STEPS_PER_EPOCH - 1:
                progress.set_postfix({
                    "G": f"{metrics['generator']:.3f}",
                    "cyc": f"{metrics['cycle']:.3f}",
                    "pA": f"{aug_A.p:.2f}",
                    "pB": f"{aug_B.p:.2f}",
                    "lr": f"{g_lr:.1e}",
                })

        if local_epoch % EVAL_EVERY != 0 and local_epoch != CONT_EPOCHS:
            continue

        results = {
            "live_p2m": direction_fid(G_B2A_live, "photo_to_monet"),
            "live_m2p": direction_fid(G_A2B_live, "monet_to_photo"),
            "ema_p2m": direction_fid(G_B2A_ema, "photo_to_monet"),
            "ema_m2p": direction_fid(G_A2B_ema, "monet_to_photo"),
        }

        improved = []
        candidates = [
            ("photo_to_monet", "live", results["live_p2m"], G_B2A_live),
            ("photo_to_monet", "ema", results["ema_p2m"], G_B2A_ema),
            ("monet_to_photo", "live", results["live_m2p"], G_A2B_live),
            ("monet_to_photo", "ema", results["ema_m2p"], G_A2B_ema),
        ]
        for direction, variant, fid, model in candidates:
            if fid < best[direction]["fid"]:
                best[direction] = {"fid": float(fid), "variant": variant, "epoch": total_epoch}
                best_states[direction] = cpu_state(model)
                improved.append(f"{direction}:{variant}")

        no_improve = 0 if improved else no_improve + 1
        best_mixed = (best["photo_to_monet"]["fid"] + best["monet_to_photo"]["fid"]) / 2.0
        live_avg = (results["live_p2m"] + results["live_m2p"]) / 2.0
        ema_avg = (results["ema_p2m"] + results["ema_m2p"]) / 2.0

        record = {
            "local_epoch": local_epoch, "total_epoch": total_epoch, "global_step": global_step,
            "g_lr": g_lr, "da_lr": da_lr, "db_lr": db_lr, "identity_weight": identity_weight,
            "ada_p_A": aug_A.p, "ada_p_B": aug_B.p,
            **results, "live_avg": live_avg, "ema_avg": ema_avg, "best_mixed_avg": best_mixed,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(CONT_DIR / "fid_history.csv", index=False)

        snapshot = {
            "epoch": total_epoch,
            "global_step": global_step,
            "G_A2B": cpu_state(G_A2B_live),
            "G_B2A": cpu_state(G_B2A_live),
            "G_A2B_EMA": cpu_state(G_A2B_ema),
            "G_B2A_EMA": cpu_state(G_B2A_ema),
            "D_A": cpu_state(D_A_live),
            "D_B": cpu_state(D_B_live),
            "ta_fid_photo_to_monet": results["live_p2m"],
            "ta_fid_monet_to_photo": results["live_m2p"],
            "ema_fid_photo_to_monet": results["ema_p2m"],
            "ema_fid_monet_to_photo": results["ema_m2p"],
            "generator_learning_rate": g_lr,
            "ada_p_A": aug_A.p,
            "ada_p_B": aug_B.p,
            "identity_weight": identity_weight,
        }
        torch.save(snapshot, SNAP_DIR / f"epoch_{total_epoch:03d}.pt")

        torch.save({
            "G_B2A": best_states["photo_to_monet"],
            "G_A2B": best_states["monet_to_photo"],
            "ta_fid_photo_to_monet": best["photo_to_monet"]["fid"],
            "ta_fid_monet_to_photo": best["monet_to_photo"]["fid"],
            "ta_fid_average": best_mixed,
            "selection": best,
        }, CONT_DIR / "best_per_direction.pt")

        print(
            f"[eval total {total_epoch}] live {live_avg:.6f} | ema {ema_avg:.6f} | "
            f"BEST MIXED {best_mixed:.6f} | pA {aug_A.p:.3f} pB {aug_B.p:.3f} | "
            f"improved {', '.join(improved) if improved else 'none'}"
        )

        if best_mixed < TARGET_AVG_FID:
            print(f"Target crossed on in-memory selector: {best_mixed:.6f} < {TARGET_AVG_FID}")
            break
        if no_improve >= PATIENCE:
            print(f"Early stop: no directional improvement for {PATIENCE} evaluations.")
            break

    return CONT_DIR / "best_per_direction.pt"


continuation_ran = False
if RUN_CONTINUATION_IF_NEEDED and phase1_summary["final_avg"] >= TARGET_AVG_FID:
    source = phase1_summary.get("best_trainable_pair")
    if source is None:
        default_source = ROOT / "strong_fresh_ttur" / "best" / "best_checkpoint.pt"
        if not default_source.exists():
            raise RuntimeError("No trainable source checkpoint with discriminators was found.")
        source_path = default_source
    else:
        source_path = Path(source["path"])
    run_continuation(source_path)
    continuation_ran = True
else:
    print("\nContinuation skipped because the mined in-memory average is already below target or continuation is disabled.")





PHASE 2: EMA + ASYMMETRIC ADA CONTINUATION
Source: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\best\best_checkpoint.pt


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(


Source fresh: P2M 99.146397 | M2P 102.283390 | AVG 100.714893
Start LR G 5.00e-05 | D_A ratio 0.30 | D_B ratio 0.50
ADA start p_A 0.50 | p_B 0.25


EMA+ADA 003/60: 100%|██████████| 1000/1000 [01:10<00:00, 14.10it/s, G=2.319, cyc=0.133, pA=0.37, pB=0.12, lr=5.0e-05]


[eval total 168] live 103.473791 | ema 101.244952 | BEST MIXED 100.601410 | pA 0.367 pB 0.115 | improved monet_to_photo:ema


...

[eval total 192] live 102.839340 | ema 103.234121 | BEST MIXED 100.601410 | pA 0.000 pB 0.000 | improved none


EMA+ADA 030/60: 100%|██████████| 1000/1000 [01:20<00:00, 12.50it/s, G=2.383, cyc=0.130, pA=0.00, pB=0.00, lr=2.8e-05]
C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(


[eval total 195] live 103.981401 | ema 103.945476 | BEST MIXED 100.601410 | pA 0.000 pB 0.000 | improved none


EMA+ADA 033/60: 100%|██████████| 1000/1000 [01:18<00:00, 12.68it/s, G=2.438, cyc=0.154, pA=0.00, pB=0.00, lr=2.5e-05]
C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(


[eval total 198] live 104.412531 | ema 103.647912 | BEST MIXED 100.601410 | pA 0.000 pB 0.000 | improved none
Early stop: no directional improvement for 10 evaluations.


### Phase 3: re-mine all old and newly generated checkpoints

In [25]:
# ============================================================================
# PHASE 3: RE-MINE, INCLUDING NEW LIVE + EMA SNAPSHOTS
# ============================================================================

phase3_summary, phase3_final = mine_checkpoints([ROOT], MINE2_DIR, "PHASE 3: FINAL FRESH RE-MINING")

# Copy canonical final checkpoint.
canonical_final = FINAL_DIR / "final_generators.pt"
shutil.copy2(phase3_summary["final_path"], canonical_final)

# Put final generators in notebook memory.
_final_ckpt = torch_load(canonical_final)
G_A2B, _ = build_generator(clean_state(_final_ckpt["G_A2B"]))
G_B2A, _ = build_generator(clean_state(_final_ckpt["G_B2A"]))
del _final_ckpt





PHASE 3: FINAL FRESH RE-MINING
Found 94 checkpoint files


Fresh checkpoint scan:   0%|          | 0/94 [00:00<?, ?it/s]C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(
Fresh checkpoint scan: 100%|██████████| 94/94 [34:41<00:00, 22.14s/it]


photo_to_monet: 92 unique fresh generators
monet_to_photo: 92 unique fresh generators

Best fresh singles
  Photo -> Monet: 98.583131
  Monet -> Photo: 100.929379
  Mixed average:  99.756255

photo_to_monet soup seed 1/8: FID 98.583131, epoch 155, dual_direction_ta_focus


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



photo_to_monet soup seed 2/8: FID 98.762600, epoch 160, dual_direction_ta_focus/continuation_155

photo_to_monet soup seed 3/8: FID 98.795643, epoch 175, reverse_focus_2p0

photo_to_monet soup seed 4/8: FID 99.036225, epoch 150, dual_direction_ta_focus

photo_to_monet soup seed 5/8: FID 99.105961, epoch 170, dual_direction_ta_focus/continuation_155

photo_to_monet soup seed 6/8: FID 99.146397, epoch 165, dual_direction_ta_focus/continuation_155

photo_to_monet soup seed 7/8: FID 99.256962, epoch 140, strong_fresh_ttur

photo_to_monet soup seed 8/8: FID 99.403592, epoch 105, continuation_v1

monet_to_photo soup seed 1/8: FID 100.929379, epoch 110, strong_fresh_ttur


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



monet_to_photo soup seed 2/8: FID 101.476010, epoch 130, strong_fresh_ttur

monet_to_photo soup seed 3/8: FID 101.921833, epoch 115, continuation_v1

monet_to_photo soup seed 4/8: FID 101.939640, epoch 130, identity_2p5_branch

monet_to_photo soup seed 5/8: FID 102.056422, epoch None, final_task3_pipeline/ema_ada_continue

monet_to_photo soup seed 6/8: FID 102.167933, epoch 140, strong_fresh_ttur

monet_to_photo soup seed 7/8: FID 102.283390, epoch 165, dual_direction_ta_focus/continuation_155

monet_to_photo soup seed 8/8: FID 102.318813, epoch 135, strong_fresh_ttur


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



------------------------------------------------------------------------------
MINING RESULT: P2M 98.583131 | M2P 100.412086 | AVG 99.497608
Saved: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\phase3_mining\final_generators.pt
------------------------------------------------------------------------------


In [28]:
# ============================================================================
# PHASE 3: FAST TARGETED FINAL RE-MINING
# ============================================================================

from pathlib import Path
import os
import shutil
import torch


print("=" * 78)
print("PHASE 3: FAST TARGETED FINAL RE-MINING")
print("=" * 78)


# ---------------------------------------------------------------------------
# 1. Build a small pool of promising checkpoints only
# ---------------------------------------------------------------------------

TARGET_POOL = ROOT / "_phase3_targeted_pool"

if TARGET_POOL.exists():
    shutil.rmtree(TARGET_POOL)

TARGET_POOL.mkdir(
    parents=True,
    exist_ok=True
)


# Strong epochs we actually care about.
# Includes neighboring epochs because one direction may be good even when
# the average score was not the best.
TARGET_EPOCHS = {
    100,
    110,
    115,
    120,
    125,
    130,
    135,
    140,
    145,
    150,
    155,
    160,
    165,
    170,
    175,
}


SEARCH_BRANCHES = [
    ROOT / "continuation_v1",
    ROOT / "identity_2p5_branch",
    ROOT / "photo_to_monet_focus",
    ROOT / "dual_direction_ta_focus",
    ROOT / "reverse_focus_2p0",
    ROOT / "strong_fresh_ttur",
    ROOT / "ema_ada_continue",
]


SEARCH_BRANCHES = [
    p for p in SEARCH_BRANCHES
    if p.exists()
]


print("Searching promising branches:")

for p in SEARCH_BRANCHES:
    print(" ", p)


# ---------------------------------------------------------------------------
# 2. Inspect checkpoints cheaply
#
# This is NOT running FID on all of them.
# It only reads metadata to decide which files should enter the small pool.
# ---------------------------------------------------------------------------

candidate_records = []


for branch in SEARCH_BRANCHES:

    files = sorted(
        set(
            list(branch.rglob("*.pt"))
            + list(branch.rglob("*.pth"))
        )
    )

    for path in files:

        keep = False
        epoch = None

        name_lower = path.name.lower()
        path_lower = str(path).lower()

        # Always keep explicitly saved "best" checkpoints.
        if (
            "best_checkpoint" in name_lower
            or path.parent.name.lower() == "best"
            or "best_per_direction" in name_lower
        ):
            keep = True

        # Read epoch metadata.
        try:

            payload = torch.load(
                path,
                map_location="cpu",
                weights_only=False,
            )

            if isinstance(payload, dict):

                raw_epoch = payload.get("epoch")

                if raw_epoch is not None:

                    try:
                        epoch = int(raw_epoch)
                    except Exception:
                        epoch = None

                if epoch in TARGET_EPOCHS:
                    keep = True

                # EMA continuation snapshots can be useful even if epoch
                # numbering does not exactly match the original runs.
                if (
                    "ema_ada" in path_lower
                    and (
                        "snapshot" in path_lower
                        or "best" in path_lower
                    )
                ):
                    keep = True

            del payload

        except Exception as error:

            print(
                "Skipping unreadable checkpoint:",
                path.name,
                "|",
                error
            )

            continue


        if keep:

            candidate_records.append(
                {
                    "path": path.resolve(),
                    "epoch": epoch,
                    "branch": branch.name,
                }
            )


# ---------------------------------------------------------------------------
# 3. Remove duplicate paths
# ---------------------------------------------------------------------------

unique = {}

for record in candidate_records:

    unique[str(record["path"])] = record

candidate_records = list(
    unique.values()
)


# ---------------------------------------------------------------------------
# 4. Keep the pool reasonably small
#
# Prioritize:
#   best files
#   epoch 165
#   epoch 150
#   epoch 175
#   epoch 130
#   then other nearby useful epochs
# ---------------------------------------------------------------------------

priority_epochs = {
    165: 0,
    150: 1,
    175: 2,
    130: 3,
    115: 4,
    170: 5,
    160: 6,
    155: 7,
    145: 8,
    140: 9,
    135: 10,
    125: 11,
    120: 12,
    110: 13,
    100: 14,
}


def candidate_priority(record):

    path = record["path"]

    best_bonus = (
        -100
        if (
            "best_checkpoint" in path.name.lower()
            or path.parent.name.lower() == "best"
            or "best_per_direction" in path.name.lower()
        )
        else 0
    )

    epoch_priority = priority_epochs.get(
        record["epoch"],
        999
    )

    return (
        best_bonus,
        epoch_priority,
        str(path),
    )


candidate_records = sorted(
    candidate_records,
    key=candidate_priority,
)


# Limit the expensive FID stage.
MAX_TARGET_CHECKPOINTS = 18

candidate_records = (
    candidate_records[
        :MAX_TARGET_CHECKPOINTS
    ]
)


print()
print(
    "Checkpoints entering fresh Phase 3:",
    len(candidate_records)
)

for record in candidate_records:

    print(
        f"epoch={str(record['epoch']):>4} | "
        f"{record['branch']:<28} | "
        f"{record['path'].name}"
    )


if not candidate_records:

    raise RuntimeError(
        "No targeted checkpoints were found."
    )


# ---------------------------------------------------------------------------
# 5. Hard-link selected checkpoints into a tiny temporary directory
#
# Hard links do NOT duplicate the large checkpoint data.
# If a hard link fails, fall back to copying.
# ---------------------------------------------------------------------------

for i, record in enumerate(
    candidate_records
):

    source = record["path"]

    destination = (
        TARGET_POOL
        / (
            f"{i:02d}_"
            f"{record['branch']}_"
            f"epoch_{record['epoch']}_"
            f"{source.name}"
        )
    )

    try:

        os.link(
            source,
            destination
        )

    except Exception:

        shutil.copy2(
            source,
            destination
        )


print()
print(
    "Target pool ready:",
    TARGET_POOL
)

print(
    "Files:",
    len(
        list(
            TARGET_POOL.glob("*.pt")
        )
        + list(
            TARGET_POOL.glob("*.pth")
        )
    )
)


# ---------------------------------------------------------------------------
# 6. Run your EXISTING miner only on this targeted pool
# ---------------------------------------------------------------------------

phase3_summary, phase3_final = mine_checkpoints(
    [TARGET_POOL],
    MINE2_DIR,
    "PHASE 3: FAST TARGETED FINAL RE-MINING"
)


# ---------------------------------------------------------------------------
# 7. Save canonical final checkpoint
# ---------------------------------------------------------------------------

canonical_final = (
    FINAL_DIR
    / "final_generators.pt"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy2(
    phase3_summary["final_path"],
    canonical_final
)


# ---------------------------------------------------------------------------
# 8. Put selected final generators into notebook memory
# ---------------------------------------------------------------------------

_final_ckpt = torch_load(
    canonical_final
)

G_A2B, _ = build_generator(
    clean_state(
        _final_ckpt["G_A2B"]
    )
)

G_B2A, _ = build_generator(
    clean_state(
        _final_ckpt["G_B2A"]
    )
)

del _final_ckpt


# ---------------------------------------------------------------------------
# 9. Print result
# ---------------------------------------------------------------------------

print()
print("=" * 78)
print("TARGETED PHASE 3 COMPLETE")
print("=" * 78)

print(
    "Final checkpoint:",
    canonical_final
)

if "final_avg" in phase3_summary:

    print(
        "In-memory final average FID:",
        phase3_summary["final_avg"]
    )

elif "final_fid" in phase3_summary:

    print(
        "Final FID:",
        phase3_summary["final_fid"]
    )

print()
print(
    "IMPORTANT: run the literal JPEG verification cell again now."
)

print(
    "Current literal baseline to beat: 98.044191"
)

print("=" * 78)

PHASE 3: FAST TARGETED FINAL RE-MINING
Searching promising branches:
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\continuation_v1
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\identity_2p5_branch
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\photo_to_monet_focus
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\reverse_focus_2p0
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\strong_fresh_ttur

Checkpoints entering fresh Phase 3: 18
epoch= 165 | dual_direction_ta_focus      | best_checkpoint.pt
epoch= 150 | photo_to_monet_focus         | best_checkpoint.pt
epoch= 130 | identity_2p5_branch          | best_checkpoint.pt
epoch= 115 | continuation_v1              | best_checkpoint.pt
epoch= 115 |

Fresh checkpoint scan:   0%|          | 0/18 [00:00<?, ?it/s]C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(
Fresh checkpoint scan: 100%|██████████| 18/18 [04:06<00:00, 13.68s/it]


photo_to_monet: 10 unique fresh generators
monet_to_photo: 10 unique fresh generators

Best fresh singles
  Photo -> Monet: 98.583131
  Monet -> Photo: 101.476010
  Mixed average:  100.029571

photo_to_monet soup seed 1/8: FID 98.583131, epoch 155, _phase3_targeted_pool


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



photo_to_monet soup seed 2/8: FID 98.795643, epoch 175, _phase3_targeted_pool

photo_to_monet soup seed 3/8: FID 99.036225, epoch 150, _phase3_targeted_pool

photo_to_monet soup seed 4/8: FID 99.146397, epoch 165, _phase3_targeted_pool

photo_to_monet soup seed 5/8: FID 99.256962, epoch 140, _phase3_targeted_pool

photo_to_monet soup seed 6/8: FID 99.679131, epoch 130, _phase3_targeted_pool

photo_to_monet soup seed 7/8: FID 99.845193, epoch 115, _phase3_targeted_pool

photo_to_monet soup seed 8/8: FID 100.236959, epoch 150, _phase3_targeted_pool

monet_to_photo soup seed 1/8: FID 101.476010, epoch 130, _phase3_targeted_pool


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



monet_to_photo soup seed 2/8: FID 101.921833, epoch 115, _phase3_targeted_pool

monet_to_photo soup seed 3/8: FID 101.939640, epoch 130, _phase3_targeted_pool

monet_to_photo soup seed 4/8: FID 102.167933, epoch 140, _phase3_targeted_pool

monet_to_photo soup seed 5/8: FID 102.283390, epoch 165, _phase3_targeted_pool

monet_to_photo soup seed 6/8: FID 103.049797, epoch 155, _phase3_targeted_pool

monet_to_photo soup seed 7/8: FID 103.414543, epoch 150, _phase3_targeted_pool

monet_to_photo soup seed 8/8: FID 103.621277, epoch 150, _phase3_targeted_pool


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



------------------------------------------------------------------------------
MINING RESULT: P2M 98.483042 | M2P 101.476010 | AVG 99.979526
Saved: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\phase3_mining\final_generators.pt
------------------------------------------------------------------------------

TARGETED PHASE 3 COMPLETE
Final checkpoint: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt
In-memory final average FID: 99.97952619876085

IMPORTANT: run the literal JPEG verification cell again now.
Current literal baseline to beat: 98.044191


### Final literal JPEG TA-style verification

In [27]:
# ============================================================================
# CELL 28 REPLACEMENT
# LOAD BEST CHECKPOINT + LITERAL JPEG TA-STYLE VERIFICATION
# ============================================================================

from pathlib import Path
import shutil
import json

import numpy as np
import torch
import torch.nn as nn
import scipy.linalg

from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    Inception_V3_Weights,
    inception_v3,
)


# ============================================================================
# SETTINGS
# ============================================================================

N_EVAL = 300
JPEG_QUALITY = 95

PROJECT_ROOT = Path(
    r"C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "task3_gan"
    / "drashti"
    / "experiment_v2"
    / "dual_direction_ta_focus"
    / "continuation_155"
    / "best"
    / "best_checkpoint.pt"
)

TA_DIR = (
    PROJECT_ROOT
    / "task3_gan"
    / "drashti"
    / "final_ta_verification"
)

TA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================================
# CHECK REQUIRED NOTEBOOK OBJECTS
# ============================================================================

required = [
    "Generator",
    "device",
    "monet_paths",
    "photo_paths",
    "eval_transform",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required notebook objects: "
        + ", ".join(missing)
        + "\nRun the earlier setup/model/data cells first."
    )


# ============================================================================
# LOAD BEST KNOWN CHECKPOINT
# ============================================================================

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT_PATH}"
    )

print("=" * 78)
print("LOADING CHECKPOINT")
print("=" * 78)
print(CHECKPOINT_PATH)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

G_A2B = Generator(
    residual_blocks=9,
    base_channels=64,
).to(device)

G_B2A = Generator(
    residual_blocks=9,
    base_channels=64,
).to(device)

G_A2B.load_state_dict(
    checkpoint["G_A2B"]
)

G_B2A.load_state_dict(
    checkpoint["G_B2A"]
)

G_A2B.eval()
G_B2A.eval()

print(
    "Epoch:",
    checkpoint.get("epoch", "unknown")
)

print("G_A2B: Monet -> Photo")
print("G_B2A: Photo -> Monet")


# ============================================================================
# IMAGE CONVERSION
# ============================================================================

def tensor_to_pil(x):

    x = (
        x.detach()
        .cpu()
        .clamp(-1, 1)
    )

    x = (
        (x + 1.0)
        * 127.5
    ).round().clamp(
        0,
        255
    ).byte()

    x = (
        x.permute(
            1,
            2,
            0
        )
        .numpy()
    )

    return Image.fromarray(x)


# ============================================================================
# LITERAL TA STYLE VERIFICATION
# ============================================================================

def literal_ta_verify(
    g_a2b,
    g_b2a
):

    pred_a2b = (
        TA_DIR
        / "pred_A2B"
    )

    pred_b2a = (
        TA_DIR
        / "pred_B2A"
    )

    # Start clean
    if pred_a2b.exists():
        shutil.rmtree(
            pred_a2b
        )

    if pred_b2a.exists():
        shutil.rmtree(
            pred_b2a
        )

    pred_a2b.mkdir(
        parents=True,
        exist_ok=True
    )

    pred_b2a.mkdir(
        parents=True,
        exist_ok=True
    )

    g_a2b.eval()
    g_b2a.eval()


    # ------------------------------------------------------------------------
    # Generate Monet -> Photo
    # ------------------------------------------------------------------------

    with torch.no_grad():

        for i, path in enumerate(
            tqdm(
                list(monet_paths)[:N_EVAL],
                desc="JPEG Monet -> Photo"
            )
        ):

            x = (
                eval_transform(path)
                .unsqueeze(0)
                .to(device)
            )

            y = g_a2b(x)[0]

            tensor_to_pil(
                y
            ).save(
                pred_a2b
                / f"{i:04d}.jpg",
                format="JPEG",
                quality=JPEG_QUALITY,
            )


    # ------------------------------------------------------------------------
    # Generate Photo -> Monet
    # ------------------------------------------------------------------------

    with torch.no_grad():

        for i, path in enumerate(
            tqdm(
                list(photo_paths)[:N_EVAL],
                desc="JPEG Photo -> Monet"
            )
        ):

            x = (
                eval_transform(path)
                .unsqueeze(0)
                .to(device)
            )

            y = g_b2a(x)[0]

            tensor_to_pil(
                y
            ).save(
                pred_b2a
                / f"{i:04d}.jpg",
                format="JPEG",
                quality=JPEG_QUALITY,
            )


    print()
    print(
        "Generated A2B:",
        len(
            list(
                pred_a2b.glob(
                    "*.jpg"
                )
            )
        )
    )

    print(
        "Generated B2A:",
        len(
            list(
                pred_b2a.glob(
                    "*.jpg"
                )
            )
        )
    )


    # =========================================================================
    # INCEPTION FEATURE EXTRACTOR
    # =========================================================================

    print()
    print("Loading Inception V3...")

    inc = inception_v3(
        weights=(
            Inception_V3_Weights
            .IMAGENET1K_V1
        )
    )

    inc.fc = nn.Identity()

    inc = (
        inc.to(device)
        .eval()
    )


    tfm = transforms.Compose([
        transforms.Resize(
            299
        ),

        transforms.CenterCrop(
            299
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],
            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


    class PathsDataset(
        Dataset
    ):

        def __init__(
            self,
            paths
        ):
            self.paths = list(
                paths
            )

        def __len__(
            self
        ):
            return len(
                self.paths
            )

        def __getitem__(
            self,
            idx
        ):

            with Image.open(
                self.paths[idx]
            ) as image:

                return tfm(
                    image.convert(
                        "RGB"
                    )
                )


    @torch.no_grad()
    def features(
        paths,
        label
    ):

        loader = DataLoader(
            PathsDataset(
                paths
            ),
            batch_size=32,
            shuffle=False,
            num_workers=0,
        )

        output = []

        for batch in tqdm(
            loader,
            desc=label
        ):

            feats = inc(
                batch.to(device)
            )

            output.append(
                feats
                .detach()
                .cpu()
                .numpy()
            )

        return (
            np.concatenate(
                output,
                axis=0
            )
            .astype(
                np.float64
            )
        )


    # =========================================================================
    # FID
    # =========================================================================

    def fid_features(
        a,
        b
    ):

        mu1 = np.mean(
            a,
            axis=0
        )

        mu2 = np.mean(
            b,
            axis=0
        )

        sigma1 = np.cov(
            a,
            rowvar=False
        )

        sigma2 = np.cov(
            b,
            rowvar=False
        )

        diff = (
            mu1
            - mu2
        )

        # No deprecated disp argument
        covmean = scipy.linalg.sqrtm(
            sigma1.dot(
                sigma2
            )
        )

        if not np.isfinite(
            covmean
        ).all():

            eps = 1e-6

            offset = (
                np.eye(
                    sigma1.shape[0]
                )
                * eps
            )

            covmean = scipy.linalg.sqrtm(
                (
                    sigma1
                    + offset
                ).dot(
                    sigma2
                    + offset
                )
            )

        if np.iscomplexobj(
            covmean
        ):
            covmean = (
                covmean.real
            )

        fid = (
            diff.dot(
                diff
            )
            + np.trace(
                sigma1
            )
            + np.trace(
                sigma2
            )
            - 2.0
            * np.trace(
                covmean
            )
        )

        return float(
            np.real(
                fid
            )
        )


    # =========================================================================
    # TA STYLE MiFID COSINE DISTANCE
    # =========================================================================

    def mean_cosine_distance(
        a,
        b
    ):

        n = min(
            len(a),
            len(b)
        )

        a = a[:n]
        b = b[:n]

        denominator = (
            np.linalg.norm(
                a,
                axis=1
            )
            * np.linalg.norm(
                b,
                axis=1
            )
        )

        denominator = np.maximum(
            denominator,
            1e-12
        )

        cosine_similarity = (
            np.sum(
                a * b,
                axis=1
            )
            / denominator
        )

        return float(
            np.mean(
                1.0
                - cosine_similarity
            )
        )


    # =========================================================================
    # EXTRACT FEATURES
    # =========================================================================

    print()
    print("=" * 78)
    print("EXTRACTING TA FEATURES")
    print("=" * 78)

    real_monet = features(
        list(
            monet_paths
        )[:N_EVAL],
        "Real Monet"
    )

    real_photo = features(
        list(
            photo_paths
        )[:N_EVAL],
        "Real Photo"
    )

    fake_photo = features(
        sorted(
            pred_a2b.glob(
                "*.jpg"
            )
        ),
        "Generated Photo"
    )

    fake_monet = features(
        sorted(
            pred_b2a.glob(
                "*.jpg"
            )
        ),
        "Generated Monet"
    )


    # =========================================================================
    # COMPUTE SCORES
    # =========================================================================

    print()
    print("Computing FID...")

    photo_to_monet_fid = (
        fid_features(
            real_monet,
            fake_monet
        )
    )

    monet_to_photo_fid = (
        fid_features(
            real_photo,
            fake_photo
        )
    )

    photo_to_monet_mifid = (
        mean_cosine_distance(
            real_monet,
            fake_monet
        )
    )

    monet_to_photo_mifid = (
        mean_cosine_distance(
            real_photo,
            fake_photo
        )
    )


    result = {

        "photo_to_monet_fid":
            float(
                photo_to_monet_fid
            ),

        "monet_to_photo_fid":
            float(
                monet_to_photo_fid
            ),

        "average_fid":
            float(
                (
                    photo_to_monet_fid
                    + monet_to_photo_fid
                )
                / 2.0
            ),

        "photo_to_monet_mifid":
            float(
                photo_to_monet_mifid
            ),

        "monet_to_photo_mifid":
            float(
                monet_to_photo_mifid
            ),

        "average_mifid":
            float(
                (
                    photo_to_monet_mifid
                    + monet_to_photo_mifid
                )
                / 2.0
            ),
    }


    with open(
        TA_DIR
        / "literal_ta_result.json",
        "w"
    ) as file:

        json.dump(
            result,
            file,
            indent=2
        )


    return result


# ============================================================================
# RUN
# ============================================================================

print()
print("=" * 78)
print("FINAL LITERAL JPEG TA-STYLE VERIFICATION")
print("=" * 78)

literal_result = literal_ta_verify(
    G_A2B,
    G_B2A
)


# ============================================================================
# FINAL OUTPUT
# ============================================================================

print()
print("=" * 78)
print("TA EVALUATION RESULTS")
print("=" * 78)

print(
    f"Photo -> Monet FID:  "
    f"{literal_result['photo_to_monet_fid']:.6f}"
)

print(
    f"Monet -> Photo FID:  "
    f"{literal_result['monet_to_photo_fid']:.6f}"
)

print(
    f"Average FID:          "
    f"{literal_result['average_fid']:.6f}"
)

print()

print(
    f"Photo -> Monet MiFID: "
    f"{literal_result['photo_to_monet_mifid']:.6f}"
)

print(
    f"Monet -> Photo MiFID: "
    f"{literal_result['monet_to_photo_mifid']:.6f}"
)

print(
    f"Average MiFID:        "
    f"{literal_result['average_mifid']:.6f}"
)

print()
print(
    "Saved result:",
    TA_DIR
    / "literal_ta_result.json"
)

print("=" * 78)

LOADING CHECKPOINT
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\best\best_checkpoint.pt
Epoch: 165
G_A2B: Monet -> Photo
G_B2A: Photo -> Monet

FINAL LITERAL JPEG TA-STYLE VERIFICATION


JPEG Photo -> Monet: 100%|██████████| 300/300 [00:01<00:00, 155.89it/s]



Generated A2B: 300
Generated B2A: 300

Loading Inception V3...

EXTRACTING TA FEATURES


Generated Monet: 100%|██████████| 10/10 [00:00<00:00, 14.57it/s]



Computing FID...

TA EVALUATION RESULTS
Photo -> Monet FID:  94.690434
Monet -> Photo FID:  101.397948
Average FID:          98.044191

Photo -> Monet MiFID: 0.404998
Monet -> Photo MiFID: 0.416858
Average MiFID:        0.410928

Saved result: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\final_ta_verification\literal_ta_result.json


In [29]:
# ============================================================================
# PHASE 3 FINAL LITERAL JPEG TA-STYLE VERIFICATION
# Uses G_A2B and G_B2A ALREADY LOADED BY PHASE 3
# DOES NOT reload epoch 165
# ============================================================================

from pathlib import Path
import shutil
import json

import numpy as np
import torch
import torch.nn as nn
import scipy.linalg

from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    Inception_V3_Weights,
    inception_v3,
)


# ============================================================================
# SETTINGS
# ============================================================================

N_EVAL = 300
JPEG_QUALITY = 95

PROJECT_ROOT = Path(
    r"C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU"
)

# Separate folder so we preserve the old epoch-165 evaluation.
TA_DIR = (
    PROJECT_ROOT
    / "task3_gan"
    / "drashti"
    / "final_ta_verification_phase3"
)

TA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================================
# CHECK THAT PHASE 3 GENERATORS ARE STILL IN MEMORY
# ============================================================================

required = [
    "device",
    "monet_paths",
    "photo_paths",
    "eval_transform",
    "G_A2B",
    "G_B2A",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required notebook objects: "
        + ", ".join(missing)
        + "\nRun the Phase 3 cell immediately before this cell."
    )


print("=" * 78)
print("USING PHASE 3 GENERATORS ALREADY LOADED IN MEMORY")
print("=" * 78)

G_A2B = G_A2B.to(device).eval()
G_B2A = G_B2A.to(device).eval()

print("G_A2B: Monet -> Photo")
print("G_B2A: Photo -> Monet")

if "canonical_final" in globals():
    print("Phase 3 checkpoint:", canonical_final)

print()
print("Previous literal baseline to beat:")
print("Photo -> Monet:  94.690434")
print("Monet -> Photo: 101.397943")
print("Average FID:      98.044191")


# ============================================================================
# IMAGE CONVERSION
# ============================================================================

def tensor_to_pil(x):

    x = (
        x.detach()
        .cpu()
        .clamp(-1, 1)
    )

    x = (
        (x + 1.0)
        * 127.5
    ).round().clamp(
        0,
        255
    ).byte()

    x = (
        x.permute(
            1,
            2,
            0
        )
        .numpy()
    )

    return Image.fromarray(x)


# ============================================================================
# LITERAL TA STYLE VERIFICATION
# ============================================================================

def literal_ta_verify(
    g_a2b,
    g_b2a
):

    pred_a2b = (
        TA_DIR
        / "pred_A2B"
    )

    pred_b2a = (
        TA_DIR
        / "pred_B2A"
    )

    # Start clean
    if pred_a2b.exists():
        shutil.rmtree(pred_a2b)

    if pred_b2a.exists():
        shutil.rmtree(pred_b2a)

    pred_a2b.mkdir(
        parents=True,
        exist_ok=True
    )

    pred_b2a.mkdir(
        parents=True,
        exist_ok=True
    )

    g_a2b.eval()
    g_b2a.eval()


    # ========================================================================
    # GENERATE MONET -> PHOTO
    # ========================================================================

    with torch.no_grad():

        for i, path in enumerate(
            tqdm(
                list(monet_paths)[:N_EVAL],
                desc="JPEG Monet -> Photo"
            )
        ):

            x = (
                eval_transform(path)
                .unsqueeze(0)
                .to(device)
            )

            y = g_a2b(x)[0]

            tensor_to_pil(
                y
            ).save(
                pred_a2b
                / f"{i:04d}.jpg",
                format="JPEG",
                quality=JPEG_QUALITY,
            )


    # ========================================================================
    # GENERATE PHOTO -> MONET
    # ========================================================================

    with torch.no_grad():

        for i, path in enumerate(
            tqdm(
                list(photo_paths)[:N_EVAL],
                desc="JPEG Photo -> Monet"
            )
        ):

            x = (
                eval_transform(path)
                .unsqueeze(0)
                .to(device)
            )

            y = g_b2a(x)[0]

            tensor_to_pil(
                y
            ).save(
                pred_b2a
                / f"{i:04d}.jpg",
                format="JPEG",
                quality=JPEG_QUALITY,
            )


    print()
    print(
        "Generated A2B:",
        len(list(pred_a2b.glob("*.jpg")))
    )

    print(
        "Generated B2A:",
        len(list(pred_b2a.glob("*.jpg")))
    )


    # ========================================================================
    # INCEPTION FEATURE EXTRACTOR
    # ========================================================================

    print()
    print("Loading Inception V3...")

    inc = inception_v3(
        weights=(
            Inception_V3_Weights
            .IMAGENET1K_V1
        )
    )

    inc.fc = nn.Identity()

    inc = (
        inc.to(device)
        .eval()
    )


    tfm = transforms.Compose([
        transforms.Resize(
            299
        ),

        transforms.CenterCrop(
            299
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],
            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


    class PathsDataset(
        Dataset
    ):

        def __init__(
            self,
            paths
        ):
            self.paths = list(paths)

        def __len__(
            self
        ):
            return len(self.paths)

        def __getitem__(
            self,
            idx
        ):

            with Image.open(
                self.paths[idx]
            ) as image:

                return tfm(
                    image.convert("RGB")
                )


    @torch.no_grad()
    def features(
        paths,
        label
    ):

        loader = DataLoader(
            PathsDataset(paths),
            batch_size=32,
            shuffle=False,
            num_workers=0,
        )

        output = []

        for batch in tqdm(
            loader,
            desc=label
        ):

            feats = inc(
                batch.to(device)
            )

            output.append(
                feats
                .detach()
                .cpu()
                .numpy()
            )

        return (
            np.concatenate(
                output,
                axis=0
            )
            .astype(np.float64)
        )


    # ========================================================================
    # FID
    # ========================================================================

    def fid_features(
        a,
        b
    ):

        mu1 = np.mean(
            a,
            axis=0
        )

        mu2 = np.mean(
            b,
            axis=0
        )

        sigma1 = np.cov(
            a,
            rowvar=False
        )

        sigma2 = np.cov(
            b,
            rowvar=False
        )

        diff = (
            mu1
            - mu2
        )

        covmean = scipy.linalg.sqrtm(
            sigma1.dot(
                sigma2
            )
        )

        if not np.isfinite(
            covmean
        ).all():

            eps = 1e-6

            offset = (
                np.eye(
                    sigma1.shape[0]
                )
                * eps
            )

            covmean = scipy.linalg.sqrtm(
                (
                    sigma1
                    + offset
                ).dot(
                    sigma2
                    + offset
                )
            )

        if np.iscomplexobj(
            covmean
        ):
            covmean = covmean.real

        fid = (
            diff.dot(diff)
            + np.trace(sigma1)
            + np.trace(sigma2)
            - 2.0
            * np.trace(covmean)
        )

        return float(
            np.real(fid)
        )


    # ========================================================================
    # TA STYLE MiFID COSINE DISTANCE
    # ========================================================================

    def mean_cosine_distance(
        a,
        b
    ):

        n = min(
            len(a),
            len(b)
        )

        a = a[:n]
        b = b[:n]

        denominator = (
            np.linalg.norm(
                a,
                axis=1
            )
            * np.linalg.norm(
                b,
                axis=1
            )
        )

        denominator = np.maximum(
            denominator,
            1e-12
        )

        cosine_similarity = (
            np.sum(
                a * b,
                axis=1
            )
            / denominator
        )

        return float(
            np.mean(
                1.0
                - cosine_similarity
            )
        )


    # ========================================================================
    # EXTRACT FEATURES
    # ========================================================================

    print()
    print("=" * 78)
    print("EXTRACTING TA FEATURES")
    print("=" * 78)

    real_monet = features(
        list(monet_paths)[:N_EVAL],
        "Real Monet"
    )

    real_photo = features(
        list(photo_paths)[:N_EVAL],
        "Real Photo"
    )

    fake_photo = features(
        sorted(
            pred_a2b.glob("*.jpg")
        ),
        "Generated Photo"
    )

    fake_monet = features(
        sorted(
            pred_b2a.glob("*.jpg")
        ),
        "Generated Monet"
    )


    # ========================================================================
    # COMPUTE SCORES
    # ========================================================================

    print()
    print("Computing FID...")

    photo_to_monet_fid = (
        fid_features(
            real_monet,
            fake_monet
        )
    )

    monet_to_photo_fid = (
        fid_features(
            real_photo,
            fake_photo
        )
    )

    photo_to_monet_mifid = (
        mean_cosine_distance(
            real_monet,
            fake_monet
        )
    )

    monet_to_photo_mifid = (
        mean_cosine_distance(
            real_photo,
            fake_photo
        )
    )


    result = {

        "photo_to_monet_fid":
            float(
                photo_to_monet_fid
            ),

        "monet_to_photo_fid":
            float(
                monet_to_photo_fid
            ),

        "average_fid":
            float(
                (
                    photo_to_monet_fid
                    + monet_to_photo_fid
                )
                / 2.0
            ),

        "photo_to_monet_mifid":
            float(
                photo_to_monet_mifid
            ),

        "monet_to_photo_mifid":
            float(
                monet_to_photo_mifid
            ),

        "average_mifid":
            float(
                (
                    photo_to_monet_mifid
                    + monet_to_photo_mifid
                )
                / 2.0
            ),
    }


    with open(
        TA_DIR
        / "literal_ta_result.json",
        "w"
    ) as file:

        json.dump(
            result,
            file,
            indent=2
        )

    return result


# ============================================================================
# RUN PHASE 3 LITERAL VERIFICATION
# ============================================================================

print()
print("=" * 78)
print("PHASE 3 LITERAL JPEG TA-STYLE VERIFICATION")
print("=" * 78)

literal_result_phase3 = literal_ta_verify(
    G_A2B,
    G_B2A
)


# ============================================================================
# FINAL OUTPUT
# ============================================================================

print()
print("=" * 78)
print("PHASE 3 TA EVALUATION RESULTS")
print("=" * 78)

print(
    f"Photo -> Monet FID:  "
    f"{literal_result_phase3['photo_to_monet_fid']:.6f}"
)

print(
    f"Monet -> Photo FID:  "
    f"{literal_result_phase3['monet_to_photo_fid']:.6f}"
)

print(
    f"Average FID:          "
    f"{literal_result_phase3['average_fid']:.6f}"
)

print()

print(
    f"Photo -> Monet MiFID: "
    f"{literal_result_phase3['photo_to_monet_mifid']:.6f}"
)

print(
    f"Monet -> Photo MiFID: "
    f"{literal_result_phase3['monet_to_photo_mifid']:.6f}"
)

print(
    f"Average MiFID:        "
    f"{literal_result_phase3['average_mifid']:.6f}"
)


# ============================================================================
# COMPARE AGAINST OLD LITERAL RESULT
# ============================================================================

OLD_P2M = 94.690434
OLD_M2P = 101.397943
OLD_AVG = 98.044191

NEW_P2M = literal_result_phase3[
    "photo_to_monet_fid"
]

NEW_M2P = literal_result_phase3[
    "monet_to_photo_fid"
]

NEW_AVG = literal_result_phase3[
    "average_fid"
]

print()
print("=" * 78)
print("OLD VS PHASE 3")
print("=" * 78)

print(
    f"{'':22}"
    f"{'OLD':>12}"
    f"{'PHASE 3':>12}"
    f"{'CHANGE':>12}"
)

print(
    f"{'Photo -> Monet':22}"
    f"{OLD_P2M:12.6f}"
    f"{NEW_P2M:12.6f}"
    f"{NEW_P2M - OLD_P2M:12.6f}"
)

print(
    f"{'Monet -> Photo':22}"
    f"{OLD_M2P:12.6f}"
    f"{NEW_M2P:12.6f}"
    f"{NEW_M2P - OLD_M2P:12.6f}"
)

print(
    f"{'Average':22}"
    f"{OLD_AVG:12.6f}"
    f"{NEW_AVG:12.6f}"
    f"{NEW_AVG - OLD_AVG:12.6f}"
)

print()

if NEW_AVG < OLD_AVG:

    print(
        "RESULT: PHASE 3 IS BETTER"
    )

    print(
        f"Improvement: "
        f"{OLD_AVG - NEW_AVG:.6f} FID"
    )

else:

    print(
        "RESULT: OLD EPOCH-165 PAIR IS BETTER"
    )

    print(
        f"Phase 3 is worse by: "
        f"{NEW_AVG - OLD_AVG:.6f} FID"
    )


print()
print(
    "Saved result:",
    TA_DIR
    / "literal_ta_result.json"
)

print("=" * 78)

USING PHASE 3 GENERATORS ALREADY LOADED IN MEMORY
G_A2B: Monet -> Photo
G_B2A: Photo -> Monet
Phase 3 checkpoint: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt

Previous literal baseline to beat:
Photo -> Monet:  94.690434
Monet -> Photo: 101.397943
Average FID:      98.044191

PHASE 3 LITERAL JPEG TA-STYLE VERIFICATION


JPEG Photo -> Monet: 100%|██████████| 300/300 [00:02<00:00, 114.71it/s]



Generated A2B: 300
Generated B2A: 300

Loading Inception V3...

EXTRACTING TA FEATURES


Generated Monet: 100%|██████████| 10/10 [00:00<00:00, 12.42it/s]



Computing FID...

PHASE 3 TA EVALUATION RESULTS
Photo -> Monet FID:  94.357201
Monet -> Photo FID:  100.383239
Average FID:          97.370220

Photo -> Monet MiFID: 0.405993
Monet -> Photo MiFID: 0.417622
Average MiFID:        0.411808

OLD VS PHASE 3
                               OLD     PHASE 3      CHANGE
Photo -> Monet           94.690434   94.357201   -0.333233
Monet -> Photo          101.397943  100.383239   -1.014704
Average                  98.044191   97.370220   -0.673971

RESULT: PHASE 3 IS BETTER
Improvement: 0.673971 FID

Saved result: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\final_ta_verification_phase3\literal_ta_result.json


In [32]:
# ============================================================================
# FAST LITERAL-JPEG TOURNAMENT
# NO CSV DEPENDENCY
# Evaluates a small handpicked set of strong checkpoints
# ============================================================================

from pathlib import Path
import shutil
import gc

import numpy as np
import torch
import torch.nn as nn
import scipy.linalg

from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    Inception_V3_Weights,
    inception_v3,
)


# ============================================================================
# SETTINGS
# ============================================================================

N_EVAL = 300
JPEG_QUALITY = 95

CURRENT_BEST_AVG = 97.370220

PROJECT_ROOT = Path(
    r"C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU"
)

EXP_ROOT = (
    PROJECT_ROOT
    / "task3_gan"
    / "drashti"
    / "experiment_v2"
)

TOURNAMENT_DIR = (
    EXP_ROOT
    / "literal_tournament"
)

TOURNAMENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================================
# HANDPICKED STRONG CHECKPOINTS
# ============================================================================

candidate_paths = [

    # Current Phase 3 final pair
    Path(canonical_final),

    # Epoch 165 best
    EXP_ROOT
    / "dual_direction_ta_focus"
    / "continuation_155"
    / "best"
    / "best_checkpoint.pt",

    # Identity branch
    EXP_ROOT
    / "identity_2p5_branch"
    / "best"
    / "best_checkpoint.pt",

    # Photo -> Monet focus
    EXP_ROOT
    / "photo_to_monet_focus"
    / "best"
    / "best_checkpoint.pt",

    # Low-LR continuation
    EXP_ROOT
    / "continuation_v1"
    / "best"
    / "best_checkpoint.pt",

    # Strong fresh TTUR
    EXP_ROOT
    / "strong_fresh_ttur"
    / "best"
    / "best_checkpoint.pt",
]


# Optional extra checkpoints if they exist
extra_candidates = [

    EXP_ROOT
    / "reverse_focus_2p0"
    / "checkpoints"
    / "epoch_175.pt",

    EXP_ROOT
    / "dual_direction_ta_focus"
    / "continuation_155"
    / "checkpoints"
    / "epoch_160.pt",

    EXP_ROOT
    / "dual_direction_ta_focus"
    / "continuation_155"
    / "checkpoints"
    / "epoch_170.pt",

    EXP_ROOT
    / "identity_2p5_branch"
    / "checkpoints"
    / "epoch_130.pt",

    EXP_ROOT
    / "photo_to_monet_focus"
    / "checkpoints"
    / "epoch_150.pt",
]


candidate_paths.extend(
    p
    for p in extra_candidates
    if p.exists()
)


candidate_paths = [
    p
    for p in candidate_paths
    if p.exists()
]


# Remove duplicates
candidate_paths = list(
    dict.fromkeys(
        p.resolve()
        for p in candidate_paths
    )
)


print("=" * 78)
print("FAST LITERAL JPEG TOURNAMENT")
print("=" * 78)
print("Candidates:", len(candidate_paths))

for p in candidate_paths:
    print(" ", p)


# ============================================================================
# HELPERS
# ============================================================================

def clean_state_local(state):

    output = {}

    for key, value in state.items():

        for prefix in (
            "_orig_mod.",
            "module.",
        ):
            if key.startswith(prefix):
                key = key[len(prefix):]

        output[key] = (
            value.detach().cpu()
            if torch.is_tensor(value)
            else value
        )

    return output


def build_generator_from_state(state):

    architectures = [
        (9, 64),
        (6, 64),
        (9, 32),
        (6, 32),
    ]

    last_error = None

    for blocks, channels in architectures:

        try:

            model = Generator(
                residual_blocks=blocks,
                base_channels=channels,
            )

            model.load_state_dict(
                state,
                strict=True,
            )

            return model.to(device).eval()

        except Exception as error:
            last_error = error

    raise RuntimeError(
        f"Could not build generator: {last_error}"
    )


def tensor_to_pil(x):

    x = (
        x.detach()
        .cpu()
        .clamp(-1, 1)
    )

    x = (
        (x + 1.0) * 127.5
    ).round().clamp(
        0,
        255
    ).byte()

    x = (
        x.permute(
            1,
            2,
            0
        )
        .numpy()
    )

    return Image.fromarray(x)


# ============================================================================
# INCEPTION
# ============================================================================

print()
print("Loading Inception V3...")

inc = inception_v3(
    weights=(
        Inception_V3_Weights
        .IMAGENET1K_V1
    )
)

inc.fc = nn.Identity()

inc = (
    inc.to(device)
    .eval()
)


literal_transform = transforms.Compose([
    transforms.Resize(299),
    transforms.CenterCrop(299),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406,
        ],
        std=[
            0.229,
            0.224,
            0.225,
        ],
    ),
])


class LiteralDataset(Dataset):

    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):

        with Image.open(
            self.paths[index]
        ) as image:

            return literal_transform(
                image.convert("RGB")
            )


@torch.no_grad()
def extract_features(
    paths,
    label
):

    loader = DataLoader(
        LiteralDataset(paths),
        batch_size=32,
        shuffle=False,
        num_workers=0,
    )

    output = []

    for batch in tqdm(
        loader,
        desc=label,
        leave=False,
    ):

        feats = inc(
            batch.to(device)
        )

        output.append(
            feats
            .detach()
            .cpu()
            .numpy()
        )

    return np.concatenate(
        output,
        axis=0,
    ).astype(
        np.float64
    )


def fid_from_arrays(a, b):

    mu1 = np.mean(a, axis=0)
    mu2 = np.mean(b, axis=0)

    sigma1 = np.cov(
        a,
        rowvar=False,
    )

    sigma2 = np.cov(
        b,
        rowvar=False,
    )

    diff = mu1 - mu2

    covmean = scipy.linalg.sqrtm(
        sigma1.dot(
            sigma2
        )
    )

    if not np.isfinite(
        covmean
    ).all():

        eps = 1e-6

        offset = (
            np.eye(
                sigma1.shape[0]
            )
            * eps
        )

        covmean = scipy.linalg.sqrtm(
            (
                sigma1
                + offset
            ).dot(
                sigma2
                + offset
            )
        )

    if np.iscomplexobj(
        covmean
    ):
        covmean = covmean.real

    return float(
        np.real(
            diff.dot(diff)
            + np.trace(sigma1)
            + np.trace(sigma2)
            - 2.0
            * np.trace(covmean)
        )
    )


# ============================================================================
# REAL REFERENCE FEATURES ONCE
# ============================================================================

print()
print("Computing real features once...")

REAL_MONET_LITERAL = extract_features(
    list(monet_paths)[:N_EVAL],
    "Real Monet"
)

REAL_PHOTO_LITERAL = extract_features(
    list(photo_paths)[:N_EVAL],
    "Real Photo"
)


# ============================================================================
# SCORE ONE GENERATOR
# ============================================================================

@torch.no_grad()
def score_generator(
    model,
    direction,
    tag
):

    temp_dir = (
        TOURNAMENT_DIR
        / tag
    )

    if temp_dir.exists():
        shutil.rmtree(
            temp_dir
        )

    temp_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    if direction == "photo_to_monet":

        input_paths = (
            list(photo_paths)[:N_EVAL]
        )

        real_features = (
            REAL_MONET_LITERAL
        )

    else:

        input_paths = (
            list(monet_paths)[:N_EVAL]
        )

        real_features = (
            REAL_PHOTO_LITERAL
        )


    for i, path in enumerate(
        tqdm(
            input_paths,
            desc=tag,
            leave=False,
        )
    ):

        x = (
            eval_transform(path)
            .unsqueeze(0)
            .to(device)
        )

        y = model(x)[0]

        tensor_to_pil(
            y
        ).save(
            temp_dir
            / f"{i:04d}.jpg",

            format="JPEG",
            quality=JPEG_QUALITY,
        )


    fake_features = extract_features(
        sorted(
            temp_dir.glob(
                "*.jpg"
            )
        ),
        f"features {tag}",
    )


    fid = fid_from_arrays(
        real_features,
        fake_features,
    )


    shutil.rmtree(
        temp_dir
    )

    return fid


# ============================================================================
# TOURNAMENT
# ============================================================================

best_p2m = None
best_m2p = None

all_results = []


for index, path in enumerate(
    candidate_paths,
    start=1,
):

    print()
    print("=" * 78)
    print(
        f"CANDIDATE {index}/{len(candidate_paths)}"
    )
    print(path)
    print("=" * 78)

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )


    # ------------------------------------------------------------------------
    # Photo -> Monet = G_B2A
    # ------------------------------------------------------------------------

    if "G_B2A" in payload:

        state_b2a = clean_state_local(
            payload["G_B2A"]
        )

        model_b2a = (
            build_generator_from_state(
                state_b2a
            )
        )

        p2m = score_generator(
            model_b2a,
            "photo_to_monet",
            f"candidate_{index}_p2m",
        )

        print(
            f"Photo -> Monet FID: "
            f"{p2m:.6f}"
        )

        if (
            best_p2m is None
            or p2m < best_p2m["fid"]
        ):

            best_p2m = {
                "fid": p2m,
                "path": str(path),
                "state": state_b2a,
            }

        del model_b2a


    else:
        p2m = None


    # ------------------------------------------------------------------------
    # Monet -> Photo = G_A2B
    # ------------------------------------------------------------------------

    if "G_A2B" in payload:

        state_a2b = clean_state_local(
            payload["G_A2B"]
        )

        model_a2b = (
            build_generator_from_state(
                state_a2b
            )
        )

        m2p = score_generator(
            model_a2b,
            "monet_to_photo",
            f"candidate_{index}_m2p",
        )

        print(
            f"Monet -> Photo FID: "
            f"{m2p:.6f}"
        )

        if (
            best_m2p is None
            or m2p < best_m2p["fid"]
        ):

            best_m2p = {
                "fid": m2p,
                "path": str(path),
                "state": state_a2b,
            }

        del model_a2b


    else:
        m2p = None


    all_results.append({
        "path": str(path),
        "photo_to_monet": p2m,
        "monet_to_photo": m2p,
    })


    del payload

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# FINAL PAIR
# ============================================================================

FINAL_AVG = (
    best_p2m["fid"]
    + best_m2p["fid"]
) / 2.0


FINAL_CHECKPOINT = (
    TOURNAMENT_DIR
    / "literal_best_generators.pt"
)


torch.save(
    {
        "G_B2A":
            best_p2m["state"],

        "G_A2B":
            best_m2p["state"],

        "literal_fid_photo_to_monet":
            best_p2m["fid"],

        "literal_fid_monet_to_photo":
            best_m2p["fid"],

        "literal_fid_average":
            FINAL_AVG,

        "photo_to_monet_source":
            best_p2m["path"],

        "monet_to_photo_source":
            best_m2p["path"],
    },

    FINAL_CHECKPOINT,
)


# Load winners into notebook memory
G_B2A = build_generator_from_state(
    best_p2m["state"]
)

G_A2B = build_generator_from_state(
    best_m2p["state"]
)


# ============================================================================
# RESULT
# ============================================================================

print()
print("=" * 78)
print("TOURNAMENT RESULT")
print("=" * 78)

print(
    f"Best Photo -> Monet FID: "
    f"{best_p2m['fid']:.6f}"
)

print(
    "Source:",
    best_p2m["path"]
)

print()

print(
    f"Best Monet -> Photo FID: "
    f"{best_m2p['fid']:.6f}"
)

print(
    "Source:",
    best_m2p["path"]
)

print()

print(
    f"NEW COMBINED FID: "
    f"{FINAL_AVG:.6f}"
)

print(
    f"Current best: "
    f"{CURRENT_BEST_AVG:.6f}"
)

print(
    f"Improvement: "
    f"{CURRENT_BEST_AVG - FINAL_AVG:.6f}"
)

print()

if FINAL_AVG < CURRENT_BEST_AVG:

    print(
        "RESULT: NEW PAIR IS BETTER"
    )

else:

    print(
        "RESULT: CURRENT PHASE 3 PAIR REMAINS BEST"
    )

print()

print(
    "Saved:",
    FINAL_CHECKPOINT
)

print("=" * 78)

FAST LITERAL JPEG TOURNAMENT
Candidates: 11
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\best\best_checkpoint.pt
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\identity_2p5_branch\best\best_checkpoint.pt
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\photo_to_monet_focus\best\best_checkpoint.pt
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\continuation_v1\best\best_checkpoint.pt
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\strong_fresh_ttur\best\best_checkpoint.pt
  C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\reverse_focus_2p0\checkpoints\epoch_175.pt
  C:\Users\drashti2\Desktop\drashti


CANDIDATE 1/11
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt


Photo -> Monet FID: 94.357201


Monet -> Photo FID: 100.383239

CANDIDATE 2/11
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\dual_direction_ta_focus\continuation_155\best\best_checkpoint.pt


Photo -> Monet FID: 94.690434


...

Monet -> Photo FID: 102.506898

CANDIDATE 10/11
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\identity_2p5_branch\checkpoints\epoch_130.pt


Photo -> Monet FID: 95.208920


Monet -> Photo FID: 101.860225

CANDIDATE 11/11
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\photo_to_monet_focus\checkpoints\epoch_150.pt


Photo -> Monet FID: 94.487540


Monet -> Photo FID: 103.030164

TOURNAMENT RESULT
Best Photo -> Monet FID: 94.357201
Source: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt

Best Monet -> Photo FID: 100.383239
Source: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt

NEW COMBINED FID: 97.370220
Current best: 97.370220
Improvement: -0.000000

RESULT: CURRENT PHASE 3 PAIR REMAINS BEST

Saved: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\literal_tournament\literal_best_generators.pt


In [34]:
# ============================================================================
# TARGETED 15-EPOCH ASYMMETRIC CONTINUATION
# Goal: improve Monet -> Photo without damaging Photo -> Monet
# Starts from Phase 3 final generators
# ADA OFF, EMA ON
# ============================================================================

import copy
import gc
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


# ============================================================================
# SETTINGS
# ============================================================================

SOURCE_GENERATORS = (
    Path(EXPERIMENT_DIR)
    / "final_task3_pipeline"
    / "final"
    / "final_generators.pt"
)

RUN_DIR = (
    Path(EXPERIMENT_DIR)
    / "targeted_asymmetric_15"
)

CHECKPOINT_DIR = (
    RUN_DIR
    / "checkpoints"
)

BEST_DIR = (
    RUN_DIR
    / "best"
)

for folder in [
    RUN_DIR,
    CHECKPOINT_DIR,
    BEST_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# Architecture
RESIDUAL_BLOCKS = 9
BASE_CHANNELS = 64

# Training
MAX_EPOCHS = 15
STEPS_PER_EPOCH = 1000
PATIENCE = 4

CYCLE_WEIGHT = 10.0
IDENTITY_WEIGHT = 1.0

REPLAY_CAPACITY = 50
EMA_DECAY = 0.999

BETAS = (
    0.5,
    0.999
)

# Asymmetric learning rates
LR_G_A2B = 1e-5   # Monet -> Photo, weaker direction
LR_G_B2A = 2e-6   # Photo -> Monet, protect stronger direction

LR_D_A = 2e-6     # Monet discriminator
LR_D_B = 5e-6     # Photo discriminator

# Diagnostic in-memory evaluation
N_EVAL = 300
EVAL_BATCH = 32

SEED = 2667


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ============================================================================
# REQUIRED OBJECTS
# ============================================================================

required = [
    "Generator",
    "PatchDiscriminator",
    "ReplayBuffer",
    "device",
    "photo_paths",
    "monet_paths",
    "train_transform",
    "eval_transform",
    "lsgan_real",
    "lsgan_fake",
    "l1",
    "inception_features_from_tensor",
    "fid_from_features",
    "REAL_MONET_FEATURES",
    "REAL_PHOTO_FEATURES",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required notebook objects: "
        + ", ".join(missing)
    )


if not SOURCE_GENERATORS.exists():
    raise FileNotFoundError(
        f"Source generators not found:\n{SOURCE_GENERATORS}"
    )


# ============================================================================
# HELPERS
# ============================================================================

def clean_state_targeted(state):

    output = {}

    for key, value in state.items():

        for prefix in (
            "_orig_mod.",
            "module.",
        ):
            if key.startswith(prefix):
                key = key[len(prefix):]

        output[key] = value

    return output


def cpu_state(model):

    return {
        key:
            value.detach().cpu().clone()

        for key, value
        in model.state_dict().items()
    }


def set_requires_grad(
    models,
    flag
):

    for model in models:

        for parameter in model.parameters():
            parameter.requires_grad_(flag)


# ============================================================================
# LOAD SOURCE GENERATORS
# ============================================================================

source = torch.load(
    SOURCE_GENERATORS,
    map_location="cpu",
    weights_only=False,
)


G_A2B = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)

G_B2A = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)


G_A2B.load_state_dict(
    clean_state_targeted(
        source["G_A2B"]
    )
)

G_B2A.load_state_dict(
    clean_state_targeted(
        source["G_B2A"]
    )
)


# ============================================================================
# DISCRIMINATORS
#
# final_generators.pt only contains generators, so D_A and D_B are rebuilt.
# We initialize them normally and let them adapt at tiny learning rates.
# ============================================================================

D_A = PatchDiscriminator(
    BASE_CHANNELS
).to(device)

D_B = PatchDiscriminator(
    BASE_CHANNELS
).to(device)


if "initialize_weights" in globals():

    initialize_weights(D_A)
    initialize_weights(D_B)


# ============================================================================
# EMA
# ============================================================================

G_A2B_EMA = copy.deepcopy(
    G_A2B
).eval()

G_B2A_EMA = copy.deepcopy(
    G_B2A
).eval()


for model in [
    G_A2B_EMA,
    G_B2A_EMA,
]:

    for parameter in model.parameters():
        parameter.requires_grad_(False)


@torch.no_grad()
def update_ema():

    for ema_model, live_model in [
        (
            G_A2B_EMA,
            G_A2B
        ),
        (
            G_B2A_EMA,
            G_B2A
        ),
    ]:

        for ema_p, live_p in zip(
            ema_model.parameters(),
            live_model.parameters(),
        ):

            ema_p.lerp_(
                live_p.detach(),
                1.0 - EMA_DECAY
            )

        for ema_b, live_b in zip(
            ema_model.buffers(),
            live_model.buffers(),
        ):

            ema_b.copy_(
                live_b
            )


# ============================================================================
# OPTIMIZERS
# ============================================================================

opt_G_A2B = torch.optim.Adam(
    G_A2B.parameters(),
    lr=LR_G_A2B,
    betas=BETAS,
)

opt_G_B2A = torch.optim.Adam(
    G_B2A.parameters(),
    lr=LR_G_B2A,
    betas=BETAS,
)

opt_D_A = torch.optim.Adam(
    D_A.parameters(),
    lr=LR_D_A,
    betas=BETAS,
)

opt_D_B = torch.optim.Adam(
    D_B.parameters(),
    lr=LR_D_B,
    betas=BETAS,
)


fake_a_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)

fake_b_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)


# ============================================================================
# FIXED IN-MEMORY EVALUATION DATA
# ============================================================================

print(
    "Loading fixed 300-image diagnostic evaluation set..."
)

EVAL_INPUTS_TARGETED = {

    "photo_to_monet":
        torch.stack([
            eval_transform(path)
            for path
            in list(photo_paths)[:N_EVAL]
        ]),

    "monet_to_photo":
        torch.stack([
            eval_transform(path)
            for path
            in list(monet_paths)[:N_EVAL]
        ]),
}


EVAL_REAL_TARGETED = {

    "photo_to_monet":
        REAL_MONET_FEATURES,

    "monet_to_photo":
        REAL_PHOTO_FEATURES,
}


@torch.no_grad()
def diagnostic_direction_fid(
    generator,
    direction
):

    generator.eval()

    inputs = (
        EVAL_INPUTS_TARGETED[
            direction
        ]
    )

    features = []

    for start in range(
        0,
        inputs.size(0),
        EVAL_BATCH,
    ):

        output = generator(
            inputs[
                start:
                start + EVAL_BATCH
            ].to(device)
        )

        features.append(
            np.asarray(
                inception_features_from_tensor(
                    output
                )
            )
        )

    return float(
        fid_from_features(
            EVAL_REAL_TARGETED[
                direction
            ],
            np.concatenate(
                features,
                axis=0
            ),
        )
    )


# ============================================================================
# DATA SAMPLING
# ============================================================================

def sample_training_batch(
    paths,
    batch_size=1
):

    selected = random.choices(
        list(paths),
        k=batch_size
    )

    return torch.stack([
        train_transform(path)
        for path in selected
    ]).to(device)


# ============================================================================
# DISCRIMINATOR STEP
# NO ADA / NO DIFFAUGMENT
# ============================================================================

def discriminator_step(
    discriminator,
    optimizer,
    real,
    fake,
    replay_buffer,
):

    optimizer.zero_grad(
        set_to_none=True
    )

    fake_replay = replay_buffer.query(
        fake.detach()
    )

    real_prediction = discriminator(
        real
    )

    fake_prediction = discriminator(
        fake_replay
    )

    loss = (
        lsgan_real(
            real_prediction
        )
        + lsgan_fake(
            fake_prediction
        )
    ) * 0.5

    if not torch.isfinite(loss):

        optimizer.zero_grad(
            set_to_none=True
        )

        return (
            float("nan"),
            1
        )

    loss.backward()

    optimizer.step()

    return (
        float(
            loss.detach()
        ),
        0
    )


# ============================================================================
# TRAIN STEP
# ============================================================================

def targeted_train_step(
    real_a,
    real_b
):

    # ------------------------------------------------------------------------
    # GENERATORS
    # ------------------------------------------------------------------------

    set_requires_grad(
        [
            D_A,
            D_B
        ],
        False
    )

    opt_G_A2B.zero_grad(
        set_to_none=True
    )

    opt_G_B2A.zero_grad(
        set_to_none=True
    )


    # A = Monet
    # B = Photo

    fake_b = G_A2B(
        real_a
    )

    fake_a = G_B2A(
        real_b
    )


    reconstructed_a = G_B2A(
        fake_b
    )

    reconstructed_b = G_A2B(
        fake_a
    )


    identity_a = G_B2A(
        real_a
    )

    identity_b = G_A2B(
        real_b
    )


    adv_a2b = lsgan_real(
        D_B(
            fake_b
        )
    )

    adv_b2a = lsgan_real(
        D_A(
            fake_a
        )
    )


    cycle_a = l1(
        reconstructed_a,
        real_a
    )

    cycle_b = l1(
        reconstructed_b,
        real_b
    )


    identity_a_loss = l1(
        identity_a,
        real_a
    )

    identity_b_loss = l1(
        identity_b,
        real_b
    )


    cycle_loss = (
        cycle_a
        + cycle_b
    )

    identity_loss = (
        identity_a_loss
        + identity_b_loss
    )


    generator_loss = (
        adv_a2b
        + adv_b2a
        + CYCLE_WEIGHT
        * cycle_loss
        + IDENTITY_WEIGHT
        * identity_loss
    )


    bad = 0

    if torch.isfinite(
        generator_loss
    ):

        generator_loss.backward()

        opt_G_A2B.step()
        opt_G_B2A.step()

        update_ema()

    else:

        bad += 1


    set_requires_grad(
        [
            D_A,
            D_B
        ],
        True
    )


    # ------------------------------------------------------------------------
    # DISCRIMINATORS
    # ------------------------------------------------------------------------

    d_a_loss, bad_a = (
        discriminator_step(
            D_A,
            opt_D_A,
            real_a,
            fake_a,
            fake_a_buffer,
        )
    )

    d_b_loss, bad_b = (
        discriminator_step(
            D_B,
            opt_D_B,
            real_b,
            fake_b,
            fake_b_buffer,
        )
    )


    return {

        "generator":
            float(
                generator_loss.detach()
            ),

        "cycle":
            float(
                cycle_loss.detach()
            ),

        "identity":
            float(
                identity_loss.detach()
            ),

        "adv_a2b":
            float(
                adv_a2b.detach()
            ),

        "adv_b2a":
            float(
                adv_b2a.detach()
            ),

        "discriminator_a":
            d_a_loss,

        "discriminator_b":
            d_b_loss,

        "nonfinite":
            (
                bad
                + bad_a
                + bad_b
            ),
    }


# ============================================================================
# BASELINE
# ============================================================================

print()
print("=" * 78)
print("TARGETED ASYMMETRIC CONTINUATION")
print("=" * 78)

print(
    "Source:",
    SOURCE_GENERATORS
)

print(
    "G_A2B LR:",
    LR_G_A2B
)

print(
    "G_B2A LR:",
    LR_G_B2A
)

print(
    "D_A LR:",
    LR_D_A
)

print(
    "D_B LR:",
    LR_D_B
)

print(
    "Cycle weight:",
    CYCLE_WEIGHT
)

print(
    "Identity weight:",
    IDENTITY_WEIGHT
)

print(
    "ADA: OFF"
)

print(
    "EMA: ON"
)


baseline_p2m = diagnostic_direction_fid(
    G_B2A,
    "photo_to_monet"
)

baseline_m2p = diagnostic_direction_fid(
    G_A2B,
    "monet_to_photo"
)

baseline_avg = (
    baseline_p2m
    + baseline_m2p
) / 2.0


print()
print(
    f"Diagnostic source FID:"
    f" P2M={baseline_p2m:.6f}"
    f" | M2P={baseline_m2p:.6f}"
    f" | AVG={baseline_avg:.6f}"
)


best = {

    "photo_to_monet": {
        "fid":
            baseline_p2m,

        "state":
            cpu_state(
                G_B2A
            ),

        "variant":
            "source",

        "epoch":
            0,
    },

    "monet_to_photo": {
        "fid":
            baseline_m2p,

        "state":
            cpu_state(
                G_A2B
            ),

        "variant":
            "source",

        "epoch":
            0,
    },
}


best_avg = baseline_avg

evaluations_without_improvement = 0

history = []


# ============================================================================
# TRAINING LOOP
# ============================================================================

for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    G_A2B.train()
    G_B2A.train()
    D_A.train()
    D_B.train()

    totals = defaultdict(
        float
    )

    epoch_start = time.perf_counter()


    progress = tqdm(
        range(
            STEPS_PER_EPOCH
        ),
        desc=(
            f"Targeted {epoch:02d}/"
            f"{MAX_EPOCHS}"
        )
    )


    for step in progress:

        real_a = sample_training_batch(
            monet_paths,
            batch_size=1,
        )

        real_b = sample_training_batch(
            photo_paths,
            batch_size=1,
        )


        metrics = targeted_train_step(
            real_a,
            real_b
        )


        for key, value in metrics.items():

            if np.isfinite(
                value
            ):
                totals[key] += value


        if (
            step % 50 == 0
            or step
            == STEPS_PER_EPOCH - 1
        ):

            count = (
                step + 1
            )

            progress.set_postfix({

                "G":
                    f"{totals['generator']/count:.3f}",

                "cyc":
                    f"{totals['cycle']/count:.3f}",

                "id":
                    f"{totals['identity']/count:.3f}",

                "DA":
                    f"{totals['discriminator_a']/count:.3f}",

                "DB":
                    f"{totals['discriminator_b']/count:.3f}",
            })


    elapsed = (
        time.perf_counter()
        - epoch_start
    )


    # ========================================================================
    # EVALUATE LIVE + EMA EVERY EPOCH
    # ========================================================================

    candidates = {

        "live": {

            "photo_to_monet":
                G_B2A,

            "monet_to_photo":
                G_A2B,
        },

        "ema": {

            "photo_to_monet":
                G_B2A_EMA,

            "monet_to_photo":
                G_A2B_EMA,
        },
    }


    epoch_results = {}


    for variant, models in candidates.items():

        p2m = diagnostic_direction_fid(
            models[
                "photo_to_monet"
            ],
            "photo_to_monet"
        )

        m2p = diagnostic_direction_fid(
            models[
                "monet_to_photo"
            ],
            "monet_to_photo"
        )

        avg = (
            p2m
            + m2p
        ) / 2.0


        epoch_results[
            variant
        ] = {

            "photo_to_monet":
                p2m,

            "monet_to_photo":
                m2p,

            "average":
                avg,
        }


    improved_any = False


    # Track P2M independently
    for variant in [
        "live",
        "ema",
    ]:

        score = (
            epoch_results[
                variant
            ][
                "photo_to_monet"
            ]
        )

        if (
            score
            < best[
                "photo_to_monet"
            ][
                "fid"
            ]
        ):

            source_model = (
                G_B2A
                if variant == "live"
                else G_B2A_EMA
            )

            best[
                "photo_to_monet"
            ] = {

                "fid":
                    score,

                "state":
                    cpu_state(
                        source_model
                    ),

                "variant":
                    variant,

                "epoch":
                    epoch,
            }

            improved_any = True


    # Track M2P independently
    for variant in [
        "live",
        "ema",
    ]:

        score = (
            epoch_results[
                variant
            ][
                "monet_to_photo"
            ]
        )

        if (
            score
            < best[
                "monet_to_photo"
            ][
                "fid"
            ]
        ):

            source_model = (
                G_A2B
                if variant == "live"
                else G_A2B_EMA
            )

            best[
                "monet_to_photo"
            ] = {

                "fid":
                    score,

                "state":
                    cpu_state(
                        source_model
                    ),

                "variant":
                    variant,

                "epoch":
                    epoch,
            }

            improved_any = True


    current_best_avg = (
        best[
            "photo_to_monet"
        ][
            "fid"
        ]
        + best[
            "monet_to_photo"
        ][
            "fid"
        ]
    ) / 2.0


    if (
        current_best_avg
        < best_avg
    ):

        best_avg = (
            current_best_avg
        )

        improved_any = True


    if improved_any:

        evaluations_without_improvement = 0

    else:

        evaluations_without_improvement += 1


    # ========================================================================
    # SAVE EPOCH CHECKPOINT
    # ========================================================================

    epoch_checkpoint = (
        CHECKPOINT_DIR
        / f"epoch_{epoch:03d}.pt"
    )


    torch.save(
        {
            "epoch":
                epoch,

            "G_A2B":
                cpu_state(
                    G_A2B
                ),

            "G_B2A":
                cpu_state(
                    G_B2A
                ),

            "G_A2B_EMA":
                cpu_state(
                    G_A2B_EMA
                ),

            "G_B2A_EMA":
                cpu_state(
                    G_B2A_EMA
                ),

            "D_A":
                cpu_state(
                    D_A
                ),

            "D_B":
                cpu_state(
                    D_B
                ),

            "live_fid_photo_to_monet":
                epoch_results[
                    "live"
                ][
                    "photo_to_monet"
                ],

            "live_fid_monet_to_photo":
                epoch_results[
                    "live"
                ][
                    "monet_to_photo"
                ],

            "ema_fid_photo_to_monet":
                epoch_results[
                    "ema"
                ][
                    "photo_to_monet"
                ],

            "ema_fid_monet_to_photo":
                epoch_results[
                    "ema"
                ][
                    "monet_to_photo"
                ],

            "settings": {

                "lr_g_a2b":
                    LR_G_A2B,

                "lr_g_b2a":
                    LR_G_B2A,

                "lr_d_a":
                    LR_D_A,

                "lr_d_b":
                    LR_D_B,

                "cycle_weight":
                    CYCLE_WEIGHT,

                "identity_weight":
                    IDENTITY_WEIGHT,

                "ema_decay":
                    EMA_DECAY,
            },
        },

        epoch_checkpoint,
    )


    # ========================================================================
    # SAVE BEST DIRECTIONAL PAIR
    # ========================================================================

    best_path = (
        BEST_DIR
        / "best_checkpoint.pt"
    )


    torch.save(
        {
            "G_B2A":
                best[
                    "photo_to_monet"
                ][
                    "state"
                ],

            "G_A2B":
                best[
                    "monet_to_photo"
                ][
                    "state"
                ],

            "ta_fid_photo_to_monet":
                best[
                    "photo_to_monet"
                ][
                    "fid"
                ],

            "ta_fid_monet_to_photo":
                best[
                    "monet_to_photo"
                ][
                    "fid"
                ],

            "ta_fid_average":
                current_best_avg,

            "selection": {

                "photo_to_monet": {

                    "epoch":
                        best[
                            "photo_to_monet"
                        ][
                            "epoch"
                        ],

                    "variant":
                        best[
                            "photo_to_monet"
                        ][
                            "variant"
                        ],
                },

                "monet_to_photo": {

                    "epoch":
                        best[
                            "monet_to_photo"
                        ][
                            "epoch"
                        ],

                    "variant":
                        best[
                            "monet_to_photo"
                        ][
                            "variant"
                        ],
                },
            },
        },

        best_path,
    )


    history.append({

        "epoch":
            epoch,

        "elapsed_sec":
            elapsed,

        "live":
            epoch_results[
                "live"
            ],

        "ema":
            epoch_results[
                "ema"
            ],

        "best_photo_to_monet":
            best[
                "photo_to_monet"
            ][
                "fid"
            ],

        "best_monet_to_photo":
            best[
                "monet_to_photo"
            ][
                "fid"
            ],

        "best_average":
            current_best_avg,
    })


    with open(
        RUN_DIR
        / "history.json",
        "w"
    ) as file:

        json.dump(
            history,
            file,
            indent=2
        )


    print()
    print("-" * 78)

    print(
        f"Epoch {epoch} complete "
        f"({elapsed:.1f}s)"
    )

    print(
        "LIVE  "
        f"P2M={epoch_results['live']['photo_to_monet']:.6f} "
        f"| M2P={epoch_results['live']['monet_to_photo']:.6f} "
        f"| AVG={epoch_results['live']['average']:.6f}"
    )

    print(
        "EMA   "
        f"P2M={epoch_results['ema']['photo_to_monet']:.6f} "
        f"| M2P={epoch_results['ema']['monet_to_photo']:.6f} "
        f"| AVG={epoch_results['ema']['average']:.6f}"
    )

    print(
        "BEST  "
        f"P2M={best['photo_to_monet']['fid']:.6f} "
        f"| M2P={best['monet_to_photo']['fid']:.6f} "
        f"| AVG={current_best_avg:.6f}"
    )

    print(
        "Best P2M source:"
        f" epoch {best['photo_to_monet']['epoch']} "
        f"{best['photo_to_monet']['variant']}"
    )

    print(
        "Best M2P source:"
        f" epoch {best['monet_to_photo']['epoch']} "
        f"{best['monet_to_photo']['variant']}"
    )

    print(
        "No improvement count:",
        evaluations_without_improvement,
        "/",
        PATIENCE
    )

    print("-" * 78)


    # ========================================================================
    # EARLY STOP
    # ========================================================================

    if (
        evaluations_without_improvement
        >= PATIENCE
    ):

        print()
        print(
            "EARLY STOPPING: "
            f"{PATIENCE} consecutive evaluations "
            "without improvement."
        )

        break


    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# LOAD BEST PAIR BACK INTO NOTEBOOK
# ============================================================================

best_payload = torch.load(
    BEST_DIR
    / "best_checkpoint.pt",
    map_location="cpu",
    weights_only=False,
)


G_A2B = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)

G_B2A = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)


G_A2B.load_state_dict(
    best_payload[
        "G_A2B"
    ]
)

G_B2A.load_state_dict(
    best_payload[
        "G_B2A"
    ]
)


G_A2B.eval()
G_B2A.eval()


print()
print("=" * 78)
print("TARGETED TRAINING FINISHED")
print("=" * 78)

print(
    "Best diagnostic Photo -> Monet:",
    best_payload[
        "ta_fid_photo_to_monet"
    ]
)

print(
    "Best diagnostic Monet -> Photo:",
    best_payload[
        "ta_fid_monet_to_photo"
    ]
)

print(
    "Best diagnostic average:",
    best_payload[
        "ta_fid_average"
    ]
)

print(
    "Saved:",
    BEST_DIR
    / "best_checkpoint.pt"
)

print()
print(
    "NEXT: run the literal JPEG verification cell "
    "on these loaded G_A2B / G_B2A models."
)

print("=" * 78)

Loading fixed 300-image diagnostic evaluation set...

TARGETED ASYMMETRIC CONTINUATION
Source: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt
G_A2B LR: 1e-05
G_B2A LR: 2e-06
D_A LR: 2e-06
D_B LR: 5e-06
Cycle weight: 10.0
Identity weight: 1.0
ADA: OFF
EMA: ON


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



Diagnostic source FID: P2M=98.483042 | M2P=101.476010 | AVG=99.979526


Targeted 01/15: 100%|██████████| 1000/1000 [01:16<00:00, 13.05it/s, G=3.520, cyc=0.192, id=0.152, DA=0.700, DB=0.557]



------------------------------------------------------------------------------
Epoch 1 complete (76.7s)
LIVE  P2M=99.690792 | M2P=103.718061 | AVG=101.704427
EMA   P2M=98.342510 | M2P=101.920849 | AVG=100.131680
BEST  P2M=98.342510 | M2P=101.476010 | AVG=99.909260
Best P2M source: epoch 1 ema
Best M2P source: epoch 0 source
No improvement count: 0 / 4
------------------------------------------------------------------------------


...


------------------------------------------------------------------------------
Epoch 3 complete (75.5s)
LIVE  P2M=102.468440 | M2P=104.888463 | AVG=103.678452
EMA   P2M=101.203039 | M2P=103.860770 | AVG=102.531905
BEST  P2M=98.342510 | M2P=101.476010 | AVG=99.909260
Best P2M source: epoch 1 ema
Best M2P source: epoch 0 source
No improvement count: 2 / 4
------------------------------------------------------------------------------


Targeted 04/15: 100%|██████████| 1000/1000 [01:15<00:00, 13.21it/s, G=2.185, cyc=0.143, id=0.147, DA=0.269, DB=0.248]



------------------------------------------------------------------------------
Epoch 4 complete (75.7s)
LIVE  P2M=103.288878 | M2P=105.470016 | AVG=104.379447
EMA   P2M=102.215010 | M2P=104.858873 | AVG=103.536942
BEST  P2M=98.342510 | M2P=101.476010 | AVG=99.909260
Best P2M source: epoch 1 ema
Best M2P source: epoch 0 source
No improvement count: 3 / 4
------------------------------------------------------------------------------


Targeted 05/15: 100%|██████████| 1000/1000 [01:16<00:00, 13.12it/s, G=2.127, cyc=0.139, id=0.144, DA=0.259, DB=0.245]



------------------------------------------------------------------------------
Epoch 5 complete (76.2s)
LIVE  P2M=103.433440 | M2P=105.273783 | AVG=104.353611
EMA   P2M=102.952694 | M2P=105.190826 | AVG=104.071760
BEST  P2M=98.342510 | M2P=101.476010 | AVG=99.909260
Best P2M source: epoch 1 ema
Best M2P source: epoch 0 source
No improvement count: 4 / 4
------------------------------------------------------------------------------

EARLY STOPPING: 4 consecutive evaluations without improvement.

TARGETED TRAINING FINISHED
Best diagnostic Photo -> Monet: 98.34251027709482
Best diagnostic Monet -> Photo: 101.47601040613984
Best diagnostic average: 99.90926034161733
Saved: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\targeted_asymmetric_15\best\best_checkpoint.pt

NEXT: run the literal JPEG verification cell on these loaded G_A2B / G_B2A models.


In [35]:
# ============================================================================
# FIND THE FULL CHECKPOINTS BEHIND THE PHASE 3 GENERATORS
# ============================================================================

from pathlib import Path
import torch
import json

FINAL_PHASE3 = (
    Path(EXPERIMENT_DIR)
    / "final_task3_pipeline"
    / "final"
    / "final_generators.pt"
)

payload = torch.load(
    FINAL_PHASE3,
    map_location="cpu",
    weights_only=False,
)

print("=" * 78)
print("PHASE 3 PROVENANCE")
print("=" * 78)

provenance = payload.get(
    "provenance",
    {}
)

print(
    json.dumps(
        provenance,
        indent=2,
        default=str,
    )
)

print()
print("=" * 78)
print("CHECKING SOURCE CHECKPOINTS")
print("=" * 78)


source_paths = []


for key in [
    "G_B2A_photo_to_monet_members",
    "G_A2B_monet_to_photo_members",
]:

    print()
    print(key)
    print("-" * 78)

    members = provenance.get(
        key,
        []
    )

    for member in members:

        path = Path(
            member["path"]
        )

        source_paths.append(
            path
        )

        print()
        print("Path:", path)
        print("Exists:", path.exists())
        print("Epoch:", member.get("epoch"))
        print("Generator key:", member.get("key"))
        print("Single FID:", member.get("single_fid"))

        if not path.exists():
            continue

        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        print(
            "Has G_A2B:",
            "G_A2B" in ckpt
        )

        print(
            "Has G_B2A:",
            "G_B2A" in ckpt
        )

        print(
            "Has D_A:",
            "D_A" in ckpt
        )

        print(
            "Has D_B:",
            "D_B" in ckpt
        )

        print(
            "Has generator optimizer:",
            "generator_optimizer" in ckpt
        )

        print(
            "Has D_A optimizer:",
            "discriminator_a_optimizer" in ckpt
        )

        print(
            "Has D_B optimizer:",
            "discriminator_b_optimizer" in ckpt
        )

        del ckpt


print()
print("=" * 78)
print("DONE")
print("=" * 78)

PHASE 3 PROVENANCE
{}

CHECKING SOURCE CHECKPOINTS

G_B2A_photo_to_monet_members
------------------------------------------------------------------------------

G_A2B_monet_to_photo_members
------------------------------------------------------------------------------

DONE


In [36]:
from pathlib import Path
import torch

FULL_SOURCE = (
    Path(EXPERIMENT_DIR)
    / "strong_fresh_ttur"
    / "best"
    / "best_checkpoint.pt"
)

ckpt = torch.load(
    FULL_SOURCE,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint:", FULL_SOURCE)
print("Epoch:", ckpt.get("epoch"))

for key in [
    "G_A2B",
    "G_B2A",
    "D_A",
    "D_B",
    "generator_optimizer",
    "discriminator_a_optimizer",
    "discriminator_b_optimizer",
]:
    print(f"{key:28}:", key in ckpt)

Checkpoint: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\strong_fresh_ttur\best\best_checkpoint.pt
Epoch: 140
G_A2B                       : True
G_B2A                       : True
D_A                         : True
D_B                         : True
generator_optimizer         : True
discriminator_a_optimizer   : True
discriminator_b_optimizer   : True


In [37]:
# ============================================================================
# COHERENT ASYMMETRIC CONTINUATION
# Source = strong_fresh_ttur epoch 140 with MATCHING discriminators
# Goal = improve Monet -> Photo while preserving Photo -> Monet
# ============================================================================

import copy
import gc
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


# ============================================================================
# SETTINGS
# ============================================================================

SOURCE_CHECKPOINT = (
    Path(EXPERIMENT_DIR)
    / "strong_fresh_ttur"
    / "best"
    / "best_checkpoint.pt"
)

RUN_DIR = (
    Path(EXPERIMENT_DIR)
    / "coherent_asymmetric_from_e140"
)

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
BEST_DIR = RUN_DIR / "best"

for folder in [
    RUN_DIR,
    CHECKPOINT_DIR,
    BEST_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


RESIDUAL_BLOCKS = 9
BASE_CHANNELS = 64

MAX_EPOCHS = 10
STEPS_PER_EPOCH = 1000
PATIENCE = 3

CYCLE_WEIGHT = 10.0
IDENTITY_WEIGHT = 1.0

EMA_DECAY = 0.999
REPLAY_CAPACITY = 50

BETAS = (
    0.5,
    0.999
)

# Weak direction = Monet -> Photo = G_A2B
LR_G_A2B = 1e-5

# Stronger direction = Photo -> Monet = G_B2A
LR_G_B2A = 2e-6

# D_A judges Monet
LR_D_A = 2e-6

# D_B judges Photo
LR_D_B = 5e-6

N_EVAL = 300
EVAL_BATCH = 32

SEED = 2671


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ============================================================================
# CHECK REQUIRED OBJECTS
# ============================================================================

required = [
    "Generator",
    "PatchDiscriminator",
    "ReplayBuffer",
    "device",
    "photo_paths",
    "monet_paths",
    "train_transform",
    "eval_transform",
    "lsgan_real",
    "lsgan_fake",
    "l1",
    "inception_features_from_tensor",
    "fid_from_features",
    "REAL_MONET_FEATURES",
    "REAL_PHOTO_FEATURES",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required notebook objects: "
        + ", ".join(missing)
    )


if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing checkpoint:\n{SOURCE_CHECKPOINT}"
    )


# ============================================================================
# HELPERS
# ============================================================================

def clean_state(state):

    output = {}

    for key, value in state.items():

        for prefix in [
            "_orig_mod.",
            "module.",
        ]:

            if key.startswith(prefix):
                key = key[len(prefix):]

        output[key] = value

    return output


def cpu_state(model):

    return {
        key:
            value.detach().cpu().clone()

        for key, value
        in model.state_dict().items()
    }


def set_requires_grad(
    models,
    flag
):

    for model in models:

        for parameter in model.parameters():
            parameter.requires_grad_(flag)


# ============================================================================
# LOAD FULL COHERENT CHECKPOINT
# ============================================================================

checkpoint = torch.load(
    SOURCE_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

SOURCE_EPOCH = int(
    checkpoint.get(
        "epoch",
        140
    )
)


G_A2B = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)

G_B2A = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)

D_A = PatchDiscriminator(
    BASE_CHANNELS
).to(device)

D_B = PatchDiscriminator(
    BASE_CHANNELS
).to(device)


G_A2B.load_state_dict(
    clean_state(
        checkpoint["G_A2B"]
    )
)

G_B2A.load_state_dict(
    clean_state(
        checkpoint["G_B2A"]
    )
)

D_A.load_state_dict(
    clean_state(
        checkpoint["D_A"]
    )
)

D_B.load_state_dict(
    clean_state(
        checkpoint["D_B"]
    )
)


print("=" * 78)
print("LOADED COHERENT FULL CYCLEGAN STATE")
print("=" * 78)
print("Source:", SOURCE_CHECKPOINT)
print("Epoch:", SOURCE_EPOCH)
print("G_A2B + G_B2A + D_A + D_B restored")
print("=" * 78)


# ============================================================================
# EMA
# ============================================================================

G_A2B_EMA = copy.deepcopy(
    G_A2B
).eval()

G_B2A_EMA = copy.deepcopy(
    G_B2A
).eval()


for model in [
    G_A2B_EMA,
    G_B2A_EMA,
]:

    for parameter in model.parameters():
        parameter.requires_grad_(False)


@torch.no_grad()
def update_ema():

    for ema_model, live_model in [
        (
            G_A2B_EMA,
            G_A2B
        ),
        (
            G_B2A_EMA,
            G_B2A
        ),
    ]:

        for ema_p, live_p in zip(
            ema_model.parameters(),
            live_model.parameters(),
        ):

            ema_p.lerp_(
                live_p.detach(),
                1.0 - EMA_DECAY
            )

        for ema_b, live_b in zip(
            ema_model.buffers(),
            live_model.buffers(),
        ):

            ema_b.copy_(
                live_b
            )


# ============================================================================
# OPTIMIZERS
#
# Fresh optimizers intentionally:
# we want different LR for G_A2B and G_B2A.
# The old checkpoint used a combined generator optimizer.
# ============================================================================

opt_G_A2B = torch.optim.Adam(
    G_A2B.parameters(),
    lr=LR_G_A2B,
    betas=BETAS,
)

opt_G_B2A = torch.optim.Adam(
    G_B2A.parameters(),
    lr=LR_G_B2A,
    betas=BETAS,
)

opt_D_A = torch.optim.Adam(
    D_A.parameters(),
    lr=LR_D_A,
    betas=BETAS,
)

opt_D_B = torch.optim.Adam(
    D_B.parameters(),
    lr=LR_D_B,
    betas=BETAS,
)


fake_a_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)

fake_b_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)


# ============================================================================
# FIXED DIAGNOSTIC EVALUATION SET
# ============================================================================

print()
print("Loading fixed 300-image diagnostic set...")


EVAL_INPUTS_COHERENT = {

    "photo_to_monet":
        torch.stack([
            eval_transform(path)
            for path
            in list(photo_paths)[:N_EVAL]
        ]),

    "monet_to_photo":
        torch.stack([
            eval_transform(path)
            for path
            in list(monet_paths)[:N_EVAL]
        ]),
}


EVAL_REAL_COHERENT = {

    "photo_to_monet":
        REAL_MONET_FEATURES,

    "monet_to_photo":
        REAL_PHOTO_FEATURES,
}


@torch.no_grad()
def direction_fid_coherent(
    generator,
    direction
):

    generator.eval()

    inputs = (
        EVAL_INPUTS_COHERENT[
            direction
        ]
    )

    features = []

    for start in range(
        0,
        inputs.size(0),
        EVAL_BATCH,
    ):

        output = generator(
            inputs[
                start:
                start + EVAL_BATCH
            ].to(device)
        )

        features.append(
            np.asarray(
                inception_features_from_tensor(
                    output
                )
            )
        )

    return float(
        fid_from_features(
            EVAL_REAL_COHERENT[
                direction
            ],
            np.concatenate(
                features,
                axis=0
            ),
        )
    )


# ============================================================================
# TRAINING DATA
# ============================================================================

def sample_batch(
    paths
):

    path = random.choice(
        list(paths)
    )

    return (
        train_transform(path)
        .unsqueeze(0)
        .to(device)
    )


# ============================================================================
# DISCRIMINATOR STEP
# ============================================================================

def discriminator_step(
    discriminator,
    optimizer,
    real,
    fake,
    replay_buffer
):

    optimizer.zero_grad(
        set_to_none=True
    )

    fake_replay = replay_buffer.query(
        fake.detach()
    )

    prediction_real = discriminator(
        real
    )

    prediction_fake = discriminator(
        fake_replay
    )

    loss = 0.5 * (
        lsgan_real(
            prediction_real
        )
        +
        lsgan_fake(
            prediction_fake
        )
    )

    if not torch.isfinite(loss):

        optimizer.zero_grad(
            set_to_none=True
        )

        return (
            float("nan"),
            1
        )

    loss.backward()
    optimizer.step()

    return (
        float(
            loss.detach()
        ),
        0
    )


# ============================================================================
# TRAIN STEP
# ============================================================================

def train_step_coherent(
    real_a,
    real_b
):

    # A = Monet
    # B = Photo

    set_requires_grad(
        [
            D_A,
            D_B
        ],
        False
    )


    opt_G_A2B.zero_grad(
        set_to_none=True
    )

    opt_G_B2A.zero_grad(
        set_to_none=True
    )


    fake_b = G_A2B(
        real_a
    )

    fake_a = G_B2A(
        real_b
    )


    reconstructed_a = G_B2A(
        fake_b
    )

    reconstructed_b = G_A2B(
        fake_a
    )


    identity_a = G_B2A(
        real_a
    )

    identity_b = G_A2B(
        real_b
    )


    adv_a2b = lsgan_real(
        D_B(
            fake_b
        )
    )

    adv_b2a = lsgan_real(
        D_A(
            fake_a
        )
    )


    cycle_a = l1(
        reconstructed_a,
        real_a
    )

    cycle_b = l1(
        reconstructed_b,
        real_b
    )


    identity_a_loss = l1(
        identity_a,
        real_a
    )

    identity_b_loss = l1(
        identity_b,
        real_b
    )


    cycle_loss = (
        cycle_a
        + cycle_b
    )

    identity_loss = (
        identity_a_loss
        + identity_b_loss
    )


    generator_loss = (
        adv_a2b
        + adv_b2a
        + CYCLE_WEIGHT
        * cycle_loss
        + IDENTITY_WEIGHT
        * identity_loss
    )


    bad = 0


    if torch.isfinite(
        generator_loss
    ):

        generator_loss.backward()

        opt_G_A2B.step()
        opt_G_B2A.step()

        update_ema()

    else:

        bad += 1


    set_requires_grad(
        [
            D_A,
            D_B
        ],
        True
    )


    d_a_loss, bad_a = discriminator_step(
        D_A,
        opt_D_A,
        real_a,
        fake_a,
        fake_a_buffer,
    )


    d_b_loss, bad_b = discriminator_step(
        D_B,
        opt_D_B,
        real_b,
        fake_b,
        fake_b_buffer,
    )


    return {

        "generator":
            float(
                generator_loss.detach()
            ),

        "cycle":
            float(
                cycle_loss.detach()
            ),

        "identity":
            float(
                identity_loss.detach()
            ),

        "adv_a2b":
            float(
                adv_a2b.detach()
            ),

        "adv_b2a":
            float(
                adv_b2a.detach()
            ),

        "discriminator_a":
            d_a_loss,

        "discriminator_b":
            d_b_loss,

        "nonfinite":
            bad
            + bad_a
            + bad_b,
    }


# ============================================================================
# SOURCE BASELINE
# ============================================================================

source_p2m = direction_fid_coherent(
    G_B2A,
    "photo_to_monet"
)

source_m2p = direction_fid_coherent(
    G_A2B,
    "monet_to_photo"
)

source_avg = (
    source_p2m
    + source_m2p
) / 2.0


print()
print("=" * 78)
print("COHERENT CONTINUATION BASELINE")
print("=" * 78)

print(
    f"P2M={source_p2m:.6f}"
)

print(
    f"M2P={source_m2p:.6f}"
)

print(
    f"AVG={source_avg:.6f}"
)

print()
print(
    "Literal JPEG score for this checkpoint from tournament:"
)

print(
    "P2M=94.678612"
)

print(
    "M2P=101.111024"
)

print(
    "AVG=97.894818"
)

print()
print(
    "Literal Phase 3 score to beat: 97.370220"
)

print("=" * 78)


# ============================================================================
# BEST TRACKING
# ============================================================================

best = {

    "photo_to_monet": {

        "fid":
            source_p2m,

        "state":
            cpu_state(
                G_B2A
            ),

        "epoch":
            SOURCE_EPOCH,

        "variant":
            "source",
    },

    "monet_to_photo": {

        "fid":
            source_m2p,

        "state":
            cpu_state(
                G_A2B
            ),

        "epoch":
            SOURCE_EPOCH,

        "variant":
            "source",
    },
}


best_avg = (
    best[
        "photo_to_monet"
    ][
        "fid"
    ]
    +
    best[
        "monet_to_photo"
    ][
        "fid"
    ]
) / 2.0


no_improvement = 0
history = []


# ============================================================================
# TRAINING LOOP
# ============================================================================

for continuation_epoch in range(
    1,
    MAX_EPOCHS + 1
):

    total_epoch = (
        SOURCE_EPOCH
        + continuation_epoch
    )


    G_A2B.train()
    G_B2A.train()
    D_A.train()
    D_B.train()


    totals = defaultdict(
        float
    )


    epoch_start = time.perf_counter()


    progress = tqdm(
        range(
            STEPS_PER_EPOCH
        ),
        desc=(
            f"Coherent "
            f"{continuation_epoch:02d}/"
            f"{MAX_EPOCHS}"
        )
    )


    for step in progress:

        real_a = sample_batch(
            monet_paths
        )

        real_b = sample_batch(
            photo_paths
        )


        metrics = train_step_coherent(
            real_a,
            real_b
        )


        for key, value in metrics.items():

            if np.isfinite(
                value
            ):
                totals[key] += value


        if (
            step % 50 == 0
            or step
            == STEPS_PER_EPOCH - 1
        ):

            n = (
                step
                + 1
            )

            progress.set_postfix({

                "G":
                    f"{totals['generator']/n:.3f}",

                "cyc":
                    f"{totals['cycle']/n:.3f}",

                "id":
                    f"{totals['identity']/n:.3f}",

                "DA":
                    f"{totals['discriminator_a']/n:.3f}",

                "DB":
                    f"{totals['discriminator_b']/n:.3f}",
            })


    elapsed = (
        time.perf_counter()
        - epoch_start
    )


    # ========================================================================
    # EVALUATE LIVE + EMA
    # ========================================================================

    variants = {

        "live": {
            "photo_to_monet":
                G_B2A,

            "monet_to_photo":
                G_A2B,
        },

        "ema": {
            "photo_to_monet":
                G_B2A_EMA,

            "monet_to_photo":
                G_A2B_EMA,
        },
    }


    results = {}


    for variant, models in variants.items():

        p2m = direction_fid_coherent(
            models[
                "photo_to_monet"
            ],
            "photo_to_monet"
        )

        m2p = direction_fid_coherent(
            models[
                "monet_to_photo"
            ],
            "monet_to_photo"
        )

        results[
            variant
        ] = {

            "photo_to_monet":
                p2m,

            "monet_to_photo":
                m2p,

            "average":
                (
                    p2m
                    + m2p
                ) / 2.0,
        }


    improved = False


    # P2M
    for variant in [
        "live",
        "ema",
    ]:

        score = (
            results[
                variant
            ][
                "photo_to_monet"
            ]
        )

        if (
            score
            <
            best[
                "photo_to_monet"
            ][
                "fid"
            ]
        ):

            model = (
                G_B2A
                if variant == "live"
                else G_B2A_EMA
            )

            best[
                "photo_to_monet"
            ] = {

                "fid":
                    score,

                "state":
                    cpu_state(
                        model
                    ),

                "epoch":
                    total_epoch,

                "variant":
                    variant,
            }

            improved = True


    # M2P
    for variant in [
        "live",
        "ema",
    ]:

        score = (
            results[
                variant
            ][
                "monet_to_photo"
            ]
        )

        if (
            score
            <
            best[
                "monet_to_photo"
            ][
                "fid"
            ]
        ):

            model = (
                G_A2B
                if variant == "live"
                else G_A2B_EMA
            )

            best[
                "monet_to_photo"
            ] = {

                "fid":
                    score,

                "state":
                    cpu_state(
                        model
                    ),

                "epoch":
                    total_epoch,

                "variant":
                    variant,
            }

            improved = True


    current_best_avg = (
        best[
            "photo_to_monet"
        ][
            "fid"
        ]
        +
        best[
            "monet_to_photo"
        ][
            "fid"
        ]
    ) / 2.0


    if (
        current_best_avg
        < best_avg
    ):

        best_avg = (
            current_best_avg
        )

        improved = True


    if improved:
        no_improvement = 0

    else:
        no_improvement += 1


    # ========================================================================
    # SAVE EVERY EPOCH
    # ========================================================================

    epoch_path = (
        CHECKPOINT_DIR
        / f"epoch_{total_epoch}.pt"
    )


    torch.save(
        {
            "epoch":
                total_epoch,

            "G_A2B":
                cpu_state(
                    G_A2B
                ),

            "G_B2A":
                cpu_state(
                    G_B2A
                ),

            "G_A2B_EMA":
                cpu_state(
                    G_A2B_EMA
                ),

            "G_B2A_EMA":
                cpu_state(
                    G_B2A_EMA
                ),

            "D_A":
                cpu_state(
                    D_A
                ),

            "D_B":
                cpu_state(
                    D_B
                ),

            "live_fid_photo_to_monet":
                results[
                    "live"
                ][
                    "photo_to_monet"
                ],

            "live_fid_monet_to_photo":
                results[
                    "live"
                ][
                    "monet_to_photo"
                ],

            "ema_fid_photo_to_monet":
                results[
                    "ema"
                ][
                    "photo_to_monet"
                ],

            "ema_fid_monet_to_photo":
                results[
                    "ema"
                ][
                    "monet_to_photo"
                ],
        },

        epoch_path,
    )


    # ========================================================================
    # SAVE BEST DIRECTIONAL PAIR
    # ========================================================================

    best_path = (
        BEST_DIR
        / "best_checkpoint.pt"
    )


    torch.save(
        {
            "G_B2A":
                best[
                    "photo_to_monet"
                ][
                    "state"
                ],

            "G_A2B":
                best[
                    "monet_to_photo"
                ][
                    "state"
                ],

            "ta_fid_photo_to_monet":
                best[
                    "photo_to_monet"
                ][
                    "fid"
                ],

            "ta_fid_monet_to_photo":
                best[
                    "monet_to_photo"
                ][
                    "fid"
                ],

            "ta_fid_average":
                current_best_avg,

            "selection": {

                "photo_to_monet": {
                    "epoch":
                        best[
                            "photo_to_monet"
                        ][
                            "epoch"
                        ],

                    "variant":
                        best[
                            "photo_to_monet"
                        ][
                            "variant"
                        ],
                },

                "monet_to_photo": {
                    "epoch":
                        best[
                            "monet_to_photo"
                        ][
                            "epoch"
                        ],

                    "variant":
                        best[
                            "monet_to_photo"
                        ][
                            "variant"
                        ],
                },
            },
        },

        best_path,
    )


    history.append({

        "epoch":
            total_epoch,

        "live":
            results[
                "live"
            ],

        "ema":
            results[
                "ema"
            ],

        "best_average":
            current_best_avg,
    })


    with open(
        RUN_DIR
        / "history.json",
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    # ========================================================================
    # OUTPUT
    # ========================================================================

    print()
    print("-" * 78)

    print(
        f"Total epoch {total_epoch} "
        f"complete in {elapsed:.1f}s"
    )

    print(
        "LIVE  "
        f"P2M={results['live']['photo_to_monet']:.6f} "
        f"| M2P={results['live']['monet_to_photo']:.6f} "
        f"| AVG={results['live']['average']:.6f}"
    )

    print(
        "EMA   "
        f"P2M={results['ema']['photo_to_monet']:.6f} "
        f"| M2P={results['ema']['monet_to_photo']:.6f} "
        f"| AVG={results['ema']['average']:.6f}"
    )

    print(
        "BEST  "
        f"P2M={best['photo_to_monet']['fid']:.6f} "
        f"| M2P={best['monet_to_photo']['fid']:.6f} "
        f"| AVG={current_best_avg:.6f}"
    )

    print(
        "Best P2M:",
        best[
            "photo_to_monet"
        ][
            "epoch"
        ],
        best[
            "photo_to_monet"
        ][
            "variant"
        ]
    )

    print(
        "Best M2P:",
        best[
            "monet_to_photo"
        ][
            "epoch"
        ],
        best[
            "monet_to_photo"
        ][
            "variant"
        ]
    )

    print(
        "No improvement:",
        no_improvement,
        "/",
        PATIENCE
    )

    print("-" * 78)


    if (
        no_improvement
        >= PATIENCE
    ):

        print()
        print(
            "EARLY STOPPING"
        )

        break


    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# LOAD BEST RESULT
# ============================================================================

best_payload = torch.load(
    BEST_DIR
    / "best_checkpoint.pt",
    map_location="cpu",
    weights_only=False,
)


G_A2B = Generator(
    residual_blocks=9,
    base_channels=64,
).to(device)

G_B2A = Generator(
    residual_blocks=9,
    base_channels=64,
).to(device)


G_A2B.load_state_dict(
    best_payload[
        "G_A2B"
    ]
)

G_B2A.load_state_dict(
    best_payload[
        "G_B2A"
    ]
)


G_A2B.eval()
G_B2A.eval()


print()
print("=" * 78)
print("COHERENT CONTINUATION FINISHED")
print("=" * 78)

print(
    "Best diagnostic P2M:",
    best_payload[
        "ta_fid_photo_to_monet"
    ]
)

print(
    "Best diagnostic M2P:",
    best_payload[
        "ta_fid_monet_to_photo"
    ]
)

print(
    "Best diagnostic AVG:",
    best_payload[
        "ta_fid_average"
    ]
)

print()

print(
    "Saved:",
    BEST_DIR
    / "best_checkpoint.pt"
)

print()

print(
    "Do NOT overwrite the Phase 3 final checkpoint yet."
)

print(
    "If this branch improves, we will literal-JPEG test "
    "its best epochs against 97.370220."
)

print("=" * 78)

LOADED COHERENT FULL CYCLEGAN STATE
Source: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\strong_fresh_ttur\best\best_checkpoint.pt
Epoch: 140
G_A2B + G_B2A + D_A + D_B restored

Loading fixed 300-image diagnostic set...


C:\Users\drashti2\AppData\Local\Temp\ipykernel_71608\1093974954.py:39: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(



COHERENT CONTINUATION BASELINE
P2M=99.256962
M2P=102.167933
AVG=100.712448

Literal JPEG score for this checkpoint from tournament:
P2M=94.678612
M2P=101.111024
AVG=97.894818

Literal Phase 3 score to beat: 97.370220


Coherent 01/10: 100%|██████████| 1000/1000 [01:16<00:00, 13.11it/s, G=2.392, cyc=0.135, id=0.150, DA=0.098, DB=0.239]



------------------------------------------------------------------------------
Total epoch 141 complete in 76.3s
LIVE  P2M=99.134366 | M2P=108.203701 | AVG=103.669033
EMA   P2M=99.146425 | M2P=103.387258 | AVG=101.266842
BEST  P2M=99.134366 | M2P=102.167933 | AVG=100.651149
Best P2M: 141 live
Best M2P: 140 source
No improvement: 0 / 3
------------------------------------------------------------------------------


...


------------------------------------------------------------------------------
Total epoch 142 complete in 75.0s
LIVE  P2M=99.687259 | M2P=107.123538 | AVG=103.405399
EMA   P2M=99.197406 | M2P=105.286600 | AVG=102.242003
BEST  P2M=99.134366 | M2P=102.167933 | AVG=100.651149
Best P2M: 141 live
Best M2P: 140 source
No improvement: 1 / 3
------------------------------------------------------------------------------


Coherent 03/10: 100%|██████████| 1000/1000 [01:15<00:00, 13.33it/s, G=2.428, cyc=0.139, id=0.152, DA=0.102, DB=0.248]



------------------------------------------------------------------------------
Total epoch 143 complete in 75.0s
LIVE  P2M=101.066467 | M2P=107.809722 | AVG=104.438095
EMA   P2M=99.625962 | M2P=105.731173 | AVG=102.678568
BEST  P2M=99.134366 | M2P=102.167933 | AVG=100.651149
Best P2M: 141 live
Best M2P: 140 source
No improvement: 2 / 3
------------------------------------------------------------------------------


Coherent 04/10: 100%|██████████| 1000/1000 [01:15<00:00, 13.33it/s, G=2.454, cyc=0.140, id=0.154, DA=0.101, DB=0.241]



------------------------------------------------------------------------------
Total epoch 144 complete in 75.0s
LIVE  P2M=100.424564 | M2P=110.624530 | AVG=105.524547
EMA   P2M=100.047851 | M2P=107.232706 | AVG=103.640278
BEST  P2M=99.134366 | M2P=102.167933 | AVG=100.651149
Best P2M: 141 live
Best M2P: 140 source
No improvement: 3 / 3
------------------------------------------------------------------------------

EARLY STOPPING

COHERENT CONTINUATION FINISHED
Best diagnostic P2M: 99.13436550309467
Best diagnostic M2P: 102.16793338279592
Best diagnostic AVG: 100.6511494429453

Saved: C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\coherent_asymmetric_from_e140\best\best_checkpoint.pt

Do NOT overwrite the Phase 3 final checkpoint yet.
If this branch improves, we will literal-JPEG test its best epochs against 97.370220.


In [38]:
# ============================================================================
# FRESH CYCLEGAN RUN V3
#
# Goal:
#   Escape the ~97 literal-FID plateau with different training dynamics.
#
# Changes from previous strong run:
#   - completely fresh model
#   - identity weight 1.0 from epoch 1
#   - NO cutout
#   - NO ADA
#   - mild DiffAugment only:
#         brightness + contrast + translation
#   - generator LR 2e-4
#   - discriminator LR 1e-4
#   - 50 epochs constant LR
#   - epochs 51-100 linear decay
#   - EMA generators
#   - independent best G_A2B / G_B2A tracking
#   - safe resume
#
# EXPECTED TIME:
#   roughly 2.5 - 3 hours on your current machine
# ============================================================================

import copy
import gc
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm


# ============================================================================
# RUN DIRECTORY
# ============================================================================

RUN_DIR = (
    Path(EXPERIMENT_DIR)
    / "fresh_v3_mild_aug_identity1"
)

SNAPSHOT_DIR = (
    RUN_DIR
    / "snapshots"
)

BEST_DIR = (
    RUN_DIR
    / "best"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SNAPSHOT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


LAST_PATH = (
    RUN_DIR
    / "last_full.pt"
)

BEST_PATH = (
    BEST_DIR
    / "best_checkpoint.pt"
)

HISTORY_PATH = (
    RUN_DIR
    / "history.json"
)


# ============================================================================
# SETTINGS
# ============================================================================

SEED = 2688

RESIDUAL_BLOCKS = 9
BASE_CHANNELS = 64

MAX_EPOCHS = 100
STEPS_PER_EPOCH = 1000

GENERATOR_LR = 2e-4
DISCRIMINATOR_LR = 1e-4

BETAS = (
    0.5,
    0.999
)

CYCLE_WEIGHT = 10.0
IDENTITY_WEIGHT = 1.0

REPLAY_CAPACITY = 50

EMA_DECAY = 0.999

N_EVAL = 300
EVAL_BATCH = 32

# First half: evaluate every 5 epochs.
# Once AVG FID < 100: evaluate every epoch.
INITIAL_EVAL_EVERY = 5

# Do not early-stop before epoch 50.
EARLY_STOP_START = 50

# After epoch 50, stop after this many evaluations
# with no directional improvement.
PATIENCE_EVALS = 10

RESUME = True


# ============================================================================
# REPRODUCIBILITY
# ============================================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ============================================================================
# REQUIRED NOTEBOOK OBJECTS
# ============================================================================

required = [
    "Generator",
    "PatchDiscriminator",
    "ReplayBuffer",
    "initialize_weights",
    "device",
    "photo_paths",
    "monet_paths",
    "train_transform",
    "eval_transform",
    "lsgan_real",
    "lsgan_fake",
    "l1",
    "rand_brightness",
    "rand_contrast",
    "rand_translation",
    "inception_features_from_tensor",
    "fid_from_features",
    "REAL_MONET_FEATURES",
    "REAL_PHOTO_FEATURES",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required notebook objects: "
        + ", ".join(missing)
        + "\nRun the earlier setup/model/data/evaluation cells first."
    )


# ============================================================================
# HELPERS
# ============================================================================

def clean_state_v3(state):

    output = {}

    for key, value in state.items():

        for prefix in (
            "_orig_mod.",
            "module.",
        ):

            if key.startswith(prefix):
                key = key[len(prefix):]

        output[key] = value

    return output


def cpu_state_v3(model):

    return {
        key:
            value.detach().cpu().clone()

        for key, value
        in model.state_dict().items()
    }


def set_requires_grad_v3(
    models,
    flag
):

    for model in models:

        for parameter in model.parameters():
            parameter.requires_grad_(flag)


# ============================================================================
# MILD DIFFAUGMENT
#
# NO saturation
# NO cutout
# NO ADA
# ============================================================================

def mild_augment_v3(x):

    x = rand_brightness(x)

    x = rand_contrast(x)

    x = rand_translation(x)

    return x


# ============================================================================
# FIXED TA DIAGNOSTIC EVALUATION SET
# ============================================================================

print(
    "Loading fixed 300-image diagnostic evaluation set..."
)


EVAL_INPUTS_V3 = {

    "photo_to_monet":
        torch.stack([
            eval_transform(path)
            for path
            in list(photo_paths)[:N_EVAL]
        ]),

    "monet_to_photo":
        torch.stack([
            eval_transform(path)
            for path
            in list(monet_paths)[:N_EVAL]
        ]),
}


EVAL_REAL_V3 = {

    "photo_to_monet":
        REAL_MONET_FEATURES,

    "monet_to_photo":
        REAL_PHOTO_FEATURES,
}


@torch.no_grad()
def direction_fid_v3(
    generator,
    direction
):

    generator.eval()

    inputs = (
        EVAL_INPUTS_V3[
            direction
        ]
    )

    features = []

    for start in range(
        0,
        inputs.size(0),
        EVAL_BATCH,
    ):

        output = generator(
            inputs[
                start:
                start + EVAL_BATCH
            ].to(device)
        )

        features.append(
            np.asarray(
                inception_features_from_tensor(
                    output
                )
            )
        )

    return float(
        fid_from_features(
            EVAL_REAL_V3[
                direction
            ],
            np.concatenate(
                features,
                axis=0
            ),
        )
    )


# ============================================================================
# BUILD MODELS
# ============================================================================

G_A2B = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)

G_B2A = Generator(
    residual_blocks=RESIDUAL_BLOCKS,
    base_channels=BASE_CHANNELS,
).to(device)

D_A = PatchDiscriminator(
    BASE_CHANNELS
).to(device)

D_B = PatchDiscriminator(
    BASE_CHANNELS
).to(device)


for model in [
    G_A2B,
    G_B2A,
    D_A,
    D_B,
]:
    initialize_weights(
        model
    )


# ============================================================================
# EMA
# ============================================================================

G_A2B_EMA = copy.deepcopy(
    G_A2B
).eval()

G_B2A_EMA = copy.deepcopy(
    G_B2A
).eval()


for model in [
    G_A2B_EMA,
    G_B2A_EMA,
]:

    for parameter in model.parameters():
        parameter.requires_grad_(False)


@torch.no_grad()
def update_ema_v3():

    for ema_model, live_model in [
        (
            G_A2B_EMA,
            G_A2B
        ),
        (
            G_B2A_EMA,
            G_B2A
        ),
    ]:

        for ema_p, live_p in zip(
            ema_model.parameters(),
            live_model.parameters(),
        ):

            ema_p.lerp_(
                live_p.detach(),
                1.0 - EMA_DECAY
            )

        for ema_b, live_b in zip(
            ema_model.buffers(),
            live_model.buffers(),
        ):

            ema_b.copy_(
                live_b
            )


# ============================================================================
# OPTIMIZERS
# ============================================================================

G_PARAMS = (
    list(
        G_A2B.parameters()
    )
    +
    list(
        G_B2A.parameters()
    )
)


generator_optimizer = torch.optim.Adam(
    G_PARAMS,
    lr=GENERATOR_LR,
    betas=BETAS,
)


discriminator_a_optimizer = torch.optim.Adam(
    D_A.parameters(),
    lr=DISCRIMINATOR_LR,
    betas=BETAS,
)


discriminator_b_optimizer = torch.optim.Adam(
    D_B.parameters(),
    lr=DISCRIMINATOR_LR,
    betas=BETAS,
)


fake_a_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)

fake_b_buffer = ReplayBuffer(
    REPLAY_CAPACITY
)


# ============================================================================
# LR SCHEDULE
# ============================================================================

def lr_multiplier_v3(epoch):

    # Epochs 1-50:
    # full LR

    if epoch <= 50:
        return 1.0

    # Epochs 51-100:
    # linear decay toward zero

    progress = (
        epoch - 50
    ) / 50.0

    return max(
        0.0,
        1.0 - progress
    )


def set_lr_v3(epoch):

    multiplier = (
        lr_multiplier_v3(
            epoch
        )
    )

    g_lr = (
        GENERATOR_LR
        * multiplier
    )

    d_lr = (
        DISCRIMINATOR_LR
        * multiplier
    )


    for group in generator_optimizer.param_groups:
        group["lr"] = g_lr


    for optimizer in [
        discriminator_a_optimizer,
        discriminator_b_optimizer,
    ]:

        for group in optimizer.param_groups:
            group["lr"] = d_lr


    return (
        g_lr,
        d_lr
    )


# ============================================================================
# DATA SAMPLING
# ============================================================================

def sample_training_batch_v3(
    paths
):

    path = random.choice(
        list(paths)
    )

    return (
        train_transform(path)
        .unsqueeze(0)
        .to(device)
    )


# ============================================================================
# DISCRIMINATOR STEP
# ============================================================================

def discriminator_step_v3(
    discriminator,
    optimizer,
    real,
    fake,
    replay_buffer,
):

    optimizer.zero_grad(
        set_to_none=True
    )


    fake_replay = replay_buffer.query(
        fake.detach()
    )


    real_prediction = discriminator(
        mild_augment_v3(
            real
        )
    )


    fake_prediction = discriminator(
        mild_augment_v3(
            fake_replay
        )
    )


    loss = 0.5 * (

        lsgan_real(
            real_prediction
        )

        +

        lsgan_fake(
            fake_prediction
        )
    )


    if not torch.isfinite(
        loss
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        return (
            float("nan"),
            float("nan"),
            1
        )


    loss.backward()


    grad_norm = float(
        torch.nn.utils.clip_grad_norm_(
            discriminator.parameters(),
            float("inf")
        )
    )


    if not np.isfinite(
        grad_norm
    ):

        optimizer.zero_grad(
            set_to_none=True
        )

        return (
            float(
                loss.detach()
            ),
            grad_norm,
            1
        )


    optimizer.step()


    return (
        float(
            loss.detach()
        ),
        grad_norm,
        0
    )


# ============================================================================
# TRAIN STEP
# ============================================================================

def train_step_v3(
    real_a,
    real_b
):

    # ------------------------------------------------------------------------
    # GENERATORS
    # ------------------------------------------------------------------------

    set_requires_grad_v3(
        [
            D_A,
            D_B
        ],
        False
    )


    generator_optimizer.zero_grad(
        set_to_none=True
    )


    # A = Monet
    # B = Photo

    fake_b = G_A2B(
        real_a
    )

    fake_a = G_B2A(
        real_b
    )


    reconstructed_a = G_B2A(
        fake_b
    )

    reconstructed_b = G_A2B(
        fake_a
    )


    identity_a = G_B2A(
        real_a
    )

    identity_b = G_A2B(
        real_b
    )


    # Mild augmentation only in the discriminator-facing branch

    adv_a2b = lsgan_real(
        D_B(
            mild_augment_v3(
                fake_b
            )
        )
    )

    adv_b2a = lsgan_real(
        D_A(
            mild_augment_v3(
                fake_a
            )
        )
    )


    cycle_a = l1(
        reconstructed_a,
        real_a
    )

    cycle_b = l1(
        reconstructed_b,
        real_b
    )


    identity_a_loss = l1(
        identity_a,
        real_a
    )

    identity_b_loss = l1(
        identity_b,
        real_b
    )


    cycle_loss = (
        cycle_a
        + cycle_b
    )


    identity_loss = (
        identity_a_loss
        + identity_b_loss
    )


    generator_loss = (
        adv_a2b
        + adv_b2a

        + CYCLE_WEIGHT
        * cycle_loss

        + IDENTITY_WEIGHT
        * identity_loss
    )


    nonfinite = 0

    generator_grad_norm = (
        float("nan")
    )


    if torch.isfinite(
        generator_loss
    ):

        generator_loss.backward()


        generator_grad_norm = float(
            torch.nn.utils.clip_grad_norm_(
                G_PARAMS,
                float("inf")
            )
        )


        if np.isfinite(
            generator_grad_norm
        ):

            generator_optimizer.step()

            update_ema_v3()

        else:

            nonfinite += 1


    else:

        nonfinite += 1


    generator_optimizer.zero_grad(
        set_to_none=True
    )


    set_requires_grad_v3(
        [
            D_A,
            D_B
        ],
        True
    )


    # ------------------------------------------------------------------------
    # DISCRIMINATORS
    # ------------------------------------------------------------------------

    (
        discriminator_a_loss,
        grad_norm_d_a,
        bad_a
    ) = discriminator_step_v3(

        D_A,
        discriminator_a_optimizer,
        real_a,
        fake_a,
        fake_a_buffer,
    )


    (
        discriminator_b_loss,
        grad_norm_d_b,
        bad_b
    ) = discriminator_step_v3(

        D_B,
        discriminator_b_optimizer,
        real_b,
        fake_b,
        fake_b_buffer,
    )


    return {

        "generator":
            float(
                generator_loss.detach()
            ),

        "cycle":
            float(
                cycle_loss.detach()
            ),

        "identity":
            float(
                identity_loss.detach()
            ),

        "adv_a2b":
            float(
                adv_a2b.detach()
            ),

        "adv_b2a":
            float(
                adv_b2a.detach()
            ),

        "discriminator_a":
            discriminator_a_loss,

        "discriminator_b":
            discriminator_b_loss,

        "grad_norm_G":
            generator_grad_norm,

        "grad_norm_D_A":
            grad_norm_d_a,

        "grad_norm_D_B":
            grad_norm_d_b,

        "nonfinite":
            (
                nonfinite
                + bad_a
                + bad_b
            ),
    }


# ============================================================================
# STATE VARIABLES
# ============================================================================

start_epoch = 1

fine_eval_mode = False

evals_without_improvement = 0

history = []

best = {

    "photo_to_monet": {
        "fid":
            float("inf"),

        "epoch":
            None,

        "variant":
            None,

        "state":
            None,
    },

    "monet_to_photo": {
        "fid":
            float("inf"),

        "epoch":
            None,

        "variant":
            None,

        "state":
            None,
    },
}


# ============================================================================
# RESUME SUPPORT
# ============================================================================

if (
    RESUME
    and LAST_PATH.exists()
):

    print()
    print("=" * 78)
    print("RESUMING FRESH V3 RUN")
    print("=" * 78)
    print(LAST_PATH)


    resume_payload = torch.load(
        LAST_PATH,
        map_location="cpu",
        weights_only=False,
    )


    G_A2B.load_state_dict(
        clean_state_v3(
            resume_payload[
                "G_A2B"
            ]
        )
    )

    G_B2A.load_state_dict(
        clean_state_v3(
            resume_payload[
                "G_B2A"
            ]
        )
    )

    D_A.load_state_dict(
        clean_state_v3(
            resume_payload[
                "D_A"
            ]
        )
    )

    D_B.load_state_dict(
        clean_state_v3(
            resume_payload[
                "D_B"
            ]
        )
    )


    G_A2B_EMA.load_state_dict(
        clean_state_v3(
            resume_payload[
                "G_A2B_EMA"
            ]
        )
    )

    G_B2A_EMA.load_state_dict(
        clean_state_v3(
            resume_payload[
                "G_B2A_EMA"
            ]
        )
    )


    generator_optimizer.load_state_dict(
        resume_payload[
            "generator_optimizer"
        ]
    )

    discriminator_a_optimizer.load_state_dict(
        resume_payload[
            "discriminator_a_optimizer"
        ]
    )

    discriminator_b_optimizer.load_state_dict(
        resume_payload[
            "discriminator_b_optimizer"
        ]
    )


    start_epoch = (
        int(
            resume_payload[
                "epoch"
            ]
        )
        + 1
    )


    best = resume_payload.get(
        "best",
        best
    )


    history = resume_payload.get(
        "history",
        []
    )


    fine_eval_mode = resume_payload.get(
        "fine_eval_mode",
        False
    )


    evals_without_improvement = int(
        resume_payload.get(
            "evals_without_improvement",
            0
        )
    )


    print(
        "Resume epoch:",
        start_epoch
    )

    print(
        "Fine-eval mode:",
        fine_eval_mode
    )

    print("=" * 78)


else:

    print()
    print("=" * 78)
    print("STARTING BRAND NEW FRESH V3 RUN")
    print("=" * 78)

    print(
        "Generator LR:",
        GENERATOR_LR
    )

    print(
        "Discriminator LR:",
        DISCRIMINATOR_LR
    )

    print(
        "Cycle weight:",
        CYCLE_WEIGHT
    )

    print(
        "Identity weight:",
        IDENTITY_WEIGHT
    )

    print(
        "EMA:",
        EMA_DECAY
    )

    print(
        "ADA: OFF"
    )

    print(
        "Cutout: OFF"
    )

    print(
        "Mild augmentation:"
        " brightness + contrast + translation"
    )

    print(
        "Epochs:",
        MAX_EPOCHS
    )

    print(
        "Steps/epoch:",
        STEPS_PER_EPOCH
    )

    print()
    print(
        "Current literal winner remains untouched:"
    )

    print(
        "Photo -> Monet  = 94.357201"
    )

    print(
        "Monet -> Photo  = 100.383239"
    )

    print(
        "Average         = 97.370220"
    )

    print("=" * 78)


# ============================================================================
# EVALUATION + BEST TRACKING
# ============================================================================

def evaluate_v3(epoch):

    global fine_eval_mode
    global evals_without_improvement


    candidates = {

        "live": {

            "photo_to_monet":
                G_B2A,

            "monet_to_photo":
                G_A2B,
        },

        "ema": {

            "photo_to_monet":
                G_B2A_EMA,

            "monet_to_photo":
                G_A2B_EMA,
        },
    }


    results = {}


    for variant, models in candidates.items():

        p2m = direction_fid_v3(
            models[
                "photo_to_monet"
            ],
            "photo_to_monet"
        )

        m2p = direction_fid_v3(
            models[
                "monet_to_photo"
            ],
            "monet_to_photo"
        )


        results[
            variant
        ] = {

            "photo_to_monet":
                p2m,

            "monet_to_photo":
                m2p,

            "average":
                (
                    p2m
                    + m2p
                ) / 2.0,
        }


    G_A2B.train()
    G_B2A.train()


    improved = []


    # ------------------------------------------------------------------------
    # PHOTO -> MONET
    # ------------------------------------------------------------------------

    for variant in [
        "live",
        "ema",
    ]:

        score = (
            results[
                variant
            ][
                "photo_to_monet"
            ]
        )


        if (
            score
            <
            best[
                "photo_to_monet"
            ][
                "fid"
            ]
        ):

            model = (
                G_B2A
                if variant == "live"
                else G_B2A_EMA
            )


            best[
                "photo_to_monet"
            ] = {

                "fid":
                    score,

                "epoch":
                    epoch,

                "variant":
                    variant,

                "state":
                    cpu_state_v3(
                        model
                    ),
            }


            improved.append(
                f"P2M:{variant}"
            )


    # ------------------------------------------------------------------------
    # MONET -> PHOTO
    # ------------------------------------------------------------------------

    for variant in [
        "live",
        "ema",
    ]:

        score = (
            results[
                variant
            ][
                "monet_to_photo"
            ]
        )


        if (
            score
            <
            best[
                "monet_to_photo"
            ][
                "fid"
            ]
        ):

            model = (
                G_A2B
                if variant == "live"
                else G_A2B_EMA
            )


            best[
                "monet_to_photo"
            ] = {

                "fid":
                    score,

                "epoch":
                    epoch,

                "variant":
                    variant,

                "state":
                    cpu_state_v3(
                        model
                    ),
            }


            improved.append(
                f"M2P:{variant}"
            )


    best_average = (

        best[
            "photo_to_monet"
        ][
            "fid"
        ]

        +

        best[
            "monet_to_photo"
        ][
            "fid"
        ]

    ) / 2.0


    # Once any live/EMA average gets under 100,
    # evaluate every epoch from then on.

    lowest_current_average = min(
        results[
            "live"
        ][
            "average"
        ],

        results[
            "ema"
        ][
            "average"
        ],
    )


    if (
        lowest_current_average
        < 100.0
    ):

        fine_eval_mode = True


    if improved:

        evals_without_improvement = 0

    else:

        evals_without_improvement += 1


    # ------------------------------------------------------------------------
    # SAVE BEST PAIR
    # ------------------------------------------------------------------------

    torch.save(
        {

            "G_B2A":
                best[
                    "photo_to_monet"
                ][
                    "state"
                ],

            "G_A2B":
                best[
                    "monet_to_photo"
                ][
                    "state"
                ],

            "ta_fid_photo_to_monet":
                best[
                    "photo_to_monet"
                ][
                    "fid"
                ],

            "ta_fid_monet_to_photo":
                best[
                    "monet_to_photo"
                ][
                    "fid"
                ],

            "ta_fid_average":
                best_average,

            "selection": {

                "photo_to_monet": {

                    "epoch":
                        best[
                            "photo_to_monet"
                        ][
                            "epoch"
                        ],

                    "variant":
                        best[
                            "photo_to_monet"
                        ][
                            "variant"
                        ],
                },

                "monet_to_photo": {

                    "epoch":
                        best[
                            "monet_to_photo"
                        ][
                            "epoch"
                        ],

                    "variant":
                        best[
                            "monet_to_photo"
                        ][
                            "variant"
                        ],
                },
            },
        },

        BEST_PATH,
    )


    # ------------------------------------------------------------------------
    # SAVE EVALUATION SNAPSHOT
    # ------------------------------------------------------------------------

    snapshot_path = (

        SNAPSHOT_DIR

        / f"epoch_{epoch:03d}.pt"
    )


    torch.save(
        {

            "epoch":
                epoch,

            "G_A2B":
                cpu_state_v3(
                    G_A2B
                ),

            "G_B2A":
                cpu_state_v3(
                    G_B2A
                ),

            "G_A2B_EMA":
                cpu_state_v3(
                    G_A2B_EMA
                ),

            "G_B2A_EMA":
                cpu_state_v3(
                    G_B2A_EMA
                ),

            "D_A":
                cpu_state_v3(
                    D_A
                ),

            "D_B":
                cpu_state_v3(
                    D_B
                ),

            "live_fid_photo_to_monet":
                results[
                    "live"
                ][
                    "photo_to_monet"
                ],

            "live_fid_monet_to_photo":
                results[
                    "live"
                ][
                    "monet_to_photo"
                ],

            "ema_fid_photo_to_monet":
                results[
                    "ema"
                ][
                    "photo_to_monet"
                ],

            "ema_fid_monet_to_photo":
                results[
                    "ema"
                ][
                    "monet_to_photo"
                ],
        },

        snapshot_path,
    )


    print()
    print("=" * 78)
    print(
        f"EVALUATION EPOCH {epoch}"
    )
    print("=" * 78)


    print(
        "LIVE  "
        f"P2M={results['live']['photo_to_monet']:.6f} "
        f"| M2P={results['live']['monet_to_photo']:.6f} "
        f"| AVG={results['live']['average']:.6f}"
    )


    print(
        "EMA   "
        f"P2M={results['ema']['photo_to_monet']:.6f} "
        f"| M2P={results['ema']['monet_to_photo']:.6f} "
        f"| AVG={results['ema']['average']:.6f}"
    )


    print()


    print(
        "BEST  "
        f"P2M={best['photo_to_monet']['fid']:.6f} "
        f"| M2P={best['monet_to_photo']['fid']:.6f} "
        f"| AVG={best_average:.6f}"
    )


    print(
        "Best P2M:",
        best[
            "photo_to_monet"
        ][
            "epoch"
        ],
        best[
            "photo_to_monet"
        ][
            "variant"
        ]
    )


    print(
        "Best M2P:",
        best[
            "monet_to_photo"
        ][
            "epoch"
        ],
        best[
            "monet_to_photo"
        ][
            "variant"
        ]
    )


    print(
        "Improved:",
        (
            ", ".join(
                improved
            )
            if improved
            else "none"
        )
    )


    print(
        "Fine evaluation mode:",
        fine_eval_mode
    )


    print(
        "No-improvement evaluations:",
        evals_without_improvement
    )


    print("=" * 78)


    return (
        results,
        best_average
    )


# ============================================================================
# TRAINING LOOP
# ============================================================================

training_start = time.perf_counter()


for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1
):

    g_lr, d_lr = (
        set_lr_v3(
            epoch
        )
    )


    G_A2B.train()
    G_B2A.train()
    D_A.train()
    D_B.train()


    totals = defaultdict(
        float
    )


    epoch_start = (
        time.perf_counter()
    )


    progress = tqdm(
        range(
            STEPS_PER_EPOCH
        ),
        desc=(
            f"Fresh V3 {epoch:03d}/"
            f"{MAX_EPOCHS}"
        ),
    )


    for step in progress:

        real_a = sample_training_batch_v3(
            monet_paths
        )

        real_b = sample_training_batch_v3(
            photo_paths
        )


        metrics = train_step_v3(
            real_a,
            real_b
        )


        for key, value in metrics.items():

            if np.isfinite(
                value
            ):

                totals[
                    key
                ] += value


        if (
            step % 50 == 0

            or

            step
            == STEPS_PER_EPOCH - 1
        ):

            n = (
                step
                + 1
            )


            progress.set_postfix({

                "G":
                    f"{totals['generator']/n:.3f}",

                "cyc":
                    f"{totals['cycle']/n:.3f}",

                "id":
                    f"{totals['identity']/n:.3f}",

                "DA":
                    f"{totals['discriminator_a']/n:.3f}",

                "DB":
                    f"{totals['discriminator_b']/n:.3f}",

                "lr":
                    f"{g_lr:.1e}",
            })


    elapsed = (
        time.perf_counter()
        - epoch_start
    )


    epoch_record = {

        "epoch":
            epoch,

        "elapsed_sec":
            elapsed,

        "generator_lr":
            g_lr,

        "discriminator_lr":
            d_lr,

        "generator_loss":
            totals[
                "generator"
            ]
            / STEPS_PER_EPOCH,

        "cycle_loss":
            totals[
                "cycle"
            ]
            / STEPS_PER_EPOCH,

        "identity_loss":
            totals[
                "identity"
            ]
            / STEPS_PER_EPOCH,

        "discriminator_a":
            totals[
                "discriminator_a"
            ]
            / STEPS_PER_EPOCH,

        "discriminator_b":
            totals[
                "discriminator_b"
            ]
            / STEPS_PER_EPOCH,

        "nonfinite":
            totals[
                "nonfinite"
            ],
    }


    # ------------------------------------------------------------------------
    # SHOULD WE EVALUATE THIS EPOCH?
    # ------------------------------------------------------------------------

    should_evaluate = (

        fine_eval_mode

        or

        epoch % INITIAL_EVAL_EVERY == 0

        or

        epoch == MAX_EPOCHS
    )


    if should_evaluate:

        results, best_average = (
            evaluate_v3(
                epoch
            )
        )


        epoch_record[
            "evaluation"
        ] = results


        epoch_record[
            "best_average"
        ] = best_average


    # ------------------------------------------------------------------------
    # SAVE HISTORY
    # ------------------------------------------------------------------------

    history.append(
        epoch_record
    )


    with open(
        HISTORY_PATH,
        "w"
    ) as file:

        json.dump(
            history,
            file,
            indent=2
        )


    # ------------------------------------------------------------------------
    # SAVE RESUME CHECKPOINT EVERY EPOCH
    # ------------------------------------------------------------------------

    torch.save(
        {

            "epoch":
                epoch,

            "G_A2B":
                cpu_state_v3(
                    G_A2B
                ),

            "G_B2A":
                cpu_state_v3(
                    G_B2A
                ),

            "G_A2B_EMA":
                cpu_state_v3(
                    G_A2B_EMA
                ),

            "G_B2A_EMA":
                cpu_state_v3(
                    G_B2A_EMA
                ),

            "D_A":
                cpu_state_v3(
                    D_A
                ),

            "D_B":
                cpu_state_v3(
                    D_B
                ),

            "generator_optimizer":
                generator_optimizer.state_dict(),

            "discriminator_a_optimizer":
                discriminator_a_optimizer.state_dict(),

            "discriminator_b_optimizer":
                discriminator_b_optimizer.state_dict(),

            "best":
                best,

            "history":
                history,

            "fine_eval_mode":
                fine_eval_mode,

            "evals_without_improvement":
                evals_without_improvement,

            "settings": {

                "generator_lr":
                    GENERATOR_LR,

                "discriminator_lr":
                    DISCRIMINATOR_LR,

                "cycle_weight":
                    CYCLE_WEIGHT,

                "identity_weight":
                    IDENTITY_WEIGHT,

                "ema_decay":
                    EMA_DECAY,

                "augmentation":
                    "brightness+contrast+translation",

                "cutout":
                    False,

                "ada":
                    False,
            },
        },

        LAST_PATH,
    )


    print()
    print(
        f"Epoch {epoch:03d} complete "
        f"in {elapsed:.1f}s "
        f"| G={epoch_record['generator_loss']:.4f} "
        f"| cycle={epoch_record['cycle_loss']:.4f} "
        f"| identity={epoch_record['identity_loss']:.4f} "
        f"| D_A={epoch_record['discriminator_a']:.4f} "
        f"| D_B={epoch_record['discriminator_b']:.4f} "
        f"| G_lr={g_lr:.8f}"
    )


    # ------------------------------------------------------------------------
    # EARLY STOPPING
    # ------------------------------------------------------------------------

    if (
        should_evaluate

        and

        epoch >= EARLY_STOP_START

        and

        evals_without_improvement
        >= PATIENCE_EVALS
    ):

        print()
        print("=" * 78)

        print(
            "EARLY STOPPING"
        )

        print(
            f"No directional improvement for "
            f"{PATIENCE_EVALS} evaluations."
        )

        print("=" * 78)

        break


    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================================
# FINISH
# ============================================================================

total_hours = (
    time.perf_counter()
    - training_start
) / 3600.0


print()
print("=" * 78)
print("FRESH V3 TRAINING FINISHED")
print("=" * 78)

print(
    f"Elapsed this session: "
    f"{total_hours:.2f} hours"
)

print()


if BEST_PATH.exists():

    best_payload = torch.load(
        BEST_PATH,
        map_location="cpu",
        weights_only=False,
    )


    print(
        "Best diagnostic Photo -> Monet:",
        best_payload[
            "ta_fid_photo_to_monet"
        ]
    )

    print(
        "Best diagnostic Monet -> Photo:",
        best_payload[
            "ta_fid_monet_to_photo"
        ]
    )

    print(
        "Best diagnostic average:",
        best_payload[
            "ta_fid_average"
        ]
    )


    print()

    print(
        "P2M selected from:",
        best_payload[
            "selection"
        ][
            "photo_to_monet"
        ]
    )

    print(
        "M2P selected from:",
        best_payload[
            "selection"
        ][
            "monet_to_photo"
        ]
    )


    # Load selected winners into notebook memory

    G_A2B = Generator(
        residual_blocks=RESIDUAL_BLOCKS,
        base_channels=BASE_CHANNELS,
    ).to(device)

    G_B2A = Generator(
        residual_blocks=RESIDUAL_BLOCKS,
        base_channels=BASE_CHANNELS,
    ).to(device)


    G_A2B.load_state_dict(
        best_payload[
            "G_A2B"
        ]
    )

    G_B2A.load_state_dict(
        best_payload[
            "G_B2A"
        ]
    )


    G_A2B.eval()
    G_B2A.eval()


    print()
    print(
        "Best V3 generators are now loaded "
        "into G_A2B and G_B2A."
    )


print()
print(
    "Current literal-JPEG score to beat: 97.370220"
)

print()

print(
    "NEXT STEP AFTER THIS RUN:"
)

print(
    "Run the literal JPEG verification on the selected V3 pair."
)

print(
    "Do not overwrite the Phase 3 winner unless "
    "the literal result is lower than 97.370220."
)

print("=" * 78)

Loading fixed 300-image diagnostic evaluation set...

STARTING BRAND NEW FRESH V3 RUN
Generator LR: 0.0002
Discriminator LR: 0.0001
Cycle weight: 10.0
Identity weight: 1.0
EMA: 0.999
ADA: OFF
Cutout: OFF
Mild augmentation: brightness + contrast + translation
Epochs: 100
Steps/epoch: 1000

Current literal winner remains untouched:
Photo -> Monet  = 94.357201
Monet -> Photo  = 100.383239
Average         = 97.370220


Fresh V3 001/100: 100%|██████████| 1000/1000 [01:20<00:00, 12.41it/s, G=7.192, cyc=0.588, id=0.575, DA=0.320, DB=0.325, lr=2.0e-04]



Epoch 001 complete in 80.6s | G=7.1919 | cycle=0.5876 | identity=0.5753 | D_A=0.3197 | D_B=0.3248 | G_lr=0.00020000


Fresh V3 002/100: 100%|██████████| 1000/1000 [01:19<00:00, 12.64it/s, G=6.132, cyc=0.496, id=0.479, DA=0.248, DB=0.244, lr=2.0e-04]



Epoch 002 complete in 79.1s | G=6.1319 | cycle=0.4964 | identity=0.4788 | D_A=0.2478 | D_B=0.2442 | G_lr=0.00020000


...


Epoch 098 complete in 78.3s | G=2.6745 | cycle=0.1503 | identity=0.2361 | D_A=0.1413 | D_B=0.1825 | G_lr=0.00000800


Fresh V3 099/100: 100%|██████████| 1000/1000 [01:18<00:00, 12.78it/s, G=2.663, cyc=0.150, id=0.233, DA=0.144, DB=0.182, lr=4.0e-06]



Epoch 099 complete in 78.3s | G=2.6627 | cycle=0.1495 | identity=0.2334 | D_A=0.1442 | D_B=0.1821 | G_lr=0.00000400


Fresh V3 100/100: 100%|██████████| 1000/1000 [01:19<00:00, 12.56it/s, G=2.630, cyc=0.149, id=0.231, DA=0.143, DB=0.177, lr=0.0e+00]



EVALUATION EPOCH 100
LIVE  P2M=100.999064 | M2P=104.607448 | AVG=102.803256
EMA   P2M=100.901105 | M2P=104.651046 | AVG=102.776075

BEST  P2M=100.369139 | M2P=103.213288 | AVG=101.791213
Best P2M: 80 ema
Best M2P: 95 live
Improved: none
Fine evaluation mode: False
No-improvement evaluations: 1

Epoch 100 complete in 79.6s | G=2.6298 | cycle=0.1486 | identity=0.2305 | D_A=0.1433 | D_B=0.1766 | G_lr=0.00000000

FRESH V3 TRAINING FINISHED
Elapsed this session: 2.46 hours

Best diagnostic Photo -> Monet: 100.36913864333593
Best diagnostic Monet -> Photo: 103.21328806813437
Best diagnostic average: 101.79121335573515

P2M selected from: {'epoch': 80, 'variant': 'ema'}
M2P selected from: {'epoch': 95, 'variant': 'live'}

Best V3 generators are now loaded into G_A2B and G_B2A.

Current literal-JPEG score to beat: 97.370220

NEXT STEP AFTER THIS RUN:
Run the literal JPEG verification on the selected V3 pair.
Do not overwrite the Phase 3 winner unless the literal result is lower than 97.370220

In [39]:
# ============================================================================
# TA EVALUATION PREP
# Load Phase 3 winner and generate pred_A2B / pred_B2A as JPEG
# ============================================================================

from pathlib import Path
from PIL import Image
import shutil
import torch
from tqdm.auto import tqdm


# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU")

BEST_CHECKPOINT = (
    PROJECT_ROOT
    / "task3_gan"
    / "drashti"
    / "experiment_v2"
    / "final_task3_pipeline"
    / "final"
    / "final_generators.pt"
)

REAL_MONET_DIR = (
    PROJECT_ROOT
    / "task3_gan"
    / "data"
    / "monet_jpg"
)

REAL_PHOTO_DIR = (
    PROJECT_ROOT
    / "task3_gan"
    / "data"
    / "photo_jpg"
)


# Make a folder matching the TA notebook structure
TA_BASE = (
    PROJECT_ROOT
    / "Part 3"
    / "Data"
)

TA_REAL_MONET = TA_BASE / "monet_jpg"
TA_REAL_PHOTO = TA_BASE / "photo_jpg"
TA_GEN_A2B = TA_BASE / "pred_A2B"
TA_GEN_B2A = TA_BASE / "pred_B2A"


for d in [
    TA_BASE,
    TA_GEN_A2B,
    TA_GEN_B2A,
]:
    d.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# WE DO NOT NEED TO COPY THOUSANDS OF REAL IMAGES
# Create directory links if possible; otherwise copy first 300 only.
# TA evaluator uses N_EVAL=300.
# ---------------------------------------------------------------------------

def prepare_real_folder(source, target):

    if target.exists() and any(target.iterdir()):
        print("Already prepared:", target)
        return

    target.mkdir(parents=True, exist_ok=True)

    files = sorted([
        p for p in source.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])[:300]

    for p in tqdm(files, desc=f"Preparing {target.name}"):
        shutil.copy2(p, target / p.name)


prepare_real_folder(
    REAL_MONET_DIR,
    TA_REAL_MONET
)

prepare_real_folder(
    REAL_PHOTO_DIR,
    TA_REAL_PHOTO
)


# ---------------------------------------------------------------------------
# LOAD PHASE 3 GENERATORS
# ---------------------------------------------------------------------------

payload = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


G_A2B = Generator(
    residual_blocks=9,
    base_channels=64,
).to(device)

G_B2A = Generator(
    residual_blocks=9,
    base_channels=64,
).to(device)


G_A2B.load_state_dict(
    payload["G_A2B"]
)

G_B2A.load_state_dict(
    payload["G_B2A"]
)

G_A2B.eval()
G_B2A.eval()


print()
print("Loaded:")
print(BEST_CHECKPOINT)


# ---------------------------------------------------------------------------
# CLEAR OLD PREDICTIONS
# ---------------------------------------------------------------------------

for folder in [
    TA_GEN_A2B,
    TA_GEN_B2A,
]:

    for p in folder.iterdir():

        if p.is_file() and p.suffix.lower() in [
            ".jpg",
            ".jpeg",
            ".png",
        ]:
            p.unlink()


# ---------------------------------------------------------------------------
# IMAGE SAVE HELPER
# [-1, 1] tensor -> JPEG
# ---------------------------------------------------------------------------

def save_tensor_as_jpeg(
    tensor,
    path,
    quality=95,
):

    x = (
        tensor
        .detach()
        .cpu()
        .squeeze(0)
        .clamp(-1, 1)
    )

    x = (
        (x + 1.0)
        / 2.0
    )

    x = (
        x
        .permute(1, 2, 0)
        .numpy()
    )

    x = (
        x * 255.0
    ).round().clip(
        0,
        255
    ).astype("uint8")

    Image.fromarray(
        x
    ).save(
        path,
        format="JPEG",
        quality=quality,
    )


# ---------------------------------------------------------------------------
# SELECT EXACT FIRST 300 SORTED FILES
# This matches TA list_images + take_n behavior.
# ---------------------------------------------------------------------------

monet_eval_paths = sorted([
    p for p in REAL_MONET_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])[:300]

photo_eval_paths = sorted([
    p for p in REAL_PHOTO_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])[:300]


print()
print("Monet inputs:", len(monet_eval_paths))
print("Photo inputs:", len(photo_eval_paths))


# ---------------------------------------------------------------------------
# GENERATE Monet -> Photo
# pred_A2B
# ---------------------------------------------------------------------------

with torch.no_grad():

    for i, path in enumerate(
        tqdm(
            monet_eval_paths,
            desc="Generating pred_A2B Monet->Photo"
        )
    ):

        x = (
            eval_transform(path)
            .unsqueeze(0)
            .to(device)
        )

        y = G_A2B(x)

        output_path = (
            TA_GEN_A2B
            / f"{i:05d}.jpg"
        )

        save_tensor_as_jpeg(
            y,
            output_path,
            quality=95
        )


# ---------------------------------------------------------------------------
# GENERATE Photo -> Monet
# pred_B2A
# ---------------------------------------------------------------------------

with torch.no_grad():

    for i, path in enumerate(
        tqdm(
            photo_eval_paths,
            desc="Generating pred_B2A Photo->Monet"
        )
    ):

        x = (
            eval_transform(path)
            .unsqueeze(0)
            .to(device)
        )

        y = G_B2A(x)

        output_path = (
            TA_GEN_B2A
            / f"{i:05d}.jpg"
        )

        save_tensor_as_jpeg(
            y,
            output_path,
            quality=95
        )


print()
print("=" * 78)
print("TA PREDICTIONS READY")
print("=" * 78)

print("Real Monet :", TA_REAL_MONET)
print("Real Photo :", TA_REAL_PHOTO)
print("pred_A2B   :", TA_GEN_A2B)
print("pred_B2A   :", TA_GEN_B2A)

print()

print(
    "pred_A2B count:",
    len(list(TA_GEN_A2B.glob("*.jpg")))
)

print(
    "pred_B2A count:",
    len(list(TA_GEN_B2A.glob("*.jpg")))
)

print("=" * 78)

Preparing photo_jpg: 100%|██████████| 300/300 [00:00<00:00, 419.08it/s]



Loaded:
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\experiment_v2\final_task3_pipeline\final\final_generators.pt

Monet inputs: 300
Photo inputs: 300


Generating pred_B2A Photo->Monet: 100%|██████████| 300/300 [00:02<00:00, 134.46it/s]


TA PREDICTIONS READY
Real Monet : C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\Part 3\Data\monet_jpg
Real Photo : C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\Part 3\Data\photo_jpg
pred_A2B   : C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\Part 3\Data\pred_A2B
pred_B2A   : C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\Part 3\Data\pred_B2A

pred_A2B count: 300
pred_B2A count: 300


In [40]:
# ============================================================================
# TA INPUT SANITY CHECK
# ============================================================================

import os
import glob

BASE = str(
    PROJECT_ROOT
    / "Part 3"
    / "Data"
)

REAL_MONET = os.path.join(
    BASE,
    "monet_jpg"
)

REAL_PHOTO = os.path.join(
    BASE,
    "photo_jpg"
)

GEN_A2B = os.path.join(
    BASE,
    "pred_A2B"
)

GEN_B2A = os.path.join(
    BASE,
    "pred_B2A"
)

N_EVAL = 300
BATCH_SIZE = 32


def list_images(folder):

    exts = (
        ".jpg",
        ".jpeg",
        ".png"
    )

    paths = []

    for ext in exts:

        paths.extend(
            glob.glob(
                os.path.join(
                    folder,
                    f"*{ext}"
                )
            )
        )

        paths.extend(
            glob.glob(
                os.path.join(
                    folder,
                    f"*{ext.upper()}"
                )
            )
        )

    return sorted(
        list(
            set(paths)
        )
    )


def take_n(paths, n):

    if n is None:
        return paths

    return paths[
        :min(
            n,
            len(paths)
        )
    ]


for d in [
    REAL_MONET,
    REAL_PHOTO,
    GEN_A2B,
    GEN_B2A,
]:

    assert os.path.isdir(d), (
        f"Missing folder: {d}"
    )


real_monet = take_n(
    list_images(
        REAL_MONET
    ),
    N_EVAL
)

real_photo = take_n(
    list_images(
        REAL_PHOTO
    ),
    N_EVAL
)

gen_a2b = take_n(
    list_images(
        GEN_A2B
    ),
    N_EVAL
)

gen_b2a = take_n(
    list_images(
        GEN_B2A
    ),
    N_EVAL
)


print("Counts used by TA evaluator:")
print()

print(
    "Real Monet:",
    len(real_monet)
)

print(
    "Generated Monet B2A:",
    len(gen_b2a)
)

print(
    "Real Photo:",
    len(real_photo)
)

print(
    "Generated Photo A2B:",
    len(gen_a2b)
)


assert len(real_monet) == 300
assert len(real_photo) == 300
assert len(gen_a2b) == 300
assert len(gen_b2a) == 300

print()
print("PASS: all four TA sets contain exactly 300 images.")

Counts used by TA evaluator:

Real Monet: 300
Generated Monet B2A: 300
Real Photo: 300
Generated Photo A2B: 300

PASS: all four TA sets contain exactly 300 images.


In [41]:
# ============================================================================
# TA EXACT INCEPTION SETUP
# ============================================================================

import numpy as np

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models

import scipy.linalg
from scipy.spatial.distance import cosine


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)


def get_ta_inception_model():

    inception = models.inception_v3(

        weights=(
            models
            .Inception_V3_Weights
            .IMAGENET1K_V1
        ),

        transform_input=False
    )

    inception.fc = (
        nn.Identity()
    )

    inception.to(
        device
    )

    inception.eval()

    return inception


TA_INCEPTION_TF = T.Compose([

    T.Resize(
        299
    ),

    T.CenterCrop(
        299
    ),

    T.ToTensor(),

    T.Normalize(
        (
            0.485,
            0.456,
            0.406
        ),
        (
            0.229,
            0.224,
            0.225
        )
    )
])


def ta_load_batch(paths):

    imgs = []

    for p in paths:

        img = (
            Image
            .open(p)
            .convert("RGB")
        )

        imgs.append(
            TA_INCEPTION_TF(
                img
            )
        )

    return torch.stack(
        imgs,
        dim=0
    )


@torch.no_grad()
def ta_get_activations(
    model,
    image_paths,
    batch_size=32,
):

    feats = []

    for i in tqdm(
        range(
            0,
            len(image_paths),
            batch_size
        ),
        desc="TA Inception activations"
    ):

        batch_paths = (
            image_paths[
                i:
                i + batch_size
            ]
        )

        x = (
            ta_load_batch(
                batch_paths
            )
            .to(device)
        )

        f = (
            model(x)
            .detach()
            .cpu()
            .numpy()
        )

        feats.append(
            f
        )

    return np.concatenate(
        feats,
        axis=0
    )


print(
    "TA Inception setup ready."
)

Device: cuda
TA Inception setup ready.


In [42]:
# ============================================================================
# TA EXACT FID + MiFID
# ============================================================================

def ta_frechet_distance(
    mu1,
    sigma1,
    mu2,
    sigma2,
    eps=1e-6,
):

    # Same computation as TA script.
    # Avoid deprecated disp=False warning where possible.

    result = scipy.linalg.sqrtm(
        sigma1.dot(
            sigma2
        )
    )

    if isinstance(
        result,
        tuple
    ):
        covmean = result[0]

    else:
        covmean = result


    if not np.isfinite(
        covmean
    ).all():

        offset = (
            np.eye(
                sigma1.shape[0]
            )
            * eps
        )

        covmean = scipy.linalg.sqrtm(

            (
                sigma1
                + offset
            ).dot(
                sigma2
                + offset
            )
        )


    if np.iscomplexobj(
        covmean
    ):

        covmean = (
            covmean.real
        )


    diff = (
        mu1
        - mu2
    )


    return float(

        diff.dot(
            diff
        )

        +

        np.trace(

            sigma1

            +

            sigma2

            -

            2
            * covmean
        )
    )


def ta_calculate_fid_mifid(
    real_paths,
    gen_paths,
    batch_size=32,
    subsample_to_match=True,
):

    real_paths = sorted(
        real_paths
    )

    gen_paths = sorted(
        gen_paths
    )


    if subsample_to_match:

        n = min(
            len(real_paths),
            len(gen_paths)
        )

        real_paths = (
            real_paths[:n]
        )

        gen_paths = (
            gen_paths[:n]
        )


    model = (
        get_ta_inception_model()
    )


    real_act = (
        ta_get_activations(
            model,
            real_paths,
            batch_size=batch_size
        )
    )


    gen_act = (
        ta_get_activations(
            model,
            gen_paths,
            batch_size=batch_size
        )
    )


    mu_r = real_act.mean(
        axis=0
    )

    sig_r = np.cov(
        real_act,
        rowvar=False
    )


    mu_g = gen_act.mean(
        axis=0
    )

    sig_g = np.cov(
        gen_act,
        rowvar=False
    )


    fid = ta_frechet_distance(

        mu_r,
        sig_r,
        mu_g,
        sig_g
    )


    # TA's MiFID implementation:
    # mean cosine distance between index-aligned features

    m = min(
        len(real_act),
        len(gen_act)
    )


    cos_dists = [

        cosine(
            real_act[i],
            gen_act[i]
        )

        for i in range(m)
    ]


    mifid = float(
        np.mean(
            cos_dists
        )
    )


    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return (
        fid,
        mifid
    )


print(
    "TA FID/MiFID functions ready."
)

TA FID/MiFID functions ready.


In [43]:
# ============================================================================
# RUN TA EXACT EVALUATION
# ============================================================================

print()
print("=" * 78)
print("TA EVALUATION")
print("=" * 78)


print()
print(
    "Evaluating Photo -> Monet (B2A)"
)


fid_B2A, mifid_B2A = (
    ta_calculate_fid_mifid(

        real_monet,
        gen_b2a,

        batch_size=BATCH_SIZE
    )
)


print(
    f"[Photo->Monet] "
    f"FID={fid_B2A:.6f} "
    f"MiFID={mifid_B2A:.6f}"
)


print()
print(
    "Evaluating Monet -> Photo (A2B)"
)


fid_A2B, mifid_A2B = (
    ta_calculate_fid_mifid(

        real_photo,
        gen_a2b,

        batch_size=BATCH_SIZE
    )
)


print(
    f"[Monet->Photo] "
    f"FID={fid_A2B:.6f} "
    f"MiFID={mifid_A2B:.6f}"
)


sub_fid = (
    fid_A2B
    + fid_B2A
) / 2.0


sub_mifid = (
    mifid_A2B
    + mifid_B2A
) / 2.0


print()
print("=" * 78)

print(
    f"AVERAGE FID   = "
    f"{sub_fid:.6f}"
)

print(
    f"AVERAGE MiFID = "
    f"{sub_mifid:.6f}"
)

print("=" * 78)


TA EVALUATION

Evaluating Photo -> Monet (B2A)


TA Inception activations: 100%|██████████| 10/10 [00:00<00:00, 13.91it/s]


[Photo->Monet] FID=95.355329 MiFID=0.406077

Evaluating Monet -> Photo (A2B)


TA Inception activations: 100%|██████████| 10/10 [00:00<00:00, 13.32it/s]


[Monet->Photo] FID=99.562875 MiFID=0.416181

AVERAGE FID   = 97.459102
AVERAGE MiFID = 0.411129


In [44]:
# ============================================================================
# CREATE TA SUBMISSION.CSV
# ============================================================================

import pandas as pd


submission = pd.DataFrame([{

    "ID":
        1,

    "FID":
        float(
            sub_fid
        ),

    "MiFID":
        float(
            sub_mifid
        ),
}])


SUBMISSION_PATH = (
    PROJECT_ROOT
    / "task3_gan"
    / "drashti"
    / "submission.csv"
)


submission.to_csv(
    SUBMISSION_PATH,
    index=False
)


print()
print("=" * 78)
print("SUBMISSION CREATED")
print("=" * 78)

print(
    submission.to_string(
        index=False
    )
)

print()

print(
    "Saved to:"
)

print(
    SUBMISSION_PATH
)

print("=" * 78)


SUBMISSION CREATED
 ID       FID    MiFID
  1 97.459102 0.411129

Saved to:
C:\Users\drashti2\Desktop\drashti\DATA266_LAB1_GPU\task3_gan\drashti\submission.csv
